# Cambrian-P on VSI-Bench: video failure audit (10% of videos)

Runs **Cambrian-P-7B** (128 frames) on every VSI-Bench question attached to a frozen, seeded **10% sample of VSI-Bench's videos**, scores the answers with the **official VSI-Bench scorer** from the pinned Cambrian-P lmms-eval code, and supports a **manual, saved review** of whether the frames the model saw contain the evidence. The deliverable is **20 distinct verified failure videos if the sample has them (10 minimum)**, each with the playable video, the question, the model's and the benchmark's answers, the official score, the reviewer's verdict and the frame evidence.

**What counts as a verified failure.** Only a case that passes one shared validator: a genuine prediction for a selected video, an official automatic failure candidate, a saved *final* review of that exact question in this run with verdict `wrong`, complete review fields, a playable video, and valid evidence (or a documented search scope). Exported, pending, overridden or uncertain cases never count. Inference completion and case-export completion are reported separately.

**Verified facts about the data** (pinned revision `3a6a1cf9ac54…`, checked 8 Nov 2026; the code re-checks them):
- Config `full`, split `test` = `test_debiased.parquet` + `test_pruned.parquet` = all **5,130 questions**, identical to `test.jsonl` (the `debiased` config would be only 2,362).
- **288 videos** = unique `(dataset, scene_name)` over ARKitScenes (150), ScanNet (88), ScanNet++ (50). 10% → **29 videos**; the question count follows from those videos (453 at this revision, 8.8% of questions).
- Media: `arkitscenes.zip`, `scannet.zip`, `scannetpp.zip` (5.7 GB), each video at `<dataset>/<scene_name>.mp4`. Zip archives can be read at random, so only the selected videos are transferred (about 0.4 GB), by HTTP range requests, with each file's size and CRC checked. If the server refuses range requests, the notebook downloads the whole archive instead and says so first.

**Run order**
1. Sections 1–2: settings, helper code, pins, environment and model (restored from the Drive cache when possible), run configuration.
2. Sections 3–5: annotations, the frozen video sample, the media plan.
3. Section 6 with `RUN_MODE = "full"` (the default): every question of the **selected ~10% of videos** (29 videos, 453 questions at the pinned revision), **not the whole benchmark**. Rerun 6b until it reports nothing pending; the whole sample runs even if 20 failures are already found. For a small smoke test first, set `RUN_MODE = "pilot"` in 1a (2 videos, at most 10 questions) and run the resume check (6c, pilot only); pilot answers are reused by the full run because the run mode is not part of the run identity.
4. Section 7 (scoring), section 8 (candidates and the **manual** frame-evidence review), section 9 (verified manifest, report, package). Sections 7–9 need no GPU.

**Reviewing later without re-running anything.** Saved answers are never recomputed: 6b only answers questions that have no saved answer, and loads the model weights only then. For a review session, a **CPU runtime** is enough: select 1a and use Runtime > Run after; setup comes from the Drive cache, 6b reports nothing to answer, and 8c opens with your saved progress.

**Leaving it running (Colab Pro+ background execution).** Approve the Drive sign-in at the start, then you can close the tab: every result is written to Drive as it is produced (answers after each video, scores, queue, exports, report), and everything the cells print goes to `logs/notebook-<time>.log` in the run folder, with a start/finish/error line per cell. Run all does not unmount Drive. The one part that needs you is the review in 8c, afterwards.

## 1. Settings and output locations

In [ ]:
# @title 1a. Settings
DATASET_REPO = "nyu-visionx/VSI-Bench"
DATASET_CONFIG = "full"
DATASET_SPLIT = "test"
DATASET_REVISION = ""  # Resolve once, persist immutable revision, reuse on resume.
SUBSET_FRACTION = 0.10
SEED = 42
MODEL_REPO = "nyu-visionx/Cambrian-P-7B"
MODEL_REVISION = ""    # Resolve once and persist.
CODE_COMMIT = ""       # Cambrian-P code (and its lmms-eval VSI-Bench formatter/scorer); resolve once and persist.
NUM_FRAMES = 128
RUN_MODE = "full"  # full selected subset; set "pilot" explicitly for a small smoke test
TARGET_FAILURE_VIDEOS = 20
MIN_FAILURE_VIDEOS = 10
RUN_ID = "vsibench-video10pct-seed42-cambrianp7b-v1"
LOCAL_ROOT = "/content/vsibench_work"
OUTPUT_ROOT = "/content/drive/MyDrive/Cambrian-P/vsibench_runs/" + RUN_ID

# Further settings (defaults are fine).
SAMPLING_UNIT = "video"     # unique (dataset, scene_name); every question of a selected video is run
PILOT_VIDEOS, PILOT_MAX_QA = 2, 10
MAX_RETRIES = 2             # bounded retries of a question after an execution error (same settings)
MEDIA_BATCH_GB = 2          # most video data on local disk per model batch
REPO_URL = "https://github.com/cambrian-mllm/cambrian-p.git"
# Generation: the pinned vsibench.yaml generation_kwargs (checked against the file in 2b).
GENERATION = {"max_new_tokens": 16, "temperature": 0, "top_p": 1.0, "num_beams": 1, "do_sample": False}
PREPROCESSING = ("frames = np.linspace(0, total-1, NUM_FRAMES, dtype=int) decoded with decord (PyAV fallback, recorded "
                 "with frame indices, timestamps and SHA-1 per frame); expand2square with the image-processor mean colour; "
                 "SigLIP image processor; fp16")
PROMPT_POLICY = "official-vsibench_doc_to_text-default-kwargs-v1"   # must equal vsibench_lib.PROMPT_POLICY

# Drive cache shared by every run and session. Environment and model caches are keyed by code commit / model
# revision (shared with other Cambrian-P notebooks); dataset files and videos are keyed by the VSI-Bench revision.
CACHE_ROOT = "/content/drive/MyDrive/Cambrian-P/cache"
CACHE_ENV = True            # installed environment (~8 GB)
CACHE_MODEL = True          # model weights
CACHE_VIDEOS = True         # the selected videos (~0.4 GB), so later sessions and the export need no transfer

import os, json, shutil, subprocess, sys
assert RUN_MODE in ("pilot", "full"), "RUN_MODE must be 'pilot' or 'full'"
from google.colab import drive
drive.mount("/content/drive")
for d in (OUTPUT_ROOT, LOCAL_ROOT, CACHE_ROOT, "/content/vsibench", os.path.join(OUTPUT_ROOT, "logs")):
    os.makedirs(d, exist_ok=True)

# Everything the cells print is also appended to a log on Drive, with a start/finish/error line per cell, so a run
# left going in the background (Colab Pro+) can be followed and checked later from Drive alone.
import time as _t
class _DriveTee:
    def __init__(self, stream, path):
        self._stream, self._f, self._vsibench_tee = stream, open(path, "a", buffering=1), True
    def write(self, s):
        self._stream.write(s)
        try:
            self._f.write(s)
        except Exception:
            pass
        return len(s)
    def flush(self):
        self._stream.flush()
        try:
            self._f.flush()
        except Exception:
            pass
    def __getattr__(self, name):
        return getattr(self._stream, name)
NOTEBOOK_LOG = os.path.join(OUTPUT_ROOT, "logs", f"notebook-{_t.strftime('%Y%m%d-%H%M%S')}.log")
_base = sys.stdout._stream if getattr(sys.stdout, "_vsibench_tee", False) else sys.stdout
sys.stdout = _DriveTee(_base, NOTEBOOK_LOG)
try:
    _ip = get_ipython()
    def _cell_title(info):
        lines = (getattr(info, "raw_cell", "") or "").splitlines()
        return next((l.strip() for l in lines if "@title" in l), lines[0][:80] if lines else "")
    def _log_start(info):
        print(f"\n===== {_t.strftime('%Y-%m-%d %H:%M:%S')} start: {_cell_title(info)}", file=sys.stdout._f, flush=True)
    def _log_end(result):
        err = result.error_before_exec or result.error_in_exec
        print(f"===== {_t.strftime('%Y-%m-%d %H:%M:%S')} " + (f"ERROR: {type(err).__name__}: {err}" if err else "finished"),
              file=sys.stdout._f, flush=True)
    for _ev, _fn in (("pre_run_cell", _log_start), ("post_run_cell", _log_end)):
        for _old in [f for f in _ip.events.callbacks[_ev] if getattr(f, "__name__", "") == _fn.__name__]:
            _ip.events.unregister(_ev, _old)       # rerunning 1a does not register twice
        _ip.events.register(_ev, _fn)
except NameError:                                   # not running under IPython
    pass

# Optional Hugging Face token from Colab secrets (key icon, name HF_TOKEN). VSI-Bench and the model are public, so
# none is needed; it only raises rate limits. Never printed.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or None
except Exception:
    pass

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
du = shutil.disk_usage("/content")
print(f"Run mode:      {RUN_MODE}")
print(f"Drive output:  {OUTPUT_ROOT}   (manifests, predictions, reviews, cases, report: persistent)")
print(f"Local work:    {LOCAL_ROOT}   (working copies, lost when the runtime ends)")
print(f"Drive cache:   {CACHE_ROOT}   (environment: {CACHE_ENV}, model: {CACHE_MODEL}, videos: {CACHE_VIDEOS})")
print(f"GPU:           {gpu or 'none found: section 6 needs a GPU (Runtime > Change runtime type > A100); 7-9 do not'}")
print(f"Local disk:    {du.free / 1e9:.0f} GB free of {du.total / 1e9:.0f} GB")
print(f"Notebook log:  {NOTEBOOK_LOG}   (all printed output and per-cell start/finish/errors)")

### 1b. Helper code: `vsibench_lib.py`

In [ ]:
%%writefile /content/vsibench/vsibench_lib.py
"""Shared logic for the Cambrian-P / VSI-Bench video failure audit (no GPU, no model code).

Imported by the notebook kernel, the inference worker and the review module. Deterministic and unit-tested by the
notebook's verification cells. VSI-Bench's own prompt formatter and scorer are NOT re-implemented here: they are
loaded from the pinned Cambrian-P lmms-eval file (see load_official), so scoring is the official code itself.
"""
import ast
import hashlib
import html
import io
import json
import math
import os
import random
import re
import time
import urllib.request
import zipfile
from collections import OrderedDict
from functools import partial

LIB_VERSION = "vsibench-lib-1"
SAMPLING_UNIT = "video"
PROMPT_POLICY = "official-vsibench_doc_to_text-default-kwargs-v1"
OFFICIAL_SCORER_PATH = "lmms-eval/lmms_eval/tasks/vsibench/utils.py"     # relative to the Cambrian-P checkout
OFFICIAL_TASK_YAML = "lmms-eval/lmms_eval/tasks/vsibench/vsibench.yaml"
ANNOTATION_FIELDS = ("id", "dataset", "scene_name", "question_type", "question", "ground_truth", "options")
VIDEO_DATASETS = ("arkitscenes", "scannet", "scannetpp")
AUTOMATIC_STATUSES = ("correct", "wrong_candidate", "imperfect", "format_error", "unscorable", "execution_error")
OPTION_RE = re.compile(r"^([A-Z])\. ")
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def stable_hash(obj, n=None):
    s = hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    return s[:n] if n else s


def safe_key(video_key):
    """Filesystem-safe, collision-proof name for a media path; the mapping is kept in the manifests."""
    base = re.sub(r"[^A-Za-z0-9._-]+", "__", os.path.splitext(video_key)[0]).strip("_")[:80]
    return f"{base}-{hashlib.sha1(video_key.encode()).hexdigest()[:8]}"




# ----------------------------------------------------------------------------------------------- files
def write_json_atomic(path, obj):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = f"{path}.tmp-{os.getpid()}"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def write_jsonl_atomic(path, rows):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = f"{path}.tmp-{os.getpid()}"
    with open(tmp, "w") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def read_jsonl(path, tolerate_truncated_tail=True):
    """Rows of a JSONL file. In tolerant mode a torn line (from an interrupted append) is skipped and counted."""
    rows, bad = [], 0
    if not os.path.exists(path):
        return rows, bad
    with open(path) as f:
        lines = f.read().split("\n")
    for k, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            # A torn line from an interrupted append (any position, since appends after a crash start a new line).
            if tolerate_truncated_tail:
                bad += 1
            else:
                raise
    return rows, bad


class Journal:
    """Append-only JSONL with fsync after every row: an interrupt loses at most the row being written."""

    def __init__(self, path):
        self.path = path
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    def append(self, row):
        with open(self.path, "a+b") as f:
            # If an earlier append was interrupted mid-line, start on a fresh line so this row stays readable.
            f.seek(0, os.SEEK_END)
            if f.tell():
                f.seek(-1, os.SEEK_END)
                if f.read(1) != b"\n":
                    f.write(b"\n")
            f.write((json.dumps(row, ensure_ascii=False, default=str) + "\n").encode())
            f.flush()
            os.fsync(f.fileno())




def safe_extract_path(dest_root, member_name):
    """Destination for a tar member, refusing absolute paths, '..' and anything outside dest_root."""
    if not member_name or member_name.startswith(("/", "\\")) or re.match(r"^[A-Za-z]:", member_name):
        raise ValueError(f"unsafe member path {member_name!r}")
    norm = os.path.normpath(member_name)
    if norm.startswith("..") or os.path.isabs(norm) or "/../" in f"/{norm}/":
        raise ValueError(f"unsafe member path {member_name!r}")
    dest = os.path.realpath(os.path.join(dest_root, norm))
    root = os.path.realpath(dest_root)
    if not dest.startswith(root + os.sep):
        raise ValueError(f"member escapes the destination: {member_name!r}")
    return dest




# ----------------------------------------------------------------------------------------------- shards
def write_shard(shard_dir, name, rows, config_hash):
    """Durable shard: rows file plus a completion marker holding its hash. Written atomically, marker last."""
    os.makedirs(shard_dir, exist_ok=True)
    data = os.path.join(shard_dir, f"{name}.jsonl")
    write_jsonl_atomic(data, rows)
    marker = {"name": name, "rows": len(rows), "sha256": sha256_file(data), "config_hash": config_hash,
              "finished": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "lib": LIB_VERSION}
    write_json_atomic(os.path.join(shard_dir, f"{name}.done.json"), marker)
    return data


def load_shards(shard_dir, config_hash):
    """Rows of every complete, verified shard; incomplete, corrupted or foreign shards are reported, not used."""
    rows, report = [], {"complete": 0, "incomplete": [], "corrupted": [], "other_config": []}
    if not os.path.isdir(shard_dir):
        return rows, report
    for fn in sorted(os.listdir(shard_dir)):
        if not fn.endswith(".jsonl") or ".tmp-" in fn:
            continue
        name = fn[:-len(".jsonl")]
        marker_path = os.path.join(shard_dir, f"{name}.done.json")
        if not os.path.exists(marker_path):
            report["incomplete"].append(name)
            continue
        marker = json.load(open(marker_path))
        if marker.get("config_hash") != config_hash:
            report["other_config"].append(name)
            continue
        if sha256_file(os.path.join(shard_dir, fn)) != marker.get("sha256"):
            report["corrupted"].append(name)
            continue
        part, _ = read_jsonl(os.path.join(shard_dir, fn), tolerate_truncated_tail=False)
        rows += part
        report["complete"] += 1
    return rows, report


def merge_results(rows, config_hash):
    """One outcome per qa_id: the first successful answer wins; failed attempts are kept separately."""
    ok, failed, video_meta = {}, {}, {}
    for r in rows:
        if r.get("config_hash") != config_hash:
            continue
        if r.get("kind") == "video_meta":
            video_meta.setdefault(r["video_key"], r)
            continue
        if r.get("status") == "ok":
            # Deterministic whatever the file order: the earliest recorded answer wins.
            prev = ok.get(r["qa_id"])
            if prev is None or (r.get("answered_at") or "") < (prev.get("answered_at") or ""):
                ok[r["qa_id"]] = r
        else:
            failed.setdefault(r["qa_id"], []).append(r)
    for q in list(failed):
        if q in ok:
            del failed[q]
    return ok, failed, video_meta




# ----------------------------------------------------------------------------------------------- frames
def sample_indices(total_frames, num_frames):
    """The model wrapper's sampling (lmms-eval cambrian_p.load_video): np.linspace(0, total-1, k, dtype=int)."""
    import numpy as np
    if total_frames <= 0:
        raise ValueError("video has no frames")
    return [int(i) for i in np.linspace(0, total_frames - 1, num_frames, dtype=int)]


def frame_name(position, frame_index, seconds):
    """Input-frame file name keeping position, frame index and timestamp as separate fields."""
    return f"in{position:03d}_frame{frame_index:06d}_t{seconds:09.3f}s.jpg"


def parse_frame_name(name):
    m = re.fullmatch(r"(in|x)(\d{3})_frame(\d{6})_t(\d+\.\d{3})s\.jpg", name)
    if not m:
        return None
    return {"set": "input" if m.group(1) == "in" else "extra", "position": int(m.group(2)),
            "frame_index": int(m.group(3)), "seconds": float(m.group(4))}




# ----------------------------------------------------------------------------------------------- report
def esc(s):
    return html.escape("" if s is None else str(s), quote=True)


def rel_link(target_path, from_dir):
    return os.path.relpath(target_path, from_dir).replace(os.sep, "/")




# ----------------------------------------------------------------------------------------------- run identity
IDENTITY_KEYS = ("DATASET_REPO", "DATASET_CONFIG", "DATASET_SPLIT", "DATASET_REVISION", "SAMPLING_UNIT",
                 "SUBSET_FRACTION", "SEED", "MODEL_REPO", "MODEL_REVISION", "CODE_COMMIT", "NUM_FRAMES",
                 "PROMPT_POLICY", "GENERATION", "PREPROCESSING", "SCORER")


def config_identity(settings):
    """Hash of every setting that changes the sample, the model inputs/outputs or the scores. Run mode excluded."""
    missing = [k for k in IDENTITY_KEYS if k not in settings]
    if missing:
        raise ValueError(f"config_identity: missing {missing}")
    return stable_hash({k: settings[k] for k in IDENTITY_KEYS}, 16)


def check_run_config(path, identity_settings):
    """Create run_config.json, or verify the existing one. Raises on any change to the run's identity."""
    current = {"identity": identity_settings, "config_hash": config_identity(identity_settings)}
    if os.path.exists(path):
        saved = json.load(open(path))
        if saved.get("config_hash") != current["config_hash"]:
            diff = {k: (saved["identity"].get(k), v) for k, v in identity_settings.items() if saved["identity"].get(k) != v}
            raise RuntimeError("These settings differ from the ones this RUN_ID was started with (saved -> now): "
                               f"{json.dumps(diff, default=str)}. Restore them, or set a new RUN_ID for a new run.")
        return saved
    write_json_atomic(path, current)
    return current


def bind_selection(path, selection_hash):
    """Record the frozen selection's hash in run_config.json once; afterwards a different selection is refused."""
    saved = json.load(open(path))
    if saved.get("selection_hash") not in (None, selection_hash):
        raise RuntimeError(f"run_config.json is bound to selection {saved['selection_hash']}, not {selection_hash}. "
                           "Results of one selection are never mixed with another; use a new RUN_ID.")
    if saved.get("selection_hash") is None:
        saved["selection_hash"] = selection_hash
        write_json_atomic(path, saved)
    return saved


# ----------------------------------------------------------------------------------------------- official code
_OFFICIAL_NAMES = {"MCA_QUESTION_TYPES", "NA_QUESTION_TYPES", "METRICS_FOR_MCA", "METRICS_FOR_NA",
                   "WORST_CASE_FOR_METRICS"}
_OFFICIAL_FUNCS = {"vsibench_doc_to_text", "fuzzy_matching", "exact_match", "abs_dist_norm",
                   "mean_relative_accuracy", "to_float", "vsibench_process_results", "vsibench_aggregate_results"}


class _Logger:
    def info(self, *a, **k):
        pass

    error = warning = info


def load_official(repo_dir):
    """VSI-Bench's prompt formatter, scorer and aggregator, executed from the pinned lmms-eval file itself.

    Only the module's constants and functions are run (its import-time YAML/cache-path lines need a dataset cache and
    are skipped); nothing is edited. Returns a namespace dict plus the file's SHA-256 and the task YAML's prompt and
    generation settings."""
    import numpy as np
    path = os.path.join(repo_dir, OFFICIAL_SCORER_PATH)
    src = open(path).read()
    tree = ast.parse(src)
    keep = [n for n in tree.body
            if (isinstance(n, ast.FunctionDef) and n.name in _OFFICIAL_FUNCS)
            or (isinstance(n, ast.Assign) and all(isinstance(t, ast.Name) and t.id in _OFFICIAL_NAMES for t in n.targets))]
    found = {n.name for n in keep if isinstance(n, ast.FunctionDef)} | {t.id for n in keep if isinstance(n, ast.Assign) for t in n.targets}
    missing = (_OFFICIAL_FUNCS | _OFFICIAL_NAMES) - found
    if missing:
        raise RuntimeError(f"{path}: official definitions not found: {sorted(missing)}")
    ns = {"np": np, "partial": partial, "OrderedDict": OrderedDict, "eval_logger": _Logger(), "os": os}
    try:
        import pandas as pd
        ns["pd"] = pd
    except ImportError:
        pass
    exec(compile(ast.Module(body=keep, type_ignores=[]), path, "exec"), ns)
    import yaml
    ypath = os.path.join(repo_dir, OFFICIAL_TASK_YAML)
    cfg = yaml.safe_load("".join(l for l in open(ypath) if "!function" not in l))
    return {"ns": ns, "sha256": sha256_file(path), "yaml_sha256": sha256_file(ypath),
            "prompt_kwargs": cfg["lmms_eval_specific_kwargs"]["default"], "generation_kwargs": cfg["generation_kwargs"],
            "dataset_path": cfg["dataset_path"], "test_split": cfg["test_split"], "path": path}


def official_prompt(official, row):
    """The exact text the benchmark gives the model (after the image token the model wrapper prepends)."""
    doc = {k: row[k] for k in ("question", "question_type", "options")}
    return official["ns"]["vsibench_doc_to_text"](doc, dict(official["prompt_kwargs"]))


# ----------------------------------------------------------------------------------------------- annotations
def check_annotation_row(r, official):
    """Schema problems of one real VSI-Bench row (empty list = valid)."""
    p = []
    for k in ANNOTATION_FIELDS:
        if k not in r:
            p.append(f"missing field {k}")
    if p:
        return p
    if not isinstance(r["id"], int):
        p.append("id is not an int")
    if r["dataset"] not in VIDEO_DATASETS:
        p.append(f"unknown dataset {r['dataset']!r}")
    if not isinstance(r["scene_name"], str) or not r["scene_name"] or "/" in r["scene_name"] or ".." in r["scene_name"]:
        p.append("bad scene_name")
    ns = official["ns"]
    qt = r["question_type"]
    if qt in ns["MCA_QUESTION_TYPES"]:
        opts = r["options"]
        if not isinstance(opts, list) or len(opts) < 2 or not all(isinstance(o, str) and OPTION_RE.match(o) for o in opts):
            p.append("multiple-choice row without valid 'X. text' options")
        elif r["ground_truth"] not in option_letters(opts):
            p.append("reference letter is not one of the options")
    elif qt in ns["NA_QUESTION_TYPES"]:
        v = ns["to_float"](r["ground_truth"])
        if v is None or not math.isfinite(v) or v == 0:
            p.append("numeric reference is not a finite non-zero number")
    else:
        p.append(f"unsupported question type {qt!r}")
    return p


def option_letters(options):
    return [OPTION_RE.match(o).group(1) for o in options or [] if OPTION_RE.match(o)]


def video_key(dataset, scene_name):
    """Canonical video identity: (dataset, scene_name), written as the archive member's folder path."""
    return f"{dataset}/{scene_name}"


def qa_id(native_id):
    """The benchmark's own question id, kept as-is (prefixed only to make file names self-describing)."""
    return f"vsibench-{int(native_id)}"


def normalize_row(r, official, pruned_ids=None):
    ns = official["ns"]
    family = "multiple_choice" if r["question_type"] in ns["MCA_QUESTION_TYPES"] else "numeric"
    return {"qa_id": qa_id(r["id"]), "native_id": r["id"], "video_key": video_key(r["dataset"], r["scene_name"]),
            "dataset": r["dataset"], "scene_name": r["scene_name"], "question_type": r["question_type"],
            "task_family": family, "question": r["question"], "options": r["options"],
            "reference_answer": str(r["ground_truth"]), "prompt": official_prompt(official, r),
            "pruned": None if pruned_ids is None else (r["id"] in pruned_ids)}


def prompt_excludes_reference(row, official):
    """The prompt cannot carry the reference: it equals the official formatter's output for a copy of the row whose
    reference is replaced by a sentinel, and the sentinel never appears in it."""
    probe = dict(row, ground_truth="@@REFERENCE-SENTINEL@@", reference_answer="@@REFERENCE-SENTINEL@@")
    rebuilt = official_prompt(official, probe)
    return rebuilt == row["prompt"] and "@@REFERENCE-SENTINEL@@" not in rebuilt


# ----------------------------------------------------------------------------------------------- selection
def select_videos(video_keys, fraction, seed):
    """ceil(fraction * V) distinct videos, uniform without replacement, with a local RNG over the SORTED unique keys
    (so input order and duplicate rows cannot change the result). Returns the selected keys, sorted."""
    population = sorted(set(video_keys))
    k = math.ceil(fraction * len(population))
    return sorted(random.Random(seed).sample(population, k))


def selection_hash(keys):
    return hashlib.sha256("\n".join(sorted(keys)).encode()).hexdigest()


def group_by_video(qa_rows):
    groups = {}
    for r in qa_rows:
        groups.setdefault(r["video_key"], []).append(r)
    return groups


def pilot_subset(qa_rows, member_bytes, n_videos=2, max_qa=10):
    """Deterministic pilot from the frozen selection: the n smallest selected videos, up to max_qa questions."""
    groups = group_by_video(qa_rows)
    size = lambda v: float("inf") if member_bytes.get(v) is None else member_bytes[v]   # missing media sorts last
    order = sorted(groups, key=lambda v: (size(v), v))
    chosen, out = order[:n_videos], []
    per_video = max(1, max_qa // max(1, len(chosen)))
    for v in chosen:
        out += sorted(groups[v], key=lambda r: r["native_id"])[:per_video]
    return out[:max_qa]


# ----------------------------------------------------------------------------------------------- media (zip archives)
def archive_for(video_key_):
    """VSI-Bench stores every video at <dataset>/<scene_name>.mp4 inside <dataset>.zip (verified for all 288)."""
    dataset = video_key_.split("/")[0]
    return f"{dataset}.zip", video_key_ + ".mp4"


class RangeNotSupported(IOError):
    """The server answered a Range request with the whole file (so random access is not available)."""


class HttpRangeFile(io.RawIOBase):
    """A read-only, seekable view of a remote file through HTTP Range requests, so zipfile can read the central
    directory and single members without downloading the archive. Raises if the server ignores Range."""

    def __init__(self, url, size, headers=None, timeout=300):
        self.url, self.size, self.pos, self.headers, self.timeout = url, size, 0, dict(headers or {}), timeout
        self.requests = self.bytes_read = 0
        req = urllib.request.Request(url, headers=dict(self.headers, Range="bytes=0-0"))
        with urllib.request.urlopen(req, timeout=timeout) as r:      # probe once, before zipfile hides the cause
            if r.status != 206:
                raise RangeNotSupported(f"server ignored the Range header (HTTP {r.status})")

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = offset if whence == 0 else self.pos + offset if whence == 1 else self.size + offset
        return self.pos

    def readinto(self, b):
        n = min(len(b), self.size - self.pos)
        if n <= 0:
            return 0
        req = urllib.request.Request(self.url, headers=dict(self.headers, Range=f"bytes={self.pos}-{self.pos + n - 1}"))
        for attempt in range(4):
            try:
                with urllib.request.urlopen(req, timeout=self.timeout) as r:
                    if r.status != 206:
                        raise RangeNotSupported(f"server ignored the Range header (HTTP {r.status})")
                    data = r.read()
                break
            except RangeNotSupported:
                raise
            except OSError:
                if attempt == 3:
                    raise
                time.sleep(2 ** attempt)
        b[:len(data)] = data
        self.pos += len(data)
        self.requests += 1
        self.bytes_read += len(data)
        return len(data)


def open_zip(fileobj):
    return zipfile.ZipFile(io.BufferedReader(fileobj, 1 << 20) if isinstance(fileobj, HttpRangeFile) else fileobj)


def zip_listing(zf):
    return {i.filename: {"bytes": i.file_size, "compressed": i.compress_size, "crc32": i.CRC}
            for i in zf.infolist() if not i.is_dir()}


def extract_zip_member(zf, member, dest_root, expected_bytes=None, expected_crc=None):
    """Write one member under dest_root (safe path), verifying size and CRC (zipfile raises on a CRC mismatch)."""
    info = zf.getinfo(member)
    if expected_bytes is not None and info.file_size != expected_bytes:
        raise IOError(f"{member}: archive says {info.file_size} bytes, manifest says {expected_bytes}")
    if expected_crc is not None and info.CRC != expected_crc:
        raise IOError(f"{member}: archive CRC {info.CRC} differs from the manifest's {expected_crc}")
    dest = safe_extract_path(dest_root, member)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    tmp = dest + ".part"
    with zf.open(info) as src, open(tmp, "wb") as out:
        while True:
            block = src.read(1 << 20)
            if not block:
                break
            out.write(block)
    if os.path.getsize(tmp) != info.file_size:
        os.remove(tmp)
        raise IOError(f"{member}: wrote {os.path.getsize(tmp)} bytes, expected {info.file_size}")
    os.replace(tmp, dest)
    return dest


def file_crc32(path):
    crc = 0
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            crc = zipfile.crc32(block, crc)
    return crc & 0xFFFFFFFF


# ----------------------------------------------------------------------------------------------- scoring
def score_answer(row, raw_answer, official, status="ok"):
    """Official score of one answer plus the automatic status used to find inspection candidates.

    official_score is exactly vsibench_process_results' value; automatic_status never changes it."""
    ns = official["ns"]
    out = {"qa_id": row["qa_id"], "native_id": row["native_id"], "video_key": row["video_key"],
           "dataset": row["dataset"], "question_type": row["question_type"], "task_family": row["task_family"],
           "raw_answer": raw_answer, "reference_answer": row["reference_answer"], "parsed_answer": None,
           "parsed_value": None, "official_metric": None, "official_score": None, "candidate_reason": None,
           "error": None, "scorer": {"file": OFFICIAL_SCORER_PATH, "sha256": official["sha256"]}}
    qt = row["question_type"]
    if status != "ok":
        out["automatic_status"] = "execution_error"
        return out
    if qt in ns["MCA_QUESTION_TYPES"]:
        metric = "accuracy"
    elif qt in ns["NA_QUESTION_TYPES"]:
        metric = "MRA:.5:.95:.05"
    else:
        out.update(automatic_status="unscorable", error=f"unsupported question type {qt!r}")
        return out
    doc = {"question_type": qt, "ground_truth": row["reference_answer"], "options": row.get("options"),
           "question": row.get("question")}
    try:
        res = ns["vsibench_process_results"](dict(doc), [raw_answer])["vsibench_score"]
    except Exception as exc:
        out.update(automatic_status="unscorable", error=f"official scorer raised {type(exc).__name__}: {exc}")
        return out
    score = float(res[metric])
    parsed = ns["fuzzy_matching"](raw_answer)
    out.update(official_metric=metric, official_score=score, parsed_answer=parsed)
    if metric == "accuracy":
        if row["reference_answer"] not in option_letters(row.get("options")):
            out.update(automatic_status="unscorable", error="reference letter is not one of the options")
        elif score == 1.0:
            out["automatic_status"] = "correct"
        elif parsed.upper() not in option_letters(row.get("options")):
            out.update(automatic_status="format_error",
                       candidate_reason=f"official parse {parsed!r} is not one of the option letters")
        else:
            out.update(automatic_status="wrong_candidate",
                       candidate_reason=f"official exact match 0: chose {parsed.upper()}, reference {row['reference_answer']}")
        return out
    ref = ns["to_float"](row["reference_answer"])
    value = ns["to_float"](parsed)
    if ref is None or not math.isfinite(ref) or ref == 0:
        out.update(automatic_status="unscorable", error="numeric reference is not a finite non-zero number")
    elif value is None or not math.isfinite(value):
        out.update(automatic_status="format_error",
                   candidate_reason=f"official parse {parsed!r} is not a number (official score {score:g})")
    else:
        out["parsed_value"] = value
        if score == 1.0:
            out["automatic_status"] = "correct"
        elif score == 0.0:
            out.update(automatic_status="wrong_candidate",
                       candidate_reason=f"official MRA 0: {value:g} vs reference {ref:g} (relative error "
                                        f"{abs(value - ref) / abs(ref):.0%})")
        else:
            out.update(automatic_status="imperfect",
                       candidate_reason=f"official MRA {score:.2f} (partial credit): {value:g} vs {ref:g}")
    return out


def official_subset_scores(scores, official):
    """The official aggregator over the scored questions of the selected subset (macro average over question types,
    as lmms-eval reports). Only meaningful as 'selected subset' accuracy; never a full-benchmark score."""
    rows = []
    for s in scores:
        if s.get("official_score") is None:
            continue
        rows.append({"question_type": s["question_type"], s["official_metric"]: s["official_score"]})
    if not rows:
        return None
    res = official["ns"]["vsibench_aggregate_results"](rows)
    return {k: v for k, v in res.items() if not k.startswith("tabulated")}

`vsibench_pipeline.py`

In [ ]:
%%writefile /content/vsibench/vsibench_pipeline.py
"""Orchestration for the VSI-Bench video failure audit: annotation discovery, the frozen video sample, media fetch,
inference batches, merging and official scoring. Network access and the GPU worker are passed in, so the flow can be
tested offline. Review, verification, export and the report live in vsibench_review.py."""
import fnmatch
import json
import os
import time
import zipfile

import vsibench_lib as L


# ----------------------------------------------------------------------------------------------- 3. discovery
def card_config_files(card_text, config, split, repo_files):
    """Files the dataset card's YAML assigns to (config, split), matched against the real file list."""
    import yaml
    if not card_text.startswith("---"):
        raise RuntimeError("dataset card has no YAML header")
    meta = yaml.safe_load(card_text.split("---", 2)[1])
    cfg = next((c for c in meta.get("configs", []) if c.get("config_name") == config), None)
    if cfg is None:
        raise RuntimeError(f"config {config!r} not in the card; available: {[c.get('config_name') for c in meta.get('configs', [])]}")
    patterns = [d["path"] for d in cfg["data_files"] if d.get("split") == split]
    if not patterns:
        raise RuntimeError(f"config {config!r} has no split {split!r}")
    files = sorted({f for f in repo_files for p in patterns for pat in ([p] if isinstance(p, str) else p)
                    if fnmatch.fnmatch(f, pat)})
    return {"config": config, "split": split, "patterns": patterns, "files": files,
            "configs": {c["config_name"]: c["data_files"] for c in meta.get("configs", [])}}


def read_parquet_rows(paths):
    """Rows of the config's parquet files, with numpy arrays/NaN turned into plain lists/None."""
    import pandas as pd
    rows = []
    for p in paths:
        df = pd.read_parquet(p)
        for rec in df.to_dict("records"):
            opts = rec.get("options")
            rec["options"] = None if opts is None or (isinstance(opts, float) and opts != opts) else [str(o) for o in opts]
            rec["id"] = int(rec["id"])
            if "pruned" in rec and rec["pruned"] is not None:
                rec["pruned"] = bool(rec["pruned"])
            rows.append(rec)
    return rows


def cross_check(config_rows, jsonl_rows):
    """The config's rows must be exactly test.jsonl's rows (same ids, same field values)."""
    a = {r["id"]: r for r in config_rows}
    b = {r["id"]: r for r in jsonl_rows}
    problems = []
    if len(a) != len(config_rows):
        problems.append("duplicate ids in the config's rows")
    if set(a) != set(b):
        problems.append(f"id sets differ: {len(set(a) - set(b))} only in the config, {len(set(b) - set(a))} only in test.jsonl")
    diff = [i for i in set(a) & set(b) if any(a[i].get(k) != b[i].get(k) for k in L.ANNOTATION_FIELDS)]
    if diff:
        problems.append(f"{len(diff)} rows differ in content, e.g. id {sorted(diff)[0]}")
    return problems


def profile(raw_rows):
    vids = {}
    prof = {"rows": len(raw_rows), "fields": {}, "datasets": {}, "question_types": {}}
    for r in raw_rows:
        ks = ",".join(sorted(r))
        prof["fields"][ks] = prof["fields"].get(ks, 0) + 1
        prof["question_types"][r.get("question_type")] = prof["question_types"].get(r.get("question_type"), 0) + 1
        vids.setdefault(L.video_key(r.get("dataset"), r.get("scene_name")), 0)
        vids[L.video_key(r.get("dataset"), r.get("scene_name"))] += 1
    for v in vids:
        d = v.split("/")[0]
        prof["datasets"][d] = prof["datasets"].get(d, 0) + 1
    prof["videos"] = len(vids)
    prof["questions_per_video"] = {"min": min(vids.values()), "max": max(vids.values()),
                                   "mean": round(len(raw_rows) / len(vids), 2)} if vids else {}
    return prof


def schema_checks(raw_rows, qa_rows, official):
    """Problems that must be fixed before selection (an empty list means the adapter is consistent)."""
    problems = []
    bad = [(r.get("id"), p) for r in raw_rows for p in L.check_annotation_row(r, official)]
    if bad:
        problems.append(f"{len(bad)} invalid rows, e.g. {bad[:3]}")
    if len({r["id"] for r in raw_rows}) != len(raw_rows):
        problems.append("native question ids are not unique")
    if len({q["qa_id"] for q in qa_rows}) != len(qa_rows):
        problems.append("qa ids are not unique")
    scenes = {}
    for q in qa_rows:
        scenes.setdefault(q["scene_name"], set()).add(q["dataset"])
    clash = [s for s, d in scenes.items() if len(d) > 1]
    if clash:
        problems.append(f"scene names shared by several datasets (video identity keeps dataset): {clash[:3]}")
    leak = [q["qa_id"] for q in qa_rows if not L.prompt_excludes_reference(q, official)]
    if leak:
        problems.append(f"{len(leak)} prompts depend on the reference answer")
    selfscore = [q["qa_id"] for q in qa_rows if L.score_answer(q, q["reference_answer"], official)["automatic_status"] != "correct"]
    if selfscore:
        problems.append(f"{len(selfscore)} references do not score 1.0 against themselves under the official scorer")
    return problems


# ----------------------------------------------------------------------------------------------- 4. selection
def freeze_selection(out_root, qa_rows, settings, annotation_hashes):
    """Sample videos once and persist; on a later run, recompute and verify instead of sampling again."""
    sel_path = os.path.join(out_root, "selection.json")
    population = sorted({q["video_key"] for q in qa_rows})
    keys = L.select_videos(population, settings["SUBSET_FRACTION"], settings["SEED"])
    chosen = sorted((q for q in qa_rows if q["video_key"] in set(keys)), key=lambda q: (q["video_key"], q["native_id"]))
    qa_hash = L.stable_hash([q["qa_id"] for q in chosen])
    identity = {k: settings[k] for k in ("DATASET_REPO", "DATASET_CONFIG", "DATASET_SPLIT", "DATASET_REVISION")}
    sel = {"dataset": identity, "annotation_sha256": annotation_hashes, "sampling_unit": L.SAMPLING_UNIT,
           "population_videos": len(population), "population_questions": len(qa_rows),
           "population_hash": L.selection_hash(population), "fraction": settings["SUBSET_FRACTION"],
           "seed": settings["SEED"], "selected_videos": len(keys), "selected_video_keys": keys,
           "selection_hash": L.selection_hash(keys), "selected_questions": len(chosen),
           "selected_qa_ids_hash": qa_hash,
           "method": "random.Random(seed).sample(sorted(unique (dataset, scene_name)), ceil(fraction * V)); "
                     "every question of each selected video is kept"}
    if os.path.exists(sel_path):
        saved = json.load(open(sel_path))
        diff = {k: (saved.get(k), v) for k, v in sel.items() if saved.get(k) != v}
        if diff:
            raise RuntimeError(f"The saved selection does not match the current data/settings: {json.dumps(diff, default=str)[:600]}")
        qa, _ = L.read_jsonl(os.path.join(out_root, "qa_manifest.jsonl"), tolerate_truncated_tail=False)
        if [q["qa_id"] for q in qa] != [q["qa_id"] for q in chosen] or qa != chosen:
            raise RuntimeError("qa_manifest.jsonl differs from the selection recomputed from the pinned annotations")
        return saved, qa, False
    sel["created"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    L.write_jsonl_atomic(os.path.join(out_root, "qa_manifest.jsonl"), chosen)
    saved = dict(sel)
    L.write_json_atomic(sel_path, saved)       # written last: its presence means the selection is complete
    saved.pop("created")
    return sel, chosen, True


# ----------------------------------------------------------------------------------------------- 5. media
def media_manifest(selected_keys, listings, archive_sizes):
    """One row per selected video: archive, member and the member's size/CRC from the archive's central directory.
    A selected video missing from its archive stays in the manifest (member_missing) and in coverage."""
    rows = []
    for v in sorted(selected_keys):
        archive, member = L.archive_for(v)
        info = (listings.get(archive) or {}).get(member)
        d, s = v.split("/", 1)
        rows.append({"video_key": v, "dataset": d, "scene_name": s, "safe_key": L.safe_key(v), "archive": archive,
                     "archive_bytes": archive_sizes.get(archive), "member": member,
                     "member_bytes": info["bytes"] if info else None, "crc32": info["crc32"] if info else None,
                     "member_missing": info is None})
    return rows


def transfer_plan(manifest, archive_sizes):
    plan = []
    for a in sorted({m["archive"] for m in manifest}):
        ms = [m for m in manifest if m["archive"] == a]
        plan.append({"archive": a, "archive_bytes": archive_sizes.get(a), "videos": len(ms),
                     "selected_bytes": sum(m["member_bytes"] or 0 for m in ms),
                     "missing": [m["video_key"] for m in ms if m["member_missing"]]})
    return plan


# ----------------------------------------------------------------------------------------------- 6. results
def gather_results(shard_dir, journal_path, config_hash):
    shard_rows, report = L.load_shards(shard_dir, config_hash)
    journal_rows, truncated = L.read_jsonl(journal_path)
    ok, failed, meta = L.merge_results(shard_rows + journal_rows, config_hash)
    report["journal_truncated_lines"] = truncated
    return ok, failed, meta, report


def run_selection(qa_subset, manifest_by_video, ctx, log=print):
    """Answer every pending question in qa_subset, video by video, in bounded media batches.

    ctx: fetch_video(video_key, dest_dir) -> local path (raises KeyError if the member is absent from its archive,
    zipfile.BadZipFile/IOError on corrupt data); run_worker(batch_path, extra_args) -> exit code; release(video_key,
    path); media_dir, work_dir, shard_dir, journal, config_hash, batch_bytes; log_missing(video_key, archive, member,
    reason); skip_videos (set already logged as missing/corrupt)."""
    ok, failed, _, _ = gather_results(ctx["shard_dir"], ctx["journal"], ctx["config_hash"])
    skip = set(ctx.get("skip_videos", ()))
    by_video = L.group_by_video([q for q in qa_subset if q["qa_id"] not in ok and q["video_key"] not in skip])
    if not by_video:
        log("Nothing pending: every question in this set has a verified answer (or its video is logged as missing).")
        return {"pending_videos": 0, "processed_videos": 0, "missing": []}
    summary = {"pending_videos": len(by_video), "processed_videos": 0, "missing": []}
    batch, size = [], 0

    def flush():
        nonlocal batch, size
        if not batch:
            return 0
        bpath = os.path.join(ctx["work_dir"], f"batch-{int(time.time() * 1000)}.json")
        L.write_json_atomic(bpath, {"videos": batch})
        log(f"  running the model on {len(batch)} video(s), {sum(len(b['qa']) for b in batch)} question(s) ...")
        code = ctx["run_worker"](bpath, [])
        for b in batch:
            ctx["release"](b["video_key"], b["local_path"])
        summary["processed_videos"] += len(batch)
        batch, size = [], 0
        return code

    for v in sorted(by_video):
        m = manifest_by_video[v]
        if m.get("member_missing"):
            ctx["log_missing"](v, m["archive"], m["member"], "not in its archive")
            summary["missing"].append(v)
            continue
        try:
            path = ctx["fetch_video"](v, ctx["media_dir"])
        except (KeyError, zipfile.BadZipFile) as exc:
            ctx["log_missing"](v, m["archive"], m["member"], f"{type(exc).__name__}: {exc}")
            summary["missing"].append(v)
            continue
        batch.append({"video_key": v, "safe_key": m["safe_key"], "local_path": path,
                      "member_bytes": os.path.getsize(path), "qa": by_video[v]})
        size += os.path.getsize(path)
        if size >= ctx["batch_bytes"]:
            code = flush()
            if code:
                log(f"  worker exited with code {code}; stopping so the cause can be checked. Rerun to resume.")
                summary["stopped"] = code
                return summary
    code = flush()
    if code:
        log(f"  worker exited with code {code}; stopping so the cause can be checked. Rerun to resume.")
        summary["stopped"] = code
    return summary


def missing_media(path):
    rows, _ = L.read_jsonl(path)
    return {r["video_key"]: r.get("reason") for r in rows}


def log_missing_media(path, video_key, archive, member, reason):
    L.Journal(path).append({"video_key": video_key, "archive": archive, "member": member, "reason": reason,
                            "logged": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})


# ----------------------------------------------------------------------------------------------- 7. scoring
def score_all(qa, ok, failed, missing, official):
    """One score row per selected question that has an outcome; questions with none are pending (no row)."""
    scores = []
    for q in qa:
        if q["qa_id"] in ok:
            s = L.score_answer(q, ok[q["qa_id"]]["raw_answer"], official)
        elif q["video_key"] in missing:
            s = L.score_answer(q, None, official, status="error")
            s["error"] = f"media unavailable: {missing[q['video_key']]}"
        elif q["qa_id"] in failed:
            s = L.score_answer(q, None, official, status="error")
            s["error"] = failed[q["qa_id"]][-1].get("error")
        else:
            continue
        scores.append(s)
    return scores


def coverage(sel, qa, ok, failed, scores, missing, mode, official):
    by_video = L.group_by_video(qa)
    scored = {s["qa_id"]: s for s in scores}
    v_state = {}
    for v, qs in by_video.items():
        if all(q["qa_id"] in ok for q in qs):
            v_state[v] = "completed"
        elif any(q["qa_id"] not in scored for q in qs):
            v_state[v] = "pending"
        else:
            v_state[v] = "failed"          # every question has an outcome, at least one is an execution error
    status = {}
    for s in scores:
        status[s["automatic_status"]] = status.get(s["automatic_status"], 0) + 1
    count = lambda st: sum(1 for x in v_state.values() if x == st)
    pending_qa = len(qa) - len(scores)
    return {
        "run_mode_last": mode,
        "dataset": sel["dataset"],
        "full_dataset": {"videos": sel["population_videos"], "questions": sel["population_questions"]},
        "selected": {"videos": sel["selected_videos"], "questions": sel["selected_questions"],
                     "video_fraction": round(sel["selected_videos"] / sel["population_videos"], 4),
                     "question_fraction": round(sel["selected_questions"] / sel["population_questions"], 4),
                     "fraction_setting": sel["fraction"], "seed": sel["seed"], "sampling_unit": sel["sampling_unit"]},
        "videos": {"completed": count("completed"), "failed": count("failed"), "pending": count("pending")},
        "questions": {"answered": len(ok), "execution_error": status.get("execution_error", 0), "pending": pending_qa},
        "missing_or_corrupt_media": missing,
        "automatic_status": status,
        "inference_complete": pending_qa == 0,
        "official_subset_scores": {
            "scope": f"selected subset only ({len([s for s in scores if s.get('official_score') is not None])} scored "
                     "questions); official lmms-eval aggregation (macro average over question types); not a "
                     "full-benchmark score",
            "scores": L.official_subset_scores(scores, official)},
    }


def write_score_outputs(out_root, ok, failed, scores, qa_by_id):
    """predictions.jsonl, errors.jsonl, scores.jsonl (failure_candidates.csv is written with the review queue, 8a)."""
    L.write_jsonl_atomic(os.path.join(out_root, "predictions.jsonl"), [ok[k] for k in sorted(ok)])
    L.write_jsonl_atomic(os.path.join(out_root, "errors.jsonl"), [r for k in sorted(failed) for r in failed[k]])
    L.write_jsonl_atomic(os.path.join(out_root, "scores.jsonl"), scores)

`vsibench_worker.py`

In [ ]:
%%writefile /content/vsibench/vsibench_worker.py
"""Cambrian-P inference worker for the VSI-Bench video failure audit. Runs inside the Cambrian-P venv.

Loads the model once, then for each video in the batch file decodes the sampled frames once and answers that
video's selected questions with the official VSI-Bench prompt (built in the notebook, stored in qa_manifest). Every answer is appended (fsync) to a local journal at once; after each video its rows
are written to a durable shard with a completion marker. Model loading and preprocessing follow the lmms-eval cambrian_p wrapper;
the question text is the official vsibench_doc_to_text output.
"""
import argparse
import json
import os
import sys
import time
import traceback

os.environ["MPLBACKEND"] = "Agg"   # Colab's inline backend leaks in through the environment; this script saves files only
import numpy as np
import torch
from PIL import Image

ap = argparse.ArgumentParser()
ap.add_argument("--batch", required=True, help="JSON: {videos: [{video_key, safe_key, local_path, qa: [...]}, ...]}")
ap.add_argument("--model_path", required=True)
ap.add_argument("--num_frames", type=int, required=True)
ap.add_argument("--config_hash", required=True)
ap.add_argument("--identity", required=True, help="JSON of the run identity, copied into every row")
ap.add_argument("--journal", required=True)
ap.add_argument("--errors", required=True)
ap.add_argument("--shard_dir", required=True)
ap.add_argument("--status", required=True)
ap.add_argument("--lib_dir", required=True)
ap.add_argument("--max_retries", type=int, default=2)
ap.add_argument("--stop_after", type=int, default=0, help="test only: exit after this many new answers")
args = ap.parse_args()
sys.path.insert(0, args.lib_dir)
import vsibench_lib as L  # noqa: E402
import hashlib  # noqa: E402

identity = json.loads(args.identity)
GEN = dict(identity["GENERATION"])
journal, errors = L.Journal(args.journal), L.Journal(args.errors)
status = {"started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "answers": 0, "errors": 0, "videos": 0}


def save_status(**kw):
    status.update(kw)
    L.write_json_atomic(args.status, status)


# ---------------------------------------------------------------------------------------- model (from run_videos.py)
import cambrianp.utils as cambrianp_utils  # noqa: E402
from lmms_eval.models.simple.cambrian_p import CambrianP, expand2square  # noqa: E402
from cambrianp.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX  # noqa: E402
from cambrianp.conversation import conv_templates  # noqa: E402
from cambrianp.mm_utils import tokenizer_image_token  # noqa: E402


# Workaround kept from the source notebook, re-checked against the pinned code (b3c1552) and checkpoint:
# cambrianp/model/builder.py:230 still re-casts the config of any path containing "cambrian" to LLaVA-OneVision
# style, which overwrites vocab_size and deletes the camera-head keys; and the checkpoint's rec_model_path is
# JSON null, which VGGTReconstructor would try to torch.load. Keep the saved config, enable the camera head.
def _keep_checkpoint_config(config, model_path):
    config.load_rec_model = True
    if not os.path.isfile(str(getattr(config, "rec_model_path", None))):
        config.rec_model_path = "None"
    return config


cambrianp_utils.cast_cambrian_config_to_llava_ov_style = _keep_checkpoint_config
save_status(phase="loading model")
m = CambrianP(pretrained=args.model_path, model_name="llava_qwen", conv_template="qwen_1_5",
              max_frames_num=args.num_frames, use_camera_tokens=True, camera_tokens_mode="camera_tokens",
              camera_tokens_place="append_to_frame", query_mode="query_after_image")
pad_id = m.tokenizer.pad_token_id if m.tokenizer.pad_token_id is not None else m.tokenizer.eos_token_id

# Check the camera tokens and trained head really loaded (the failure mode the workaround exists for).
inner = m.model.get_model()
rec_head = getattr(inner, "rec_head", None)
cam = getattr(rec_head, "camera_tokens", None) if rec_head is not None else None
head_ok = rec_head is not None and cam is not None and float(cam.detach().float().abs().sum()) > 0
save_status(phase="model loaded", camera_head_loaded=bool(head_ok),
            camera_tokens_shape=list(cam.shape) if cam is not None else None,
            vocab_size=int(m.model.config.vocab_size), gpu=torch.cuda.get_device_name(0),
            torch=torch.__version__, cuda=torch.version.cuda)
if not head_ok:
    raise SystemExit("Camera tokens / trained head did not load: refusing to run (see the model log above).")


def ask(pixels, image_size, video, question):
    conv = conv_templates["qwen_1_5"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + question)
    conv.append_message(conv.roles[1], None)
    ids = tokenizer_image_token(conv.get_prompt(), m.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).to(m.device)
    return m._run_inference_step(ids, ids.ne(pad_id), [pixels], image_size, "video", dict(GEN), video)[0]


# ---------------------------------------------------------------------------------------- frames with provenance
def decode(path, k):
    """Decode exactly the frames the wrapper's load_video would sample, and say which they are."""
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(path, ctx=cpu(0), num_threads=0)
        total, fps = len(vr), float(vr.get_avg_fps())
        idx = L.sample_indices(total, k)
        frames = vr.get_batch(idx).asnumpy()
        reader = "decord"
    except Exception as exc:  # same fallback idea as the wrapper; recorded so it can be judged
        import av
        with av.open(path) as c:
            s = c.streams.video[0]
            all_frames = [f.to_ndarray(format="rgb24") for f in c.decode(s)]
            fps = float(s.average_rate or 0) or 30.0
        total = len(all_frames)
        idx = L.sample_indices(total, k)
        frames = np.stack([all_frames[i] for i in idx])
        reader = f"pyav (decord failed: {type(exc).__name__})"
    times = [i / fps for i in idx]
    # Fingerprint of every decoded input frame, so exported evidence can be proven to be the model's exact input.
    sha = [hashlib.sha1(np.ascontiguousarray(f).tobytes()).hexdigest() for f in frames]
    return frames, {"frame_indices": idx, "timestamps_s": times, "total_frames": total, "fps": fps, "reader": reader,
                    "num_frames": k, "frame_size_hw": list(frames.shape[1:3]), "frame_sha1": sha}


def preprocess(raw):
    bg = tuple(int(x * 255) for x in m._image_processor.image_mean)
    frames = [expand2square(Image.fromarray(f, mode="RGB"), bg) for f in raw]
    pixels = m._image_processor.preprocess(frames, return_tensors="pt")["pixel_values"].half().cuda()
    desc = (f"pad to square with mean color {bg} (expand2square), then the SigLIP image processor "
            f"(resize to {tuple(pixels.shape[-2:])}, normalize); fp16")
    return pixels, desc


# ---------------------------------------------------------------------------------------- main loop
batch = json.load(open(args.batch))
answered = 0
for vk, video in enumerate(batch["videos"], 1):
    rows = []
    save_status(phase=f"video {vk}/{len(batch['videos'])}", current=video["video_key"])
    t0 = time.time()
    try:
        raw, prov = decode(video["local_path"], args.num_frames)
        pixels, prov["preprocessing"] = preprocess(raw)
        image_size = list(raw.shape[1:3])
    except Exception:
        err = traceback.format_exc().strip().splitlines()[-1]
        for q in video["qa"]:
            e = {"kind": "answer", "qa_id": q["qa_id"], "video_key": video["video_key"], "status": "error",
                 "error": f"decode: {err}", "attempt": 1, "config_hash": args.config_hash,
                 "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
            errors.append(e); rows.append(e)
        L.write_shard(args.shard_dir, video["safe_key"] + f"-{int(time.time())}", rows, args.config_hash)
        status["errors"] += len(video["qa"])
        continue
    meta = dict({"kind": "video_meta", "video_key": video["video_key"], "safe_key": video["safe_key"],
                 "config_hash": args.config_hash, "decode_seconds": round(time.time() - t0, 2)}, **prov)
    rows.append(meta)
    journal.append(meta)
    m.model.rec_head_preds = None          # poses are cached per generate(); not used here, cleared per video
    for q in video["qa"]:
        last_err = None
        for attempt in range(1, args.max_retries + 2):
            t1 = time.time()
            # Only the model call is caught and retried. Saving the answer happens outside the try, so a bookkeeping
            # bug stops the worker at once with its traceback instead of being recorded as a model failure.
            try:
                answer = ask(pixels, image_size, video["local_path"], q["prompt"])
            except torch.cuda.OutOfMemoryError:
                e = {"kind": "answer", "qa_id": q["qa_id"], "video_key": video["video_key"], "status": "error",
                     "error": "CUDA out of memory", "attempt": attempt, "config_hash": args.config_hash,
                     "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
                errors.append(e); rows.append(e)
                L.write_shard(args.shard_dir, video["safe_key"] + f"-{int(time.time())}", rows, args.config_hash)
                save_status(phase="stopped: out of GPU memory")
                raise SystemExit(f"Out of GPU memory at {args.num_frames} frames on {video['video_key']}. Frames and "
                                 "model are not changed automatically: use a GPU with more memory, or start a new "
                                 "RUN_ID with different settings.")
            except Exception:
                last_err = traceback.format_exc().strip().splitlines()[-1]
                e = {"kind": "answer", "qa_id": q["qa_id"], "video_key": video["video_key"], "status": "error",
                     "error": last_err, "attempt": attempt, "config_hash": args.config_hash,
                     "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
                errors.append(e)
                if attempt == args.max_retries + 1:
                    rows.append(e)
                    status["errors"] += 1
                continue
            r = {"kind": "answer", "qa_id": q["qa_id"], "native_id": q["native_id"], "video_key": video["video_key"],
                 "status": "ok", "raw_answer": answer, "reference_answer": q["reference_answer"],
                 "attempt": attempt, "elapsed_s": round(time.time() - t1, 2), "config_hash": args.config_hash,
                 "identity": identity, "frame_provenance": "see video_meta row of this video",
                 "answered_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
            journal.append(r); rows.append(r)
            answered += 1
            status["answers"] += 1
            break
        if args.stop_after and answered >= args.stop_after:
            save_status(phase=f"stopped on purpose after {answered} answers (resume test)")
            sys.exit(75)  # the video's shard is NOT written: its answers live only in the journal
    L.write_shard(args.shard_dir, video["safe_key"] + f"-{int(time.time())}", rows, args.config_hash)
    status["videos"] += 1
    torch.cuda.empty_cache()
save_status(phase="done", finished=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))

`vsibench_frames.py`

In [ ]:
%%writefile /content/vsibench/vsibench_frames.py
"""Evidence frames for one VSI-Bench failure candidate. Runs in the Cambrian-P venv (decord, PyAV fallback).

input mode  -> decode check of the playable video (video_check.json); the recorded model-input frames at the
               exact indices the worker logged, each re-hashed and compared with the worker's frame_sha1 (so the
               saved images are verified reproductions of the model inputs, or labelled as unverified); full-
               resolution JPEGs, contact sheets, and frames_manifest.json.
extra mode  -> diagnostic frames outside the model input: --fps over the whole video and/or dense --window
               start,end,fps intervals. Kept in frames_extra/, listed in frames_manifest.json as frame_set
               "extra", with their own contact sheets. Repeated calls add to the existing extra set.
The frame hash is sha1 of the decoded RGB uint8 array's bytes (np.ascontiguousarray(frame).tobytes()), the same
definition the inference worker uses.
"""
import argparse
import hashlib
import json
import os
import sys
import time

import numpy as np
from PIL import Image, ImageDraw

ap = argparse.ArgumentParser()
ap.add_argument("--video", required=True)
ap.add_argument("--case_dir", required=True, help="the candidate's folder; frames go to frames_input/ and frames_extra/")
ap.add_argument("--lib_dir", required=True)
ap.add_argument("--mode", choices=["input", "extra"], required=True)
ap.add_argument("--provenance", help="input mode: JSON with frame_indices, timestamps_s, total_frames, fps, frame_sha1")
ap.add_argument("--fps", type=float, default=1.0, help="extra mode: frames per second over the whole video (0 = none)")
ap.add_argument("--window", action="append", default=[], help="extra mode: start,end,fps in seconds (repeatable)")
ap.add_argument("--max_extra", type=int, default=600)
args = ap.parse_args()
sys.path.insert(0, args.lib_dir)
import vsibench_lib as L  # noqa: E402

MANIFEST = os.path.join(args.case_dir, "frames_manifest.json")
IN_DIR, X_DIR = os.path.join(args.case_dir, "frames_input"), os.path.join(args.case_dir, "frames_extra")


def frame_sha1(frame):
    return hashlib.sha1(np.ascontiguousarray(frame).tobytes()).hexdigest()


def extra_name(frame_index, seconds):
    return f"x_frame{frame_index:06d}_t{seconds:09.3f}s.jpg"


def open_reader(path):
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(path, ctx=cpu(0), num_threads=0)
        return len(vr), float(vr.get_avg_fps()), lambda idx: vr.get_batch(list(idx)).asnumpy(), "decord"
    except Exception as exc:
        import av
        with av.open(path) as c:
            s = c.streams.video[0]
            frames = [f.to_ndarray(format="rgb24") for f in c.decode(s)]
            fps = float(s.average_rate or 0) or 30.0
        return len(frames), fps, lambda idx: np.stack([frames[i] for i in idx]), f"pyav (decord failed: {type(exc).__name__})"


def contact_sheets(items, out_dir, prefix, label):
    """items: [(caption, PIL thumbnail)]; returns sheet file names (32 per sheet, 8 columns)."""
    for f in os.listdir(out_dir):
        if f.startswith(prefix):
            os.remove(os.path.join(out_dir, f))
    names, per_sheet, cols = [], 32, 8
    for s in range(0, len(items), per_sheet):
        part = items[s:s + per_sheet]
        w, h = max(t.width for _, t in part), max(t.height for _, t in part) + 16
        sheet = Image.new("RGB", (cols * w, -(-len(part) // cols) * h), "white")
        draw = ImageDraw.Draw(sheet)
        for k, (cap, t) in enumerate(part):
            x, y = (k % cols) * w, (k // cols) * h
            sheet.paste(t, (x, y + 16))
            draw.text((x + 2, y + 2), cap, fill=(0, 0, 0))
        name = f"{prefix}{s // per_sheet + 1:02d}_{label}.jpg"
        sheet.save(os.path.join(out_dir, name), quality=88)
        names.append(name)
    return names


def thumb(f):
    t = Image.fromarray(f)
    t.thumbnail((240, 240))
    return t


def load_manifest():
    return json.load(open(MANIFEST)) if os.path.exists(MANIFEST) else {"entries": []}


try:
    total, fps, get, reader = open_reader(args.video)
    first_last = get([0, total - 1]) if total > 0 else None
    check = {"ok": total > 0 and first_last is not None and first_last.shape[0] == 2, "reader": reader,
             "total_frames": total, "fps": fps, "frame_size_hw": list(first_last.shape[1:3]) if total > 0 else None}
except Exception as exc:  # the video cannot be played back: recorded, never silently ignored
    total, check = 0, {"ok": False, "error": f"{type(exc).__name__}: {exc}"}
check.update(bytes=os.path.getsize(args.video) if os.path.exists(args.video) else None,
             sha256=L.sha256_file(args.video) if os.path.exists(args.video) else None,
             video=os.path.basename(args.video), checked_at=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
L.write_json_atomic(os.path.join(args.case_dir, "video_check.json"), check)
if not check["ok"]:
    print(json.dumps({"error": "decode check failed", **check}))
    sys.exit(3)

man = load_manifest()
if args.mode == "input":
    prov = json.load(open(args.provenance))
    idx, times = [int(i) for i in prov["frame_indices"]], [float(t) for t in prov["timestamps_s"]]
    recorded = prov.get("frame_sha1") or []
    if total != prov["total_frames"]:
        print(json.dumps({"error": f"frame count differs from the run ({total} vs {prov['total_frames']}); these would "
                                   "not be the frames the model saw"}))
        sys.exit(4)
    os.makedirs(IN_DIR, exist_ok=True)
    entries, items, mismatches = [], [], 0
    for start in range(0, len(idx), 16):
        for k, f in enumerate(get(idx[start:start + 16])):
            pos = start + k
            name = L.frame_name(pos, idx[pos], times[pos])
            Image.fromarray(f).save(os.path.join(IN_DIR, name), quality=92)
            h = frame_sha1(f)
            ok = pos < len(recorded) and recorded[pos] == h
            mismatches += 0 if ok else 1
            entries.append({"frame_set": "input", "position": pos, "file": f"frames_input/{name}", "frame_index": idx[pos],
                            "timestamp_s": round(times[pos], 3), "sha1": h, "verified": ok})
            items.append((f"#{pos} f{idx[pos]} {times[pos]:.2f}s", thumb(f)))
    sheets = contact_sheets(items, IN_DIR, "sheet_", "inputs")
    verified = bool(recorded) and mismatches == 0 and len(entries) == int(prov.get("num_frames", len(idx)))
    man.update({"video": os.path.basename(args.video), "fps": fps, "total_frames": total, "reader": reader,
                "recorded_reader": prov.get("reader"), "num_frames_expected": int(prov.get("num_frames", len(idx))),
                "preprocessing": prov.get("preprocessing"), "image_encoding": "JPEG quality 92 of the decoded frame",
                "inputs_verified": verified,
                "verification": ("sha1 of every re-decoded frame matches the worker's record" if verified else
                                 "no frame hashes were recorded by the run" if not recorded else
                                 f"{mismatches} of {len(entries)} frames differ from the worker's record"),
                "input_sheets": [f"frames_input/{s}" for s in sheets]})
    man["entries"] = entries + [e for e in man["entries"] if e["frame_set"] == "extra"]
    L.write_json_atomic(MANIFEST, man)
    print(json.dumps({"written": len(entries), "inputs_verified": verified, "reader": reader, "total_frames": total}))
else:
    want = {e["frame_index"] for e in man["entries"] if e["frame_set"] == "extra"}
    if args.fps > 0:
        want.update(range(0, total, max(1, int(round(fps / args.fps)))))
    for w in args.window:
        a, b, wf = (float(x) for x in w.split(","))
        want.update(range(max(0, int(a * fps)), min(total, int(b * fps) + 1), max(1, int(round(fps / wf)))))
    idx = sorted(want)
    capped = len(idx) > args.max_extra
    idx = idx[:args.max_extra]
    os.makedirs(X_DIR, exist_ok=True)
    entries, items = [], []
    for start in range(0, len(idx), 32):
        for k, f in enumerate(get(idx[start:start + 32])):
            fi = idx[start + k]
            name = extra_name(fi, fi / fps)
            if not os.path.exists(os.path.join(X_DIR, name)):
                Image.fromarray(f).save(os.path.join(X_DIR, name), quality=88)
            entries.append({"frame_set": "extra", "file": f"frames_extra/{name}", "frame_index": fi,
                            "timestamp_s": round(fi / fps, 3), "sha1": frame_sha1(f), "verified": None})
            items.append((f"f{fi} {fi / fps:.2f}s", thumb(f)))
    sheets = contact_sheets(items, X_DIR, "sheet_extra_", "extra")
    man.setdefault("fps", fps)
    man.setdefault("total_frames", total)
    man["extra_sheets"] = [f"frames_extra/{s}" for s in sheets]
    man["extra_requests"] = man.get("extra_requests", []) + [{"fps": args.fps, "windows": args.window,
                                                              "at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}]
    man["entries"] = [e for e in man["entries"] if e["frame_set"] == "input"] + entries
    L.write_json_atomic(MANIFEST, man)
    print(json.dumps({"written": len(entries), "sheets": len(sheets), "reader": reader, "capped": capped}))

`vsibench_review.py`

In [ ]:
%%writefile /content/vsibench/vsibench_review.py
"""Failure candidates, manual frame-evidence review, the verified-failure validator, final export and report.

Benchmark score and review verdict are kept apart: the official VSI-Bench score (scores.jsonl) is never changed
here; a reviewer's verdict is stored next to it. One function, validate_verified_failure(), decides whether a
case counts as a verified failure, and every count (UI, summary.csv, report.html, verified_failure_manifest.json,
completion gate) is computed from it, so they cannot disagree.
"""
import csv
import hashlib
import html
import json
import os
import re
import shutil
import subprocess
import time
import zipfile

import vsibench_lib as L

REVIEW_VERSION = "vsibench-review-1"
EVIDENCE_STATUSES = ("sufficient_in_input", "partial_in_input", "found_only_in_extra_frames",
                     "not_found_in_reviewed_frames", "ambiguous")
EVIDENCE_HELP = {
    "sufficient_in_input": "the reviewed model inputs support determining the answer",
    "partial_in_input": "some needed evidence is visible in the inputs, but coverage or interpretation is incomplete",
    "found_only_in_extra_frames": "supporting evidence is found only outside the model's sampled inputs",
    "not_found_in_reviewed_frames": "the search did not find sufficient evidence (not proof it is absent from the video)",
    "ambiguous": "the available images cannot establish the answer reliably",
}
FINAL_VERDICTS = ("wrong", "correct", "uncertain")
CONFIDENCE = ("low", "medium", "high")
# What a confirmed wrong answer is, given the evidence; only a documented-in-input case is a visual-reasoning error.
FINAL_CATEGORIES = {
    "visual_reasoning_error": ("sufficient_in_input", "partial_in_input"),
    "evidence_not_in_input": ("found_only_in_extra_frames",),
    "answer_failure_unresolved_evidence": ("not_found_in_reviewed_frames", "ambiguous"),
    "format_failure": EVIDENCE_STATUSES,
    "other": EVIDENCE_STATUSES,
}
CANDIDATE_STATUSES = ("wrong_candidate", "format_error")     # queued for review (primary QAs)
INSPECTION_STATUSES = ("wrong_candidate", "format_error", "imperfect")   # official score below its maximum
VIDEO_STATES = ("verified_wrong", "invalid_or_missing_evidence", "overridden_correct", "uncertain",
                "provisional_only", "unreviewed")


def now():
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


def _h(x):
    return hashlib.sha1(str(x).encode()).hexdigest()


def _blank(s):
    return not isinstance(s, str) or len(s.strip()) < 3


# ============================================================================================ candidate queue
def candidate_entries(scores, qa_by_id, selected_video_keys, max_supporting=2):
    """One entry per selected video with at least one automatic failure candidate, in review-priority order.

    Primary QA: a wrong_candidate before a format_error, then the least-represented question type, then SHA-1.
    Video order: videos with a wrong_candidate first, then least-represented source dataset and question type,
    then SHA-1 of the video key. Deterministic; uses only the frozen selected sample."""
    selected = set(selected_video_keys)
    by_video = {}
    for s in scores:
        if s.get("automatic_status") in CANDIDATE_STATUSES and s["video_key"] in selected and s["qa_id"] in qa_by_id:
            by_video.setdefault(s["video_key"], []).append(s)
    tier = lambda s: CANDIDATE_STATUSES.index(s["automatic_status"])
    remaining, src_n, type_n, out = dict(by_video), {}, {}, []
    while remaining:
        def vkey(v):
            ss = remaining[v]
            return (min(tier(s) for s in ss), src_n.get(ss[0]["dataset"], 0),
                    min(type_n.get(s["question_type"], 0) for s in ss), _h(v))
        v = min(remaining, key=vkey)
        ss = sorted(remaining.pop(v), key=lambda s: (tier(s), type_n.get(s["question_type"], 0), _h(s["qa_id"])))
        p = ss[0]
        src_n[p["dataset"]] = src_n.get(p["dataset"], 0) + 1
        type_n[p["question_type"]] = type_n.get(p["question_type"], 0) + 1
        out.append({"video_key": v, "safe_key": L.safe_key(v), "dataset": p["dataset"],
                    "primary_qa_id": p["qa_id"], "primary_question_type": p["question_type"],
                    "supporting_qa_ids": [s["qa_id"] for s in ss[1:1 + max_supporting]],
                    "other_candidate_qa_ids": [s["qa_id"] for s in ss[1 + max_supporting:]],
                    "candidate_reason": p.get("candidate_reason") or p["automatic_status"],
                    "automatic_status": p["automatic_status"]})
    return out


def update_queue(path, scores, qa_by_id, selected_video_keys, target):
    """Persist the exact queue. Existing order is never reshuffled; videos that became candidates later (more
    inference) are appended in priority order. The first `target` are active; the rest are the reserve."""
    fresh = candidate_entries(scores, qa_by_id, selected_video_keys)
    q = json.load(open(path)) if os.path.exists(path) else {
        "created": now(), "target": target, "entries": [], "active": [], "promotions": [],
        "rule": "one primary QA per video (+ up to 2 supporting); wrong_candidate before format_error; then least-"
                "represented source and question type; ties by SHA-1. Only videos of the frozen selection."}
    known = {e["video_key"] for e in q["entries"]}
    q["entries"] += [e for e in fresh if e["video_key"] not in known]
    q["entries"] = [e for e in q["entries"] if e["video_key"] in set(selected_video_keys)]
    keys = [e["video_key"] for e in q["entries"]]
    for k in keys:
        if len(q["active"]) >= target:
            break
        if k not in q["active"]:
            q["active"].append(k)
    q["updated"] = now()
    L.write_json_atomic(path, q)
    return q


def queue_split(q):
    active = [e for e in q["entries"] if e["video_key"] in set(q["active"])]
    active.sort(key=lambda e: q["active"].index(e["video_key"]))
    reserve = [e for e in q["entries"] if e["video_key"] not in set(q["active"])]
    return active, reserve


def promote_reserve(path, q, states, target):
    """After overrides, activate reserve entries until `target` active videos are not resolved as not-wrong.
    Activation is permanent (history kept); returns the newly offered entries."""
    resolved = {k for k, s in states.items() if s in ("overridden_correct", "uncertain")}
    _, reserve = queue_split(q)
    offered = []
    while sum(1 for k in q["active"] if k not in resolved) < target and reserve:
        e = reserve.pop(0)
        q["active"].append(e["video_key"])
        q["promotions"].append({"video_key": e["video_key"], "at": now(),
                                "reason": f"{len(resolved)} active candidate(s) overridden as correct/uncertain"})
        offered.append(e)
    if offered:
        L.write_json_atomic(path, q)
    return offered


def reselect_primary(path, q, video_key, qa_id, candidate_qa_ids):
    """Switch a video's primary question to another of its candidate QAs (a fresh QA-specific review is needed)."""
    e = next(e for e in q["entries"] if e["video_key"] == video_key)
    if qa_id not in candidate_qa_ids:
        raise ValueError(f"{qa_id} is not a failure candidate of {video_key}")
    e.setdefault("primary_history", []).append({"from": e["primary_qa_id"], "to": qa_id, "at": now()})
    others = [x for x in [e["primary_qa_id"]] + e["supporting_qa_ids"] if x != qa_id]
    e["primary_qa_id"], e["supporting_qa_ids"] = qa_id, others[:2]
    L.write_json_atomic(path, q)
    return e


def write_failure_candidates(path, scores, qa_by_id):
    """Every inspection candidate (official score below its maximum), with the reason; not a failure count."""
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["qa_id", "native_id", "video_key", "dataset", "question_type", "automatic_status",
                    "official_metric", "official_score", "raw_answer", "parsed_answer", "reference_answer",
                    "candidate_reason"])
        for s in scores:
            if s.get("automatic_status") in INSPECTION_STATUSES:
                w.writerow([s["qa_id"], s.get("native_id"), s["video_key"], s.get("dataset"), s["question_type"],
                            s["automatic_status"], s.get("official_metric"), s.get("official_score"),
                            s.get("raw_answer"), s.get("parsed_answer"), s.get("reference_answer"),
                            s.get("candidate_reason")])


# ============================================================================================ candidate export
def case_dir(output_root, entry):
    return os.path.join(output_root, "candidate_cases", entry["safe_key"])


def export_candidate(entry, ctx):
    """Copy the playable video to Drive and write the exact input frames, contact sheets, frame manifest and
    decode check. ctx: output_root, venv_py, env, lib_dir, fetch_video, meta, qa_by_id, scores_by_id, config_hash.
    Returns a list of problems (empty when the export is complete)."""
    cdir = case_dir(ctx["output_root"], entry)
    os.makedirs(cdir, exist_ok=True)
    meta = ctx["meta"].get(entry["video_key"])
    if not meta:
        return ["no video_meta (frame provenance) recorded by the run for this video"]
    vfile = os.path.join(cdir, "video.mp4")
    if not os.path.exists(vfile):
        src = ctx["fetch_video"](entry["video_key"])
        shutil.copyfile(src, vfile + ".part")
        os.replace(vfile + ".part", vfile)
    prov = os.path.join(cdir, "provenance.json")
    L.write_json_atomic(prov, meta)
    problems = []
    man = load_frames_manifest(cdir)
    if not man or not man.get("input_sheets") or not os.path.exists(os.path.join(cdir, "video_check.json")):
        r = subprocess.run([ctx["venv_py"], os.path.join(ctx["lib_dir"], "vsibench_frames.py"), "--video", vfile,
                            "--case_dir", cdir, "--lib_dir", ctx["lib_dir"], "--mode", "input", "--provenance", prov],
                           env=ctx.get("env"), capture_output=True, text=True)
        if r.returncode:
            problems.append(f"frame export failed: {(r.stdout + r.stderr).strip()[-400:]}")
    qa_ids = [entry["primary_qa_id"]] + entry["supporting_qa_ids"]
    L.write_json_atomic(os.path.join(cdir, "case.json"), {
        "entry": entry, "config_hash": ctx["config_hash"], "exported_at": now(),
        "qa": [ctx["qa_by_id"][q] for q in qa_ids], "scores": [ctx["scores_by_id"].get(q) for q in qa_ids]})
    return problems


def load_frames_manifest(cdir):
    p = os.path.join(cdir, "frames_manifest.json")
    return json.load(open(p)) if os.path.exists(p) else None


def run_extra_frames(entry, ctx, fps=1.0, windows=()):
    cdir = case_dir(ctx["output_root"], entry)
    cmd = [ctx["venv_py"], os.path.join(ctx["lib_dir"], "vsibench_frames.py"), "--video", os.path.join(cdir, "video.mp4"),
           "--case_dir", cdir, "--lib_dir", ctx["lib_dir"], "--mode", "extra", "--fps", str(fps)]
    for w in windows:
        cmd += ["--window", w]
    r = subprocess.run(cmd, env=ctx.get("env"), capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).strip()


def parse_windows(text):
    """'12,20' or '12,20,5' entries separated by ';' -> ['12,20,5', ...] (default 5 fps), with errors."""
    out, errors = [], []
    for part in [p.strip() for p in (text or "").split(";") if p.strip()]:
        try:
            nums = [float(x) for x in part.split(",")]
            if len(nums) == 2:
                nums.append(5.0)
            if len(nums) != 3 or nums[0] < 0 or nums[1] <= nums[0] or nums[2] <= 0:
                raise ValueError
            out.append(",".join(f"{x:g}" for x in nums))
        except ValueError:
            errors.append(f"window '{part}': use start,end[,fps] in seconds with end > start")
    return out, errors


# ============================================================================================ reviews
def review_path(output_root, qa_id):
    return os.path.join(output_root, "reviews", re.sub(r"[^A-Za-z0-9._-]", "_", qa_id) + ".json")


def load_review(output_root, qa_id):
    p = review_path(output_root, qa_id)
    return json.load(open(p)) if os.path.exists(p) else None


def load_reviews(output_root):
    d = os.path.join(output_root, "reviews")
    out = {}
    for fn in sorted(os.listdir(d)) if os.path.isdir(d) else []:
        if fn.endswith(".json"):
            r = json.load(open(os.path.join(d, fn)))
            if isinstance(r, dict) and r.get("qa_id"):
                out[r["qa_id"]] = r
    return out


def validate_provisional(prov):
    """A short answer such as 'B' or '3' is fine; empty is not. 'cannot determine' is an explicit, valid judgment."""
    errors = []
    if not isinstance(prov, dict) or not isinstance(prov.get("answer"), str) or not prov["answer"].strip():
        errors.append("blind judgment: enter your own answer before the reveal (an explicit 'cannot determine' is valid)")
    ev = (prov or {}).get("evidence_status") or ""
    if ev and ev not in EVIDENCE_STATUSES:
        errors.append(f"blind judgment: unknown evidence status {ev!r}")
    return errors


def save_provisional(output_root, qa, config_hash, run_id, answer, evidence_status="", note=""):
    """Save the blind judgment (before any answer is shown). It is written once and never replaced."""
    r = load_review(output_root, qa["qa_id"])
    if r and r.get("config_hash") == config_hash and r.get("provisional"):
        return r, ["the blind judgment for this question is already saved and cannot be changed after the reveal"]
    prov = {"answer": (answer or "").strip(), "evidence_status": evidence_status or "", "note": (note or "").strip()}
    errors = validate_provisional(prov)
    if errors:
        return r, errors
    r = r if r and r.get("config_hash") == config_hash else {
        "qa_id": qa["qa_id"], "video_key": qa["video_key"], "config_hash": config_hash, "run_id": run_id,
        "review_version": REVIEW_VERSION, "history": (r or {}).get("history", [])}
    prov["saved_at"] = now()
    r.update(stage="provisional", provisional=prov)
    r["history"].append({"event": "provisional", "saved_at": prov["saved_at"], "config_hash": config_hash, "data": prov})
    L.write_json_atomic(review_path(output_root, qa["qa_id"]), r)
    return r, []


CIT_RE = re.compile(r"^(input|extra)\b(.*)$", re.I)


def parse_citations(text):
    """One citation per line: '<input|extra> [frame=<index>] [t=<seconds>] [note]'. Index and seconds are kept
    in separate fields; either or both may be given (both must then agree)."""
    out, errors = [], []
    for n, line in enumerate([x.strip() for x in (text or "").splitlines()], 1):
        if not line:
            continue
        m = CIT_RE.match(line)
        if not m:
            errors.append(f"citation line {n}: start with 'input' or 'extra'")
            continue
        c = {"frame_set": m.group(1).lower(), "frame_index": None, "timestamp_s": None}
        rest = []
        for tok in m.group(2).split():
            k, _, v = tok.partition("=")
            try:
                if k in ("frame", "f", "index") and v:
                    c["frame_index"] = int(v)
                elif k in ("t", "time", "s") and v:
                    c["timestamp_s"] = float(v.rstrip("s"))
                else:
                    rest.append(tok)
            except ValueError:
                errors.append(f"citation line {n}: bad number in {tok!r}")
        c["note"] = " ".join(rest)
        if c["frame_index"] is None and c["timestamp_s"] is None:
            errors.append(f"citation line {n}: give frame=<index> and/or t=<seconds>")
        out.append(c)
    return out, errors


def resolve_citation(c, frames_manifest, case_root=None):
    """The manifest entry a citation points to, or an error. Resolves only against this video's manifest."""
    if c.get("frame_set") not in ("input", "extra"):
        return None, "frame_set must be 'input' or 'extra'"
    fi, ts = c.get("frame_index"), c.get("timestamp_s")
    if fi is None and ts is None:
        return None, "citation has neither frame index nor timestamp"
    if fi is not None and (isinstance(fi, bool) or not isinstance(fi, int)):
        return None, "frame_index must be an integer frame index (seconds go in timestamp_s)"
    if ts is not None and (isinstance(ts, bool) or not isinstance(ts, (int, float))):
        return None, "timestamp_s must be a number of seconds"
    entries = [e for e in (frames_manifest or {}).get("entries", []) if e["frame_set"] == c["frame_set"]]
    fps = (frames_manifest or {}).get("fps") or 30.0
    tol = 0.5 / fps + 1e-3
    if fi is not None:
        e = next((e for e in entries if e["frame_index"] == fi), None)
        if e is None:
            return None, f"frame {fi} is not one of this video's exported {c['frame_set']} frames"
        if ts is not None and abs(e["timestamp_s"] - ts) > tol:
            return None, f"frame {fi} is at {e['timestamp_s']:.3f}s, not {ts}s (index and timestamp disagree)"
    else:
        e = min(entries, key=lambda e: abs(e["timestamp_s"] - ts), default=None)
        if e is None or abs(e["timestamp_s"] - ts) > tol:
            return None, f"no exported {c['frame_set']} frame at {ts}s"
    if case_root and not os.path.exists(os.path.join(case_root, e["file"])):
        return None, f"cited image {e['file']} is missing"
    return e, None


def validate_final(final, frames_manifest, case_root, automatic_status=None):
    """Errors in a final review (empty list = valid). Defaults and blank fields never pass."""
    errors = []
    f = final or {}
    if f.get("verdict") not in FINAL_VERDICTS:
        errors.append("final verdict: choose wrong, correct or uncertain")
    if f.get("evidence_status") not in EVIDENCE_STATUSES:
        errors.append("evidence status: choose one of the five statuses")
    if f.get("confidence") not in CONFIDENCE:
        errors.append("confidence: choose low, medium or high")
    if _blank(f.get("observations")):
        errors.append("observations: describe what you saw (with frame times)")
    if _blank(f.get("rationale")):
        errors.append("rationale: explain the verdict")
    cat = f.get("category") or ""
    if f.get("verdict") == "wrong":
        if cat not in FINAL_CATEGORIES:
            errors.append(f"category: required for a wrong verdict ({', '.join(FINAL_CATEGORIES)})")
        elif f.get("evidence_status") in EVIDENCE_STATUSES and f["evidence_status"] not in FINAL_CATEGORIES[cat]:
            errors.append(f"category {cat} does not fit evidence status {f['evidence_status']}")
        if automatic_status == "format_error" and cat != "format_failure":
            errors.append("this answer was a format error: a confirmed failure must use category format_failure")
        if cat == "format_failure" and automatic_status != "format_error":
            errors.append("format_failure applies only to answers the official parser could not read as an option/number")
    elif cat and cat not in FINAL_CATEGORIES:
        errors.append(f"unknown category {cat!r}")
    cites = f.get("citations") or []
    if isinstance(cites, str):
        cites, perr = parse_citations(cites)
        errors += perr
    resolved = []
    for c in cites:
        if not isinstance(c, dict):
            errors.append(f"citation {c!r} is not a structured citation")
            continue
        e, err = resolve_citation(c, frames_manifest, case_root)
        if err:
            errors.append(f"citation {c.get('frame_set')} frame={c.get('frame_index')} t={c.get('timestamp_s')}: {err}")
        else:
            resolved.append(e)
    scope = f.get("reviewed_scope") or {}
    scope_in, scope_x = (scope.get("input") or "").strip(), (scope.get("extra") or "").strip()
    ev = f.get("evidence_status")
    if ev in ("sufficient_in_input", "partial_in_input"):
        if not any(e["frame_set"] == "input" for e in resolved):
            errors.append(f"{ev}: cite at least one valid input frame")
        if ev == "sufficient_in_input" and not (frames_manifest or {}).get("inputs_verified"):
            errors.append("sufficient_in_input needs verified model inputs; these frames are not verified "
                          f"reproductions ({(frames_manifest or {}).get('verification', 'no frame manifest')}): "
                          "use partial_in_input or ambiguous")
    if ev == "found_only_in_extra_frames":
        if not any(e["frame_set"] == "extra" for e in resolved):
            errors.append("found_only_in_extra_frames: cite at least one valid extra frame")
        if not scope_in:
            errors.append("found_only_in_extra_frames: describe how the input frames were reviewed (input scope)")
    if ev in ("not_found_in_reviewed_frames", "ambiguous") and not (scope_in or scope_x):
        errors.append(f"{ev}: save the reviewed scope (which frames/intervals were searched)")
    return errors


# ----------------------------------------------------------------------------------------------- short review form
# The 8c form asks plain questions; quick_final turns the answers into the same final review record the validator
# checks (nothing about what counts as a verified failure changes).
VERDICT_CHOICES = (("Wrong: the model's answer is wrong", "wrong"),
                   ("Actually correct (the benchmark answer looks wrong)", "correct"),
                   ("Not sure", "uncertain"))
EVIDENCE_CHOICES = (("Yes, clearly, in the model's frames", "sufficient_in_input"),
                    ("Partly, in the model's frames", "partial_in_input"),
                    ("Not in the model's frames, but in the extra frames I requested", "found_only_in_extra_frames"),
                    ("No, I couldn't find it in the frames I looked at", "not_found_in_reviewed_frames"),
                    ("Can't tell: the images are too unclear", "ambiguous"))
CATEGORY_FOR_EVIDENCE = {"sufficient_in_input": "visual_reasoning_error", "partial_in_input": "visual_reasoning_error",
                         "found_only_in_extra_frames": "evidence_not_in_input",
                         "not_found_in_reviewed_frames": "answer_failure_unresolved_evidence",
                         "ambiguous": "answer_failure_unresolved_evidence"}
FRAME_REF_RE = re.compile(r"^(#|f)?(\d+)$", re.I)


def parse_frame_refs(text, frames_manifest):
    """'#12, #40, f1234' -> structured citations. '#N' (or a bare N) is input frame N as numbered on the input contact
    sheets; 'fN' is video frame N on the extra-frame sheets (or an input frame with that video frame number)."""
    entries = (frames_manifest or {}).get("entries", [])
    by_pos = {e["position"]: e for e in entries if e["frame_set"] == "input" and "position" in e}
    out, errors = [], []
    for tok in [t for t in re.split(r"[\s,;]+", text or "") if t]:
        m = FRAME_REF_RE.match(tok)
        if not m:
            errors.append(f"'{tok}': write frames as #12 (input sheet number) or f1234 (extra sheet number)")
            continue
        n = int(m.group(2))
        if (m.group(1) or "#") == "#":
            e = by_pos.get(n)
            if e is None:
                errors.append(f"#{n}: the input sheets are numbered #0-#{max(by_pos) if by_pos else 0}")
                continue
            out.append({"frame_set": "input", "frame_index": e["frame_index"], "timestamp_s": None, "note": f"#{n}"})
        else:
            sets = [x["frame_set"] for x in entries if x["frame_index"] == n]
            if not sets:
                errors.append(f"f{n}: no exported frame has video frame number {n}")
                continue
            fs = "extra" if "extra" in sets else "input"
            out.append({"frame_set": fs, "frame_index": n, "timestamp_s": None, "note": f"f{n}"})
    return out, errors


def frames_text(citations, frames_manifest):
    """Saved citations back to the short '#12, f1234' form (for reopening a review)."""
    entries = (frames_manifest or {}).get("entries", [])
    out = []
    for c in citations or []:
        if c.get("frame_set") == "input":
            e = next((x for x in entries if x["frame_set"] == "input" and x["frame_index"] == c.get("frame_index")), None)
            out.append(f"#{e['position']}" if e and "position" in e else f"f{c.get('frame_index')}")
        elif c.get("frame_index") is not None:
            out.append(f"f{c['frame_index']}")
    return ", ".join(out)


def quick_final(verdict, evidence, confidence, frames, notes, frames_manifest, automatic_status=None):
    """The short form's answers as a final review record, plus friendly errors. The category follows from the
    verdict and the evidence answer; the reviewed scope records what the review screen showed."""
    errors = []
    cites, perr = parse_frame_refs(frames, frames_manifest)
    errors += perr
    if evidence in ("sufficient_in_input", "partial_in_input", "found_only_in_extra_frames") and not cites and not perr:
        errors.append("frames: type the frame number(s) that show the evidence, e.g. #12, #40"
                      + (" or f1234 for an extra frame" if evidence == "found_only_in_extra_frames" else ""))
    man = frames_manifest or {}
    n_in = sum(1 for e in man.get("entries", []) if e["frame_set"] == "input")
    extra_req = man.get("extra_requests") or []
    scope = {"input": f"all {n_in} model input frames on the {len(man.get('input_sheets') or [])} input contact sheet(s) "
                      "shown in the review",
             "extra": ("extra frames requested in the review: " + "; ".join(
                 f"{r.get('fps')} fps" + (f", windows {', '.join(r.get('windows') or [])}" if r.get("windows") else "")
                 for r in extra_req)) if extra_req else ""}
    category = ""
    if verdict == "wrong":
        category = "format_failure" if automatic_status == "format_error" else CATEGORY_FOR_EVIDENCE.get(evidence, "")
    final = {"verdict": verdict, "category": category, "evidence_status": evidence, "confidence": confidence,
             "citations": cites, "reviewed_scope": scope, "observations": notes, "rationale": notes, "form": "short-v1"}
    return final, errors


def save_final(output_root, qa, config_hash, final, frames_manifest, case_root, automatic_status=None):
    """Validate and save a final review. Invalid input saves nothing and returns the errors."""
    r = load_review(output_root, qa["qa_id"])
    if not r or r.get("config_hash") != config_hash or not r.get("provisional"):
        return r, ["save the blind judgment first (answers stay hidden until then)"]
    final = dict(final)
    final["citations"], cit_err = (final.get("citations"), []) if isinstance(final.get("citations"), list) \
        else parse_citations(final.get("citations") or "")
    errors = cit_err + validate_final(final, frames_manifest, case_root, automatic_status)
    if errors:
        return r, errors
    final["saved_at"] = now()
    final["frames_manifest_sha1"] = L.stable_hash(frames_manifest or {}, 16)
    r.update(stage="final", final=final)
    r["history"].append({"event": "final", "saved_at": final["saved_at"], "config_hash": config_hash, "data": final})
    L.write_json_atomic(review_path(output_root, qa["qa_id"]), r)
    return r, []


def current_final(review):
    """The most recent final decision in the review's history (later corrections supersede earlier ones)."""
    finals = [h for h in (review or {}).get("history", []) if h.get("event") == "final"]
    return finals[-1]["data"] if finals else None


# ============================================================================================ the validator
_SHA_CACHE = {}


def _sha256_cached(path):
    st = os.stat(path)
    key = (path, st.st_size, st.st_mtime)
    if key not in _SHA_CACHE:
        _SHA_CACHE[key] = L.sha256_file(path)
    return _SHA_CACHE[key]


def validate_verified_failure(case, qa, prediction, review, media_row, *, score, output_root, config_hash,
                              selected_video_keys):
    """Is this (case, question) a verified failure? Returns {"eligible", "reasons", "verdict"}.

    All six conditions of the brief: frozen selection + genuine prediction + valid reference; a saved final review
    for this exact qa_id and config hash; final verdict "wrong" (no defaults); valid blind judgment and final
    review; playable decoded video, required input assets and valid citations/scope; not superseded."""
    reasons = []
    vk = case.get("video_key")
    qid = (qa or {}).get("qa_id")
    # 1. frozen sample, genuine prediction, valid reference
    if vk not in set(selected_video_keys):
        reasons.append("video is not in the frozen selected sample")
    if not qa or qa.get("video_key") != vk:
        reasons.append("question does not belong to this video")
    if qid not in [case.get("primary_qa_id")] + list(case.get("supporting_qa_ids") or []):
        reasons.append("question is not one of this case's candidate questions")
    if not media_row or media_row.get("video_key") != vk:
        reasons.append("no media-manifest entry for this video")
    if not prediction or prediction.get("status") != "ok" or prediction.get("config_hash") != config_hash \
            or prediction.get("qa_id") != qid or not isinstance(prediction.get("raw_answer"), str):
        reasons.append("no successful Cambrian-P prediction for this question in this run")
    if not score or score.get("qa_id") != qid:
        reasons.append("no official score")
    elif score.get("automatic_status") in ("execution_error", "unscorable"):
        reasons.append(f"automatic status {score['automatic_status']}: never a verified failure")
    elif score.get("automatic_status") not in CANDIDATE_STATUSES:
        reasons.append(f"automatic status {score.get('automatic_status')} is not a failure candidate")
    if qa and (qa.get("reference_answer") in (None, "")):
        reasons.append("reference answer missing")
    # 2. review for this qa_id and run
    verdict = None
    if not review:
        reasons.append("not reviewed")
    else:
        if review.get("qa_id") != qid:
            reasons.append("the saved review is for a different question")
        if review.get("config_hash") != config_hash:
            reasons.append("the saved review belongs to a different run/configuration")
        fin = current_final(review)
        # 3. final stage, verdict wrong, nothing defaulted
        if review.get("stage") != "final" or not fin:
            reasons.append("no final review (blind judgment only)" if review.get("provisional") else "no final review")
        else:
            verdict = fin.get("verdict")
            if verdict != "wrong":
                reasons.append(f"final verdict is {verdict!r}, not 'wrong'")
            # 6. not superseded
            if (review.get("final") or {}).get("saved_at") != fin.get("saved_at") or \
                    (review.get("final") or {}).get("verdict") != verdict:
                reasons.append("the final verdict was superseded by a newer correction")
        # 4. blind judgment and final review valid
        prov_events = [h for h in review.get("history", []) if h.get("event") == "provisional"]
        reasons += validate_provisional(review.get("provisional"))
        if fin and (not prov_events or prov_events[0]["saved_at"] > fin.get("saved_at", "")):
            reasons.append("the blind judgment was not saved before the final review")
    # 5. assets
    root = case_dir(output_root, case)
    vfile = os.path.join(root, "video.mp4")
    vc_path = os.path.join(root, "video_check.json")
    if not os.path.exists(vfile):
        reasons.append("playable video missing")
    elif not os.path.exists(vc_path):
        reasons.append("video was never decode-checked")
    else:
        vc = json.load(open(vc_path))
        if not vc.get("ok"):
            reasons.append(f"video failed its decode check: {vc.get('error', 'no frames')}")
        elif vc.get("bytes") != os.path.getsize(vfile) or vc.get("sha256") != _sha256_cached(vfile):
            reasons.append("video file changed since its decode check")
    man = load_frames_manifest(root)
    if not man:
        reasons.append("input frames missing (no frame manifest)")
    else:
        inputs = [e for e in man["entries"] if e["frame_set"] == "input"]
        if len(inputs) != man.get("num_frames_expected") or not inputs:
            reasons.append(f"input frames incomplete ({len(inputs)} of {man.get('num_frames_expected')})")
        if any(not os.path.exists(os.path.join(root, e["file"])) for e in inputs):
            reasons.append("some input frame images are missing")
        if not man.get("input_sheets") or any(not os.path.exists(os.path.join(root, s)) for s in man["input_sheets"]):
            reasons.append("input contact sheets missing")
    if review and current_final(review) and verdict is not None:
        reasons += validate_final(current_final(review), man, root, (score or {}).get("automatic_status"))
    return {"eligible": not reasons, "reasons": reasons, "verdict": verdict, "qa_id": qid, "video_key": vk}


# ============================================================================================ counts
def evaluate(entries, ctx):
    """Validator results for every candidate QA of every given entry, and one state per video.
    ctx: output_root, config_hash, selected_video_keys, qa_by_id, ok (predictions), scores_by_id, media (by video),
    reviews (by qa_id)."""
    per_video = {}
    for e in entries:
        results = []
        for qid in [e["primary_qa_id"]] + e["supporting_qa_ids"]:
            res = validate_verified_failure(
                e, ctx["qa_by_id"].get(qid), ctx["ok"].get(qid), ctx["reviews"].get(qid), ctx["media"].get(e["video_key"]),
                score=ctx["scores_by_id"].get(qid), output_root=ctx["output_root"], config_hash=ctx["config_hash"],
                selected_video_keys=ctx["selected_video_keys"])
            results.append(res)
        prim_review = ctx["reviews"].get(e["primary_qa_id"])
        prim_ok_run = prim_review and prim_review.get("config_hash") == ctx["config_hash"] \
            and prim_review.get("qa_id") == e["primary_qa_id"]
        prim_final = current_final(prim_review) if prim_ok_run else None
        if any(r["eligible"] for r in results):
            state = "verified_wrong"
        elif any(r["verdict"] == "wrong" for r in results):
            state = "invalid_or_missing_evidence"
        elif prim_final and prim_final.get("verdict") == "correct":
            state = "overridden_correct"
        elif prim_final and prim_final.get("verdict") == "uncertain":
            state = "uncertain"
        elif prim_ok_run and prim_review.get("provisional"):
            state = "provisional_only"
        else:
            state = "unreviewed"
        per_video[e["video_key"]] = {"entry": e, "state": state, "results": results,
                                     "verified_qa_ids": [r["qa_id"] for r in results if r["eligible"]]}
    return per_video


def counts(all_entries, active_entries, per_video, output_root, target, minimum):
    exported = [e for e in active_entries
                if os.path.exists(os.path.join(case_dir(output_root, e), "frames_manifest.json"))]
    states = {k: v["state"] for k, v in per_video.items()}
    final_reviewed = {k for k, v in per_video.items() if any(r["verdict"] for r in v["results"])}
    c = {"candidate_video_count": len({e["video_key"] for e in all_entries}),
         "exported_video_count": len({e["video_key"] for e in exported}),
         "final_review_video_count": len(final_reviewed),
         "verified_wrong_video_count": len({k for k, s in states.items() if s == "verified_wrong"}),
         "overridden_correct_video_count": len({k for k, s in states.items() if s == "overridden_correct"}),
         "uncertain_video_count": len({k for k, s in states.items() if s == "uncertain"}),
         "invalid_or_missing_evidence_video_count": len({k for k, s in states.items()
                                                         if s == "invalid_or_missing_evidence"}),
         "target": target, "minimum": minimum}
    c["target_met"] = c["verified_wrong_video_count"] >= target
    c["minimum_met"] = c["verified_wrong_video_count"] >= minimum
    return c


def shortfall_message(c, inference_complete):
    n, t, m = c["verified_wrong_video_count"], c["target"], c["minimum"]
    scope = "" if inference_complete else " Inference on the selected videos is not complete yet."
    if n >= t:
        return f"Target met: {n} distinct videos with verified failures.{scope}"
    if n >= m:
        return (f"{n} verified failure videos: minimum ({m}) met, target ({t}) shortfall — "
                f"{c['candidate_video_count'] - n} other candidate videos were overridden, uncertain, invalid or not "
                f"reviewed yet.{scope}")
    return (f"Minimum NOT met: {n} distinct videos with verified failures (minimum {m}, target {t}). The requested "
            f"output is incomplete. The sample was not expanded.{scope}")


def completion_gate(c, coverage):
    inference_complete = bool((coverage or {}).get("inference_complete"))
    return {"inference_complete": inference_complete, "case_export_complete": c["minimum_met"],
            "target_met": c["target_met"], "message": shortfall_message(c, inference_complete)}


# ============================================================================================ final deliverable
def build_final(output_root, per_video, active_entries, ctx, c, gate, target):
    """failure_videos/ and cases/ hold exactly the qualifying cases (up to `target`, in queue order);
    verified_failure_manifest.json lists them. Non-qualifying files are removed from those two folders only
    (their history stays in candidate_cases/ and reviews/)."""
    order = [e["video_key"] for e in active_entries]
    order += [k for k in per_video if k not in order]
    verified = [per_video[k] for k in order if k in per_video and per_video[k]["state"] == "verified_wrong"]
    final = verified[:target]
    fv_dir, cs_dir = os.path.join(output_root, "failure_videos"), os.path.join(output_root, "cases")
    os.makedirs(fv_dir, exist_ok=True)
    os.makedirs(cs_dir, exist_ok=True)
    keep = {v["entry"]["safe_key"] for v in final}
    for f in os.listdir(fv_dir):
        if f.endswith(".mp4") and f[:-4] not in keep:
            os.remove(os.path.join(fv_dir, f))
    for d in os.listdir(cs_dir):
        if d not in keep and os.path.isdir(os.path.join(cs_dir, d)):
            shutil.rmtree(os.path.join(cs_dir, d))
    cases_out = []
    for v in final:
        e = v["entry"]
        src = case_dir(output_root, e)
        dst = os.path.join(cs_dir, e["safe_key"])
        vdst = os.path.join(fv_dir, e["safe_key"] + ".mp4")
        if not os.path.exists(vdst) or os.path.getsize(vdst) != os.path.getsize(os.path.join(src, "video.mp4")):
            shutil.copyfile(os.path.join(src, "video.mp4"), vdst + ".part")
            os.replace(vdst + ".part", vdst)
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns("video.mp4", "*.part"))
        qas = [ctx["qa_by_id"][q] for q in v["verified_qa_ids"]]
        reviews = {q: ctx["reviews"][q] for q in v["verified_qa_ids"]}
        L.write_json_atomic(os.path.join(dst, "reviews.json"), reviews)
        case = {"video_key": e["video_key"], "safe_key": e["safe_key"], "dataset": e["dataset"],
                "verified_qa_ids": v["verified_qa_ids"], "primary_qa_id": e["primary_qa_id"],
                "video_file": f"failure_videos/{e['safe_key']}.mp4", "video_sha256": _sha256_cached(vdst),
                "case_dir": f"cases/{e['safe_key']}", "questions": []}
        for q in qas:
            s, fin = ctx["scores_by_id"][q["qa_id"]], current_final(reviews[q["qa_id"]])
            case["questions"].append({
                "qa_id": q["qa_id"], "native_id": q.get("native_id"), "question_type": q["question_type"],
                "question": q["question"], "options": q.get("options"), "raw_answer": s.get("raw_answer"),
                "parsed_answer": s.get("parsed_answer"), "reference_answer": q["reference_answer"],
                "official_metric": s.get("official_metric"), "official_score": s.get("official_score"),
                "automatic_status": s.get("automatic_status"), "final_verdict": fin["verdict"],
                "category": fin.get("category"), "evidence_status": fin["evidence_status"],
                "confidence": fin["confidence"], "observations": fin["observations"], "rationale": fin["rationale"],
                "reviewed_scope": fin.get("reviewed_scope"), "blind_judgment": reviews[q["qa_id"]]["provisional"],
                "citations": [dict(c, file=f"cases/{e['safe_key']}/" +
                                   resolve_citation(c, load_frames_manifest(dst), dst)[0]["file"])
                              for c in fin.get("citations") or []]})
        L.write_json_atomic(os.path.join(dst, "case.json"), case)
        with open(os.path.join(dst, "case.md"), "w") as f:
            f.write(case_markdown(case))
        cases_out.append(case)
    manifest = {"run_id": ctx.get("run_id"), "config_hash": ctx["config_hash"], "created": now(),
                "review_version": REVIEW_VERSION, "counts": c, "gate": gate,
                "verified_video_count_in_manifest": len({x["video_key"] for x in cases_out}),
                "verified_beyond_target": len(verified) - len(final), "cases": cases_out}
    L.write_json_atomic(os.path.join(output_root, "verified_failure_manifest.json"), manifest)
    return manifest


def case_markdown(case):
    lines = [f"# {case['video_key']}", "", f"Playable video: `{case['video_file']}`", ""]
    for q in case["questions"]:
        opts = "\n".join(f"- {o}" for o in q.get("options") or [])
        lines += [f"## `{q['qa_id']}` (VSI-Bench id {q['native_id']}, {q['question_type']})", "", q["question"], "",
                  opts or "(numeric answer)", "",
                  f"- Model raw answer: `{q['raw_answer']}`; officially parsed: `{q['parsed_answer']}`",
                  f"- Benchmark answer: `{q['reference_answer']}`",
                  f"- Official {q['official_metric']}: {q['official_score']} (automatic status {q['automatic_status']})",
                  f"- Final review verdict: **{q['final_verdict']}** ({q['category']}); evidence: {q['evidence_status']};"
                  f" confidence {q['confidence']}",
                  f"- Blind judgment before the reveal: {q['blind_judgment'].get('answer')}",
                  f"- Observations: {q['observations']}", f"- Rationale: {q['rationale']}",
                  f"- Reviewed scope: {q.get('reviewed_scope')}", ""]
        lines += [f"  - cited {c['frame_set']} frame index {c.get('frame_index')} / {c.get('timestamp_s')} s: "
                  f"`{c['file']}` {c.get('note', '')}" for c in q["citations"]]
    lines += ["", "_Selected because the answer was wrong; not an estimate of the error rate._"]
    return "\n".join(lines)


def write_summary(path, per_video, active_entries, ctx):
    """One row per candidate video, with its state from the shared validator and the reasons."""
    order = [e["video_key"] for e in active_entries] + [k for k in per_video if k not in {e["video_key"] for e in active_entries}]
    active = {e["video_key"] for e in active_entries}
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["video_key", "safe_key", "dataset", "queue", "state", "verified", "primary_qa_id", "question_type",
                    "raw_answer", "parsed_answer", "reference_answer", "official_metric", "official_score",
                    "automatic_status", "final_verdict", "evidence_status", "confidence", "reasons", "video_file"])
        for k in order:
            v = per_video[k]
            e = v["entry"]
            qid = v["verified_qa_ids"][0] if v["verified_qa_ids"] else e["primary_qa_id"]
            s = ctx["scores_by_id"].get(qid) or {}
            fin = current_final(ctx["reviews"].get(qid)) or {}
            res = next(r for r in v["results"] if r["qa_id"] == qid)
            w.writerow([k, e["safe_key"], e["dataset"], "active" if k in active else "reserve", v["state"],
                        v["state"] == "verified_wrong", qid, s.get("question_type"), s.get("raw_answer"),
                        s.get("parsed_answer"), s.get("reference_answer"), s.get("official_metric"),
                        s.get("official_score"), s.get("automatic_status"), fin.get("verdict", ""),
                        fin.get("evidence_status", ""), fin.get("confidence", ""), "; ".join(res["reasons"]),
                        f"failure_videos/{e['safe_key']}.mp4" if v["state"] == "verified_wrong" else ""])


def esc(s):
    return html.escape("" if s is None else str(s), quote=True)


def report_html(output_root, manifest, per_video, info):
    """Self-contained report; links are relative to output_root (copy the folder as a whole)."""
    c, gate = manifest["counts"], manifest["gate"]
    p = ["<!DOCTYPE html><html><head><meta charset='utf-8'><title>Cambrian-P VSI-Bench failure audit</title>",
         "<style>body{font-family:system-ui,sans-serif;max-width:1100px;margin:24px auto;padding:0 16px;color:#111}"
         "section{border-top:1px solid #ddd;padding:16px 0}video{max-width:100%;max-height:360px}img{max-width:100%}"
         ".k{color:#555}code{background:#f3f3f3;padding:1px 4px}table{border-collapse:collapse}"
         "td,th{border:1px solid #ddd;padding:3px 6px;font-size:13px;text-align:left}</style></head><body>",
         "<h1>Cambrian-P-7B on VSI-Bench: audit of a 10% video sample</h1>",
         f"<p><b>{esc(gate['message'])}</b></p>",
         "<p>Cases are selected because the answer was wrong after manual review; they are not an estimate of the "
         "failure rate. Frame-evidence review is manual; exported frames alone do not establish answerability.</p>",
         "<h2>Scope</h2><ul>"]
    p += [f"<li>{esc(k)}: {esc(v)}</li>" for k, v in info.items()]
    p.append(f"<li>Inference complete for the selected videos: {esc(gate['inference_complete'])}; case export "
             f"complete (minimum met): {esc(gate['case_export_complete'])}</li></ul><h2>Verification counts</h2><ul>")
    p += [f"<li>{esc(k)}: {esc(v)}</li>" for k, v in c.items()]
    p.append("</ul>")
    for case in manifest["cases"]:
        p.append(f"<section id='{esc(case['safe_key'])}'><h2>{esc(case['video_key'])}</h2>"
                 f"<video controls preload='metadata' src='{esc(case['video_file'])}'></video>"
                 f"<p class='k'>{esc(case['dataset'])} · <a href='{esc(case['case_dir'])}/case.md'>case.md</a></p>")
        for q in case["questions"]:
            opts = "".join(f"<li>{esc(o)}</li>" for o in q.get("options") or [])
            cites = "".join(f"<figure><img src='{esc(ct['file'])}' alt='cited frame'><figcaption>{esc(ct['frame_set'])} "
                            f"frame {esc(ct.get('frame_index'))} / {esc(ct.get('timestamp_s'))} s {esc(ct.get('note'))}"
                            "</figcaption></figure>" for ct in q["citations"])
            p.append(f"<h3><code>{esc(q['qa_id'])}</code> · {esc(q['question_type'])}</h3><p>{esc(q['question'])}</p>"
                     f"<ul>{opts}</ul><p><b>Model:</b> {esc(q['raw_answer'])} → parsed {esc(q['parsed_answer'])} · "
                     f"<b>Benchmark answer:</b> {esc(q['reference_answer'])} · <b>{esc(q['official_metric'])}:</b> "
                     f"{esc(q['official_score'])}</p><p><b>Final verdict:</b> {esc(q['final_verdict'])} "
                     f"({esc(q['category'])}) · <b>Evidence:</b> {esc(q['evidence_status'])} · <b>Confidence:</b> "
                     f"{esc(q['confidence'])}</p><p>{esc(q['observations'])}</p><p class='k'>{esc(q['rationale'])}</p>"
                     f"<p class='k'>Reviewed scope: {esc(q.get('reviewed_scope'))}</p>{cites}")
        man = load_frames_manifest(os.path.join(output_root, case["case_dir"])) or {}
        p += [f"<img src='{esc(case['case_dir'] + '/' + s)}' alt='input contact sheet'>" for s in man.get("input_sheets", [])]
        p.append("</section>")
    others = [v for v in per_video.values() if v["state"] != "verified_wrong"]
    if others:
        p.append("<h2>Other candidates (not verified failures)</h2><table><tr><th>video</th><th>state</th>"
                 "<th>primary question</th><th>reasons</th></tr>")
        p += [f"<tr><td>{esc(v['entry']['video_key'])}</td><td>{esc(v['state'])}</td><td>{esc(v['entry']['primary_qa_id'])}"
              f"</td><td>{esc('; '.join(v['results'][0]['reasons'][:3]))}</td></tr>" for v in others]
        p.append("</table>")
    p.append("</body></html>")
    with open(os.path.join(output_root, "report.html"), "w") as f:
        f.write("".join(p))


PACKAGE_FILES = ("report.html", "summary.csv", "verified_failure_manifest.json", "coverage.json", "selection.json",
                 "run_manifest.json", "failure_candidates.csv")


def package(output_root, zip_path):
    """Zip only the final deliverable named by verified_failure_manifest.json (never candidate history)."""
    man = json.load(open(os.path.join(output_root, "verified_failure_manifest.json")))
    names = [f for f in PACKAGE_FILES if os.path.exists(os.path.join(output_root, f))]
    for case in man["cases"]:
        names.append(case["video_file"])
        root = os.path.join(output_root, case["case_dir"])
        for d, _, fs in os.walk(root):
            names += [os.path.relpath(os.path.join(d, f), output_root) for f in fs]
    tmp = zip_path + ".part"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:
        for n in names:
            z.write(os.path.join(output_root, n), n)
    os.replace(tmp, zip_path)
    return names

`vsibench_diagnostics.py`

In [ ]:
%%writefile /content/vsibench/vsibench_diagnostics.py
"""Read-only diagnostics for the VSI-Bench review queue: why are there (no) candidates, and what to run next.

Used by cells 8a and 8c. Nothing here starts inference, re-scores, edits the queue or deletes anything: it reads the
frozen selection, merges the persisted prediction shards (the same merge 7a uses), compares content fingerprints
recorded by 7a (scoring_state.json) and 8a (queue_state.json), and reports facts plus the next cells to run.
Missing state is reported as unknown / not initialized, never as zero.
"""
import hashlib
import json
import os
import time

import vsibench_lib as L
import vsibench_pipeline as P
import vsibench_review as R

SCORING_STATE = "scoring_state.json"
QUEUE_STATE = "queue_state.json"
UNKNOWN = "unknown"

# Globals each later cell needs, and the cell that defines them (for prerequisite messages).
DEFINED_IN = {
    "RUN_MODE": "1a", "OUTPUT_ROOT": "1a", "LOCAL_ROOT": "1a", "TARGET_FAILURE_VIDEOS": "1a", "MIN_FAILURE_VIDEOS": "1a",
    "RUN_ID": "1a", "L": "1c", "P": "1c", "VENV_PY": "2a", "WORKER_ENV": "2a", "CONFIG_HASH": "2b", "JOURNAL": "2b",
    "IDENTITY": "2b", "SEL": "4a", "QA": "4a", "QA_BY_ID": "4a", "MANIFEST": "5a", "MANIFEST_BY_VIDEO": "5a",
    "fetch_video": "6a", "OK": "7a", "META": "7a", "SCORES": "7a", "SCORES_BY_ID": "7a", "COVERAGE": "7a",
    "R": "8a", "QUEUE": "8a", "QUEUE_PATH": "8a", "ACTIVE": "8a", "RESERVE": "8a", "EXPORT_CTX": "8a", "review_ctx": "8a",
}
NEEDS_8A = ("L", "OUTPUT_ROOT", "CONFIG_HASH", "MANIFEST", "QA_BY_ID", "SCORES", "SCORES_BY_ID", "OK", "META",
            "VENV_PY", "WORKER_ENV", "fetch_video", "TARGET_FAILURE_VIDEOS", "RUN_ID")
NEEDS_8C = NEEDS_8A + ("R", "QUEUE", "QUEUE_PATH", "ACTIVE", "RESERVE", "EXPORT_CTX", "review_ctx", "MANIFEST_BY_VIDEO",
                       "MIN_FAILURE_VIDEOS")


# ----------------------------------------------------------------------------------------------- fingerprints
def _fp(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()


def prediction_fingerprint(selected_qa, ok, failed, missing, config_hash, scorer_sha256):
    """Content identity of the scoring inputs: per selected question its outcome and the answer text or error,
    plus the run configuration and scorer. Replacing one answer with another changes it even if counts do not."""
    rows = []
    for q in sorted(selected_qa, key=lambda q: q["qa_id"]):
        k = q["qa_id"]
        if k in ok:
            rows.append((k, "ok", ok[k].get("raw_answer")))
        elif q["video_key"] in missing:
            rows.append((k, "media_missing", missing[q["video_key"]]))
        elif k in failed:
            rows.append((k, "error", failed[k][-1].get("error")))
    return _fp({"config_hash": config_hash, "scorer": scorer_sha256, "outcomes": rows})


def scores_fingerprint(scores, selected_qa_ids):
    sel = set(selected_qa_ids)
    rows = sorted((s["qa_id"], s.get("automatic_status"), s.get("official_score"), s.get("raw_answer"))
                  for s in scores if s.get("qa_id") in sel)
    return _fp(rows)


def record_scoring(output_root, prediction_fp, scores_fp, n_scores):
    """Written by 7a after it writes scores.jsonl (a small sidecar; scores and predictions are not touched)."""
    L.write_json_atomic(os.path.join(output_root, SCORING_STATE),
                        {"prediction_fingerprint": prediction_fp, "scores_fingerprint": scores_fp,
                         "n_scores": n_scores, "scored_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})


def record_queue(output_root, scores_fp, n_entries):
    """Written by 8a after it updates review_queue.json."""
    L.write_json_atomic(os.path.join(output_root, QUEUE_STATE),
                        {"scores_fingerprint": scores_fp, "n_entries": n_entries,
                         "built_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})


# ----------------------------------------------------------------------------------------------- state collection
def _load(path):
    try:
        return json.load(open(path))
    except (OSError, ValueError):
        return None


def collect_runtime_state(g):
    """What this kernel has in memory (g = globals()). Absent names are listed, never defaulted."""
    present = {k: k in g for k in DEFINED_IN}
    rt = {"run_mode": g.get("RUN_MODE", UNKNOWN) if "RUN_MODE" in g else UNKNOWN,
          "output_root": g.get("OUTPUT_ROOT"), "config_hash": g.get("CONFIG_HASH"), "journal": g.get("JOURNAL"),
          "present": present, "scores_loaded": "SCORES" in g, "queue_loaded": "QUEUE" in g}
    if "SCORES" in g and "SEL" in g and "QA" in g:
        rt["scores_fingerprint"] = scores_fingerprint(g["SCORES"], [q["qa_id"] for q in g["QA"]])
    if "ACTIVE" in g and "RESERVE" in g:
        rt["active"], rt["reserve"] = len(g["ACTIVE"]), len(g["RESERVE"])
    return rt


def collect_persisted_state(output_root, config_hash=None, journal_path=None):
    """What is saved for this run on Drive (plus the local journal if given). Read-only."""
    st = {"output_root": output_root, "available": bool(output_root) and os.path.isdir(output_root or "")}
    if not st["available"]:
        return st
    j = lambda *p: os.path.join(output_root, *p)
    run_config = _load(j("run_config.json")) or {}
    st["config_hash"] = config_hash or run_config.get("config_hash")
    st["scorer_sha256"] = ((run_config.get("identity") or {}).get("SCORER") or {}).get("sha256")
    st["last_inference_mode"] = ((_load(j("run_manifest.json")) or {}).get("last_run") or {}).get("mode")
    sel = _load(j("selection.json"))
    qa, _ = L.read_jsonl(j("qa_manifest.jsonl")) if sel else ([], 0)
    st["selection"] = None if not sel else {"videos": sel.get("selected_videos"), "questions": sel.get("selected_questions"),
                                            "video_keys": sel.get("selected_video_keys")}
    st["selected_qa"] = qa
    if not (sel and st["config_hash"]):
        return st
    ok, failed, _, report = P.gather_results(j("predictions"), journal_path or "", st["config_hash"])
    missing = P.missing_media(j("logs", "media_missing.jsonl"))
    ids = {q["qa_id"] for q in qa}
    vids = {q["qa_id"]: q["video_key"] for q in qa}
    ok_sel = {k: v for k, v in ok.items() if k in ids}
    media_qa = {k for k in ids if k not in ok_sel and vids[k] in missing}
    err_qa = {k for k in ids if k not in ok_sel and k not in media_qa and k in failed}
    st["predictions"] = {"ok": len(ok_sel), "execution_errors": len(err_qa), "media_missing": len(media_qa),
                         "pending": len(ids) - len(ok_sel) - len(err_qa) - len(media_qa),
                         "rows_outside_selection": len(set(ok) - ids)}
    st["merge"] = {"complete_shards": report["complete"], "incomplete": len(report["incomplete"]),
                   "corrupted": len(report["corrupted"]), "other_config": len(report["other_config"]),
                   "journal_torn_lines": report.get("journal_truncated_lines", 0),
                   "journal_read": bool(journal_path and os.path.exists(journal_path))}
    cats = {}
    for k in err_qa:
        e = str(failed[k][-1].get("error") or "unknown error")
        cats[e.split(":")[0][:80]] = cats.get(e.split(":")[0][:80], 0) + 1
    for v, why in missing.items():
        k = f"video media unavailable ({str(why).split(':')[0][:60]})"
        cats[k] = cats.get(k, 0) + 1
    st["error_categories"] = dict(sorted(cats.items(), key=lambda kv: -kv[1])[:5])
    st["prediction_fingerprint"] = prediction_fingerprint(qa, ok_sel, failed, missing, st["config_hash"], st["scorer_sha256"])
    st["scoring_state"] = _load(j(SCORING_STATE))
    scores = L.read_jsonl(j("scores.jsonl"))[0] if os.path.exists(j("scores.jsonl")) else None
    st["scores_file"] = scores is not None
    if scores is not None:
        sc = [s for s in scores if s.get("qa_id") in ids]
        st["scores_fingerprint"] = scores_fingerprint(sc, ids)
        st["status_counts"] = {}
        for s in sc:
            st["status_counts"][s.get("automatic_status")] = st["status_counts"].get(s.get("automatic_status"), 0) + 1
        st["inspection_candidates"] = sum(s.get("automatic_status") in R.INSPECTION_STATUSES for s in sc)
        elig = [s for s in sc if s.get("automatic_status") in R.CANDIDATE_STATUSES]
        st["queue_eligible_qa"] = len(elig)
        st["queue_eligible_videos"] = len({s["video_key"] for s in elig})
    q = _load(j("review_queue.json"))
    st["queue_file"] = q is not None
    if q is not None:
        active, reserve = R.queue_split(q)
        st["queue"] = {"entries": len(q.get("entries", [])), "active": len(active), "reserve": len(reserve)}
        ready = 0
        for e in active:
            cdir = R.case_dir(output_root, e)
            man = R.load_frames_manifest(cdir) or {}
            if (os.path.exists(os.path.join(cdir, "video.mp4")) and man.get("input_sheets")
                    and os.path.exists(os.path.join(cdir, "video_check.json"))):
                ready += 1
        st["queue"]["exports_ready"] = ready
    st["queue_state"] = _load(j(QUEUE_STATE))
    return st


# ----------------------------------------------------------------------------------------------- diagnosis
def _freshness(recorded, current):
    if current is None:
        return "missing"
    if recorded is None:
        return UNKNOWN
    return "current" if recorded == current else "stale"


def diagnose_candidate_state(runtime, persisted, needs=NEEDS_8C):
    """Structured facts + findings (each with a code, an explanation and the next cells to run)."""
    d = {"run_mode": runtime.get("run_mode", UNKNOWN), "output_root": persisted.get("output_root"),
         "config_hash": persisted.get("config_hash") or runtime.get("config_hash"),
         "last_inference_mode": persisted.get("last_inference_mode"), "findings": []}
    add = lambda code, text, nxt="": d["findings"].append({"code": code, "text": text, "next": nxt})
    missing = [k for k in needs if not runtime.get("present", {}).get(k)]
    d["missing_prerequisites"] = {k: DEFINED_IN[k] for k in missing}
    if missing:
        cells = sorted(set(DEFINED_IN[k] for k in missing), key=lambda c: (c[0], c))
        add("prerequisites_missing", f"Not initialized in this runtime: {', '.join(missing)}.",
            f"Run cells {', '.join(cells)} (in order) in this runtime, then rerun this cell.")
    if not persisted.get("available"):
        add("no_output", "No run folder found" + (f" at {persisted.get('output_root')}" if persisted.get("output_root")
                                                  else " (OUTPUT_ROOT unknown: run 1a)") + "; nothing can be diagnosed.",
            "Run 1a (mounts Drive) and check RUN_ID / OUTPUT_ROOT.")
        return d
    sel = persisted.get("selection")
    d["selected"] = {k: sel[k] for k in ("videos", "questions")} if sel else None
    if not sel:
        add("no_selection", "The 10% video sample has not been frozen for this run (no selection.json).", "Run 1a-4a.")
        return d
    if "predictions" not in persisted:
        add("config_unknown", "The run configuration hash is unknown, so saved predictions cannot be merged.", "Run 2b.")
        return d
    pr, Q = persisted["predictions"], sel["questions"]
    d["predictions"], d["merge"] = pr, persisted["merge"]
    errors = pr["execution_errors"] + pr["media_missing"]
    mode = d["run_mode"]
    if pr["pending"] or errors:
        cov = f"{pr['ok']}/{Q} selected questions answered; {pr['pending']} pending; {errors} with errors"
        if mode == "pilot" and pr["pending"]:
            add("pilot_partial", f"Pilot mode is selected: 6b answers only the pilot questions. Coverage: {cov}.",
                'To process the rest: set RUN_MODE = "full" in 1a, run 1a, then 6b; then rerun 7a and 8a -> 8b -> 8c.')
        elif mode == "full" and pr["pending"]:
            add("full_incomplete", "Full mode is selected, but inference has not successfully answered the selected "
                f"subset. Coverage: {cov}.",
                "Rerun 6b to resume (it continues where it stopped) and check its output and logs/worker-*.log; "
                "then rerun 7a and 8a -> 8b -> 8c.")
        elif pr["pending"]:
            add("mode_unknown_partial", f"Run mode is {mode} in this runtime. Coverage: {cov}.",
                "Run 1a (full is the default), then 6b; then 7a and 8a -> 8b -> 8c.")
        if errors:
            cats = "; ".join(f"{k} x{v}" for k, v in persisted.get("error_categories", {}).items())
            parts = ([f"{pr['execution_errors']} question(s) ended in execution errors"] if pr["execution_errors"] else []) \
                + ([f"{pr['media_missing']} question(s) have unavailable media"] if pr["media_missing"] else [])
            add("errors_present", f"{' and '.join(parts)} ({cats}). These are not wrong answers and changing run "
                "mode will not fix them.",
                "See errors.jsonl, logs/media_missing.jsonl and logs/worker-*.log (all in the run folder). Rerunning 6b "
                "retries execution errors; a video logged as missing is retried only after its line is removed from "
                "logs/media_missing.jsonl.")
    m = d["merge"]
    if m["incomplete"] or m["corrupted"] or m["other_config"]:
        add("shards_ignored", f"Ignored prediction shards: {m['incomplete']} incomplete, {m['corrupted']} corrupted, "
            f"{m['other_config']} from another configuration (kept on disk, not used).")
    if pr["rows_outside_selection"]:
        add("rows_outside_selection", f"{pr['rows_outside_selection']} saved answers belong to questions outside the "
            "frozen selection; they are not counted.")
    # scoring freshness (persisted), and the in-memory copy
    ss = persisted.get("scoring_state") or {}
    scoring = "missing" if not persisted.get("scores_file") else \
        _freshness(ss.get("prediction_fingerprint"), persisted.get("prediction_fingerprint")) if ss else UNKNOWN
    if persisted.get("scores_file") and ss and ss.get("scores_fingerprint") != persisted.get("scores_fingerprint"):
        scoring = "stale"
    d["scoring"] = {"state": scoring}
    rt_fp = runtime.get("scores_fingerprint")
    d["scoring"]["in_memory"] = ("not loaded" if rt_fp is None else
                                 "current" if rt_fp == persisted.get("scores_fingerprint") and scoring == "current" else
                                 UNKNOWN if scoring == UNKNOWN else "stale")
    has_outcomes = pr["ok"] or errors
    if has_outcomes and scoring == "missing":
        add("scoring_missing", f"{pr['ok']} answers are saved but have not been scored (no scores.jsonl).",
            "Run 7a, then 8a -> 8b -> 8c. Running or changing 6b does not update an already computed score list.")
    elif has_outcomes and scoring == "stale":
        add("scoring_stale", "Scoring is stale: the saved predictions changed after the last scoring pass.",
            "Run 7a, then 8a -> 8b -> 8c." + (" Full mode is already selected." if mode == "full" else ""))
    elif has_outcomes and scoring == UNKNOWN:
        add("scoring_unknown", "Scoring freshness is unknown: scores.jsonl was written before freshness fingerprints "
            "existed. Nothing is discarded.", "Rerun 7a, then 8a -> 8b -> 8c.")
    elif d["scoring"]["in_memory"] == "stale":
        add("runtime_scores_stale", "The scores in this runtime's memory differ from the last saved scoring pass.",
            "Rerun 7a, then 8a -> 8b -> 8c.")
    counts = persisted.get("status_counts")
    d["status_counts"] = counts
    d["inspection_candidates"] = persisted.get("inspection_candidates")
    d["queue_eligible_qa"] = persisted.get("queue_eligible_qa")
    d["queue_eligible_videos"] = persisted.get("queue_eligible_videos")
    if counts is not None and scoring in ("current", UNKNOWN):
        n = sum(counts.values())
        scored = sum(v for k, v in counts.items() if k not in ("execution_error", "unscorable"))
        tail = (f" Not yet answered: {pr['pending']} pending, {errors} with errors." if (pr["pending"] or errors)
                else " Every selected question has an outcome.")
        if n == 0:
            add("no_scores", "No selected question has been scored yet." + tail, "Run 6b, then 7a.")
        elif not d["queue_eligible_qa"]:
            imperfect = counts.get("imperfect", 0)
            bad = counts.get("execution_error", 0) + counts.get("unscorable", 0)
            if scored and imperfect == 0 and counts.get("correct", 0) == scored:
                add("all_correct", f"No eligible failures: all {scored} successfully scored answers are correct "
                    f"(official score 1).{tail}")
            elif scored and imperfect:
                add("only_correct_imperfect", f"No automatic failure candidates: of {scored} scored answers, "
                    f"{counts.get('correct', 0)} are correct and {imperfect} received partial numeric credit "
                    "('imperfect'). Under the current policy partial credit is listed for inspection but not queued for "
                    f"review.{tail}", "See failure_candidates.csv for the partial-credit answers.")
            if bad:
                add("only_errors_unscorable" if not scored else "errors_unscorable_not_candidates",
                    f"{counts.get('execution_error', 0)} execution errors and {counts.get('unscorable', 0)} unscorable "
                    "questions are not wrong-answer candidates.",
                    "Execution errors: errors.jsonl and logs/. Unscorable: the scoring 'error' field in scores.jsonl.")
    # queue
    qs = persisted.get("queue_state") or {}
    queue = "missing" if not persisted.get("queue_file") else \
        (_freshness(qs.get("scores_fingerprint"), persisted.get("scores_fingerprint")) if qs else UNKNOWN)
    d["queue"] = dict(persisted.get("queue") or {}, state=queue)
    if runtime.get("active") is not None:
        d["queue"]["active_in_memory"] = runtime["active"]
    eligible = d["queue_eligible_videos"] or 0
    if eligible and scoring in ("current", UNKNOWN):
        if queue in ("missing", "stale", UNKNOWN):
            add(f"queue_{'unknown' if queue == UNKNOWN else queue}",
                f"{eligible} selected video(s) have automatic failure candidates, but the review queue is {queue}"
                + (" (built before freshness fingerprints existed)" if queue == UNKNOWN else "") + ".",
                "Run 8a -> 8b -> 8c.")
        elif d["queue"].get("entries") and not d["queue"].get("active"):
            add("queue_no_active", f"The queue has {d['queue']['entries']} entries ({d['queue'].get('reserve', 0)} in "
                "reserve) but none active: a queue-state inconsistency or review-state transition, not an absence of "
                "failures.", "Rerun 8a to rebuild the active list (reviews are kept), then 8b -> 8c.")
        elif d["queue"].get("active") and d["queue"].get("exports_ready", 0) < d["queue"]["active"]:
            add("exports_missing", f"{d['queue']['active']} active candidate(s), of which {d['queue']['exports_ready']} "
                "have a complete export (video, decode check, input frames). This is an export problem, not a lack of "
                "candidates.", "Run 8b, then 8c.")
        elif d["queue"].get("active"):
            add("ready", f"{d['queue']['active']} candidate video(s) are ready for review.")
    if runtime.get("active") == 0 and d["queue"].get("active"):
        add("runtime_queue_stale", "This runtime's ACTIVE list is empty but the saved queue has active entries.",
            "Rerun 8a, then 8c.")
    return d


def render_candidate_diagnostic(d, title="Review queue status"):
    sel = d.get("selected")
    pr = d.get("predictions")
    lines = [title + ":"]
    lines.append(f"  Current runtime mode: {d['run_mode']}"
                 + (" (full = every question of the selected ~10% of videos, not the whole benchmark)"
                    if d["run_mode"] == "full" else ""))
    lines.append(f"  Last recorded inference mode: {d.get('last_inference_mode') or 'unknown (none recorded)'}")
    lines.append(f"  Run folder: {d.get('output_root') or 'unknown'} · config hash: {d.get('config_hash') or 'unknown'}")
    lines.append(f"  Selected subset: {sel['videos']} videos / {sel['questions']} questions" if sel
                 else "  Selected subset: unknown (not frozen / not found)")
    if pr:
        lines.append(f"  Saved successful predictions: {pr['ok']}; pending: {pr['pending']}; execution errors: "
                     f"{pr['execution_errors']}; media unavailable: {pr['media_missing']}")
    else:
        lines.append("  Saved predictions: unknown (could not be merged)")
    if "scoring" in d:
        lines.append(f"  Scoring: {d['scoring']['state']} (in memory: {d['scoring']['in_memory']})")
    if d.get("status_counts") is not None:
        lines.append(f"  Automatic status: {d['status_counts']}")
        lines.append(f"  Inspection candidates (score below max): {d.get('inspection_candidates')}; queue-eligible "
                     f"questions: {d.get('queue_eligible_qa')} on {d.get('queue_eligible_videos')} video(s)")
    if "queue" in d:
        q = d["queue"]
        lines.append(f"  Review queue: {q.get('state')}; entries {q.get('entries', 'n/a')}, active {q.get('active', 'n/a')}, "
                     f"reserve {q.get('reserve', 'n/a')}, exports ready {q.get('exports_ready', 'n/a')}")
    for f in d["findings"]:
        lines.append(f"  - {f['text']}")
        if f["next"]:
            lines.append(f"    Next: {f['next']}")
    if not d["findings"]:
        lines.append("  - No problems found.")
    return "\n".join(lines)


def diagnose_from_globals(g, needs=NEEDS_8C):
    """Convenience for the notebook: runtime + persisted state from this kernel's globals."""
    rt = collect_runtime_state(g)
    ps = collect_persisted_state(g.get("OUTPUT_ROOT"), g.get("CONFIG_HASH"), g.get("JOURNAL"))
    return diagnose_candidate_state(rt, ps, needs)

`vsibench_selftest.py`

In [ ]:
%%writefile /content/vsibench/vsibench_selftest.py
"""Self-tests for the data / inference / scoring side of the VSI-Bench audit (no GPU, no model).

Covers the brief's checks 1-5 (selection, identity, real annotations, official-scorer parity, resume) plus media
access (zip range reads, CRC, safe paths). Exits nonzero if any check fails.

  python vsibench_selftest.py --repo <cambrian-p checkout> [--annotations test.jsonl] [--config_file x.parquet ...]
                              [--listings <dir of <archive>.json>]
"""
import argparse
import http.server
import json
import os
import random
import sys
import tempfile
import threading
import zipfile

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import vsibench_lib as L  # noqa: E402
import vsibench_pipeline as P  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--repo", required=True)
ap.add_argument("--annotations")
ap.add_argument("--config_file", action="append", default=[])
ap.add_argument("--listings")
args = ap.parse_args()

# SHA-256 of lmms-eval/lmms_eval/tasks/vsibench/utils.py at the commit this notebook was validated against.
PINNED_SCORER = {"b3c15527a9f9c4b70020b874eab66f67b2b03901":
                 "1cdfd6a839120c4e8d4de6640a3b1d9009cf93755a586e1348ddd4687693bc97"}
RESULTS = []


def check(name, cond, detail=""):
    RESULTS.append((name, bool(cond)))
    print(("PASS " if cond else "FAIL ") + name + (f"  [{detail}]" if detail and not cond else ""))


def raises(fn, exc=Exception):
    try:
        fn()
    except exc:
        return True
    return False


O = L.load_official(args.repo)


def synth_rows(n_videos=288, seed=0):
    """Synthetic VSI-Bench-shaped rows: n_videos (dataset, scene) pairs with 1-6 questions each."""
    rng = random.Random(seed)
    rows, nid = [], 0
    for v in range(n_videos):
        ds = L.VIDEO_DATASETS[v % 3]
        for _ in range(rng.randint(1, 6)):
            if rng.random() < 0.5:
                rows.append({"id": nid, "dataset": ds, "scene_name": f"s{v:04d}", "question_type": "object_counting",
                             "question": "How many chair(s) are in this room?", "ground_truth": str(rng.randint(1, 9)),
                             "options": None})
            else:
                rows.append({"id": nid, "dataset": ds, "scene_name": f"s{v:04d}", "question_type": "route_planning",
                             "question": "Which route?", "ground_truth": "B",
                             "options": ["A. left", "B. right", "C. back"]})
            nid += 1
    return rows


# ------------------------------------------------------------------------------------------- 1. selection
rows = synth_rows()
qa = [L.normalize_row(r, O) for r in rows]
keys = L.select_videos([q["video_key"] for q in qa], 0.10, 42)
check("1 288 unique videos give ceil(10%) = 29 selected videos", len(keys) == 29 and len(set(keys)) == 29)
check("1 repeated question rows never increase V", len({q["video_key"] for q in qa}) == 288 and len(qa) > 288)
chosen = [q for q in qa if q["video_key"] in set(keys)]
check("1 every question of each selected video is included",
      all(sum(1 for q in chosen if q["video_key"] == k) == sum(1 for q in qa if q["video_key"] == k) for k in keys))
check("1 selection only contains population videos", set(keys) <= {q["video_key"] for q in qa})

# ------------------------------------------------------------------------------------------- 2. determinism, identity
shuffled = list(qa)
random.Random(7).shuffle(shuffled)
check("2 same population and seed give the same selection whatever the row order",
      L.select_videos([q["video_key"] for q in shuffled], 0.10, 42) == keys)
check("2 a different seed gives a different selection", L.select_videos([q["video_key"] for q in qa], 0.10, 43) != keys)
base = {"DATASET_REPO": "nyu-visionx/VSI-Bench", "DATASET_CONFIG": "full", "DATASET_SPLIT": "test",
        "DATASET_REVISION": "rev1", "SAMPLING_UNIT": "video", "SUBSET_FRACTION": 0.1, "SEED": 42, "MODEL_REPO": "m",
        "MODEL_REVISION": "mr", "CODE_COMMIT": "c", "NUM_FRAMES": 128, "PROMPT_POLICY": L.PROMPT_POLICY,
        "GENERATION": {"max_new_tokens": 16}, "PREPROCESSING": "p", "SCORER": {"sha256": O["sha256"]}}
with tempfile.TemporaryDirectory() as d:
    rc = os.path.join(d, "run_config.json")
    L.check_run_config(rc, base)
    ok_same = not raises(lambda: L.check_run_config(rc, dict(base)))
    bad = [k for k in ("DATASET_CONFIG", "DATASET_REVISION", "SEED", "SCORER", "NUM_FRAMES", "GENERATION")
           if not raises(lambda: L.check_run_config(rc, dict(base, **{k: "changed"})), RuntimeError)]
    check("2 an unchanged identity is accepted under the same RUN_ID", ok_same)
    check("2 a changed dataset config/revision, seed, scorer, frames or generation is rejected", not bad, bad)
    L.bind_selection(rc, "h1")
    check("2 run_config is bound to one selection hash", raises(lambda: L.bind_selection(rc, "h2"), RuntimeError)
          and not raises(lambda: L.bind_selection(rc, "h1")))
    sel, q1, new = P.freeze_selection(d, qa, base, {"f": "sha"})
    sel2, q2, new2 = P.freeze_selection(d, shuffled, base, {"f": "sha"})
    check("2 a frozen selection is verified, not resampled, on a rerun", new and not new2 and q1 == q2)
    check("2 a frozen selection is refused for another dataset revision",
          raises(lambda: P.freeze_selection(d, qa, dict(base, DATASET_REVISION="rev2"), {"f": "sha"}), RuntimeError))
    check("2 a frozen selection is refused for changed annotations",
          raises(lambda: P.freeze_selection(d, qa[:-3], base, {"f": "sha"}), RuntimeError))
    check("2 selection.json records V, k, fraction, seed, keys and hash",
          sel["population_videos"] == 288 and sel["selected_videos"] == 29 and sel["sampling_unit"] == "video"
          and sel["selection_hash"] == L.selection_hash(keys) and sel["selected_video_keys"] == keys)

# ------------------------------------------------------------------------------------------- 3. real annotations
if args.annotations:
    raw = [json.loads(l) for l in open(args.annotations)]
    real = [L.normalize_row(r, O) for r in raw]
    probs = P.schema_checks(raw, real, O)
    check("3 real annotations pass the schema checks (ids, types, options, references, prompts)", not probs, probs)
    check("3 native ids are kept as qa ids", all(q["qa_id"] == f"vsibench-{q['native_id']}" for q in real))
    vids = sorted({q["video_key"] for q in real})
    check(f"3 V = {len(vids)} unique (dataset, scene_name) videos; sample = {L.math.ceil(0.1 * len(vids))}",
          len(L.select_videos(vids, 0.10, 42)) == L.math.ceil(0.1 * len(vids)))
    if args.config_file:
        cfg_rows = P.read_parquet_rows(args.config_file)
        check("3 the 'full' config's rows equal test.jsonl", not P.cross_check(cfg_rows, raw))
    if args.listings:
        members = {}
        for fn in os.listdir(args.listings):
            if fn.endswith(".zip.json"):
                members[fn[:-5]] = json.load(open(os.path.join(args.listings, fn)))
        mapped = [v for v in vids if L.archive_for(v)[0] in members]
        found = [v for v in mapped if L.archive_for(v)[1] in members[L.archive_for(v)[0]]]
        check(f"3 media paths: {len(found)}/{len(mapped)} videos of the listed archives are <dataset>/<scene>.mp4 members",
              mapped and len(found) == len(mapped))

# ------------------------------------------------------------------------------------------- 4. official scorer parity
commit = os.popen(f"git -C {args.repo} rev-parse HEAD 2>/dev/null").read().strip()
if commit in PINNED_SCORER:
    check("4 the official scorer file is byte-identical to the validated one", O["sha256"] == PINNED_SCORER[commit])
ns = O["ns"]
mc = {"qa_id": "vsibench-1", "native_id": 1, "video_key": "scannet/x", "dataset": "scannet", "question_type":
      "object_rel_direction_hard", "task_family": "multiple_choice", "question": "q?", "options":
      ["A. left", "B. right", "C. back", "D. front"], "reference_answer": "B"}
na = dict(mc, question_type="object_abs_distance", task_family="numeric", options=None, reference_answer="2.0")
cases = [  # (row, raw, expected official score, expected automatic status)
    (mc, "B", 1.0, "correct"), (mc, "b.", 1.0, "correct"), (mc, "B. right", 1.0, "correct"),
    (mc, "C", 0.0, "wrong_candidate"), (mc, "The answer is B", 0.0, "format_error"), (mc, "", 0.0, "format_error"),
    (mc, "E", 0.0, "format_error"),
    (na, "2.0", 1.0, "correct"), (na, "2.05 m", 1.0, "correct"), (na, "2.6", 0.5, "imperfect"),
    (na, "20", 0.0, "wrong_candidate"), (na, "two", 0.0, "format_error"), (na, "", 0.0, "format_error"),
    (na, "inf", 0.0, "format_error"),
]
bad = []
for row, raw, want_score, want_status in cases:
    s = L.score_answer(row, raw, O)
    direct = ns["vsibench_process_results"](
        {"question_type": row["question_type"], "ground_truth": row["reference_answer"]}, [raw])["vsibench_score"]
    metric = "accuracy" if row["task_family"] == "multiple_choice" else "MRA:.5:.95:.05"
    if not (abs(s["official_score"] - want_score) < 1e-9 and s["official_score"] == float(direct[metric])
            and s["automatic_status"] == want_status and s["official_metric"] == metric):
        bad.append((raw, s["official_score"], s["automatic_status"]))
check("4 official parity: MC correct/wrong/format, numeric perfect/partial/zero/format (score == official function)",
      not bad, bad)
check("4 MRA thresholds follow the official .5:.95:.05 grid",
      abs(ns["mean_relative_accuracy"](2.1, 2.0, .5, .95, .05) - 1.0) < 1e-9
      and abs(ns["mean_relative_accuracy"](3.0, 2.0, .5, .95, .05) - 0.1) < 1e-9)
s = L.score_answer(dict(mc, question_type="made_up_type"), "B", O)
check("4 an unsupported question type is unscorable", s["automatic_status"] == "unscorable")
s = L.score_answer(dict(na, reference_answer="n/a"), "2", O)
check("4 an invalid numeric reference is unscorable", s["automatic_status"] == "unscorable")
s = L.score_answer(dict(mc, reference_answer="Z"), "B", O)
check("4 a reference letter outside the options is unscorable", s["automatic_status"] == "unscorable")
s = L.score_answer(mc, None, O, status="error")
check("4 an execution error is never a wrong answer", s["automatic_status"] == "execution_error"
      and s["official_score"] is None)
check("4 the prompt is the official formatter's output and independent of the reference",
      L.prompt_excludes_reference(L.normalize_row({"id": 1, "dataset": "scannet", "scene_name": "x", "question_type":
                                                   "object_counting", "question": "How many 2?", "ground_truth": "2",
                                                   "options": None}, O), O))

# ------------------------------------------------------------------------------------------- 5. resume
with tempfile.TemporaryDirectory() as d:
    sub = [q for q in qa if q["video_key"] in set(keys[:3])]
    shard_dir, journal = os.path.join(d, "shards"), os.path.join(d, "j.jsonl")
    media = os.path.join(d, "media")
    state = {"stop": 2}

    def fake_worker(batch_path, extra):
        j = L.Journal(journal)
        n = 0
        for v in json.load(open(batch_path))["videos"]:
            done, _, _, _ = P.gather_results(shard_dir, journal, "H")
            rows_ = []
            for q in v["qa"]:
                if q["qa_id"] in done:
                    continue
                r = {"kind": "answer", "qa_id": q["qa_id"], "video_key": v["video_key"], "status": "ok",
                     "raw_answer": f"ans-{q['qa_id']}-{len(os.listdir(d))}", "config_hash": "H",
                     "answered_at": L.time.strftime("%Y-%m-%dT%H:%M:%S.") + str(L.time.time_ns())}
                j.append(r)
                rows_.append(r)
                n += 1
                if state["stop"] and n >= state["stop"]:
                    return 75
            L.write_shard(shard_dir, v["safe_key"] + f"-{L.time.time_ns()}", rows_, "H")
        return 0

    def fetch(v, dest):
        p = os.path.join(dest, v + ".mp4")
        os.makedirs(os.path.dirname(p), exist_ok=True)
        open(p, "wb").write(b"x")
        return p

    man = {k: {"safe_key": L.safe_key(k), "archive": "a.zip", "member": k + ".mp4", "member_missing": False}
           for k in keys[:3]}
    ctx = {"fetch_video": fetch, "run_worker": fake_worker, "release": lambda v, p: os.remove(p), "media_dir": media,
           "work_dir": d, "shard_dir": shard_dir, "journal": journal, "config_hash": "H", "batch_bytes": 10 ** 9,
           "log_missing": lambda *a: None, "skip_videos": set()}
    out = P.run_selection(sub, man, ctx, log=lambda *a: None)
    ok1, _, _, _ = P.gather_results(shard_dir, journal, "H")
    first = {k: v["raw_answer"] for k, v in ok1.items()}
    with open(journal, "a") as f:
        f.write('{"kind": "answer", "qa_id": "trunc')            # an interrupted append
    os.makedirs(shard_dir, exist_ok=True)
    open(os.path.join(shard_dir, "orphan.jsonl"), "w").write("{}\n")   # a shard without a completion marker
    state["stop"] = 0
    P.run_selection(sub, man, ctx, log=lambda *a: None)
    ok2, _, _, rep = P.gather_results(shard_dir, journal, "H")
    jrows, trunc = L.read_jsonl(journal)
    per_q = {}
    for r in jrows:
        if r.get("status") == "ok":
            per_q[r["qa_id"]] = per_q.get(r["qa_id"], 0) + 1
    check("5 the interruption stopped the run early", out.get("stopped") == 75 and 1 <= len(first) < len(sub))
    check("5 after restart every question is answered", all(q["qa_id"] in ok2 for q in sub))
    check("5 answers from before the interruption are kept unchanged", all(ok2[k]["raw_answer"] == v for k, v in first.items()))
    check("5 no question is answered twice", per_q and max(per_q.values()) == 1)
    check("5 a truncated journal line and an unmarked shard are ignored and reported",
          trunc == 1 and "orphan" in rep["incomplete"])
    ctx3 = dict(ctx, run_worker=lambda b, e: (_ for _ in ()).throw(AssertionError("worker called")))
    check("5 a finished set starts no worker", P.run_selection(sub, man, ctx3, log=lambda *a: None)["pending_videos"] == 0)

# ------------------------------------------------------------------------------------------- media access
class RangeHandler(http.server.SimpleHTTPRequestHandler):
    ignore_range = False

    def log_message(self, *a):
        pass

    def do_GET(self):
        path = self.translate_path(self.path)
        data = open(path, "rb").read()
        rng = self.headers.get("Range")
        if rng and not self.ignore_range:
            a, b = rng.split("=")[1].split("-")
            a, b = int(a), min(int(b), len(data) - 1)
            self.send_response(206)
            body = data[a:b + 1]
        else:
            self.send_response(200)
            body = data
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)


with tempfile.TemporaryDirectory() as d:
    payload = {"scannet/sceneA.mp4": os.urandom(300_000), "scannet/sceneB.mp4": os.urandom(5000)}
    zpath = os.path.join(d, "scannet.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for k, v in payload.items():
            z.writestr(k, v)
        z.writestr("../evil.mp4", b"x")
    srv = http.server.ThreadingHTTPServer(("127.0.0.1", 0), lambda *a, **k: RangeHandler(*a, directory=d, **k))
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    url = f"http://127.0.0.1:{srv.server_port}/scannet.zip"
    f = L.HttpRangeFile(url, os.path.getsize(zpath))
    zf = L.open_zip(f)
    listing = L.zip_listing(zf)
    out = L.extract_zip_member(zf, "scannet/sceneB.mp4", os.path.join(d, "out"), listing["scannet/sceneB.mp4"]["bytes"],
                               listing["scannet/sceneB.mp4"]["crc32"])
    check("media: a member is read by range requests without fetching the whole archive",
          open(out, "rb").read() == payload["scannet/sceneB.mp4"] and f.bytes_read < os.path.getsize(zpath) / 2,
          f.bytes_read)
    check("media: the CRC of the written file matches the archive", L.file_crc32(out) == listing["scannet/sceneB.mp4"]["crc32"])
    check("media: a manifest CRC mismatch is refused",
          raises(lambda: L.extract_zip_member(zf, "scannet/sceneA.mp4", os.path.join(d, "o2"), None, 123), IOError))
    check("media: a member absent from the archive raises KeyError",
          raises(lambda: L.extract_zip_member(zf, "scannet/nope.mp4", os.path.join(d, "o3")), KeyError))
    check("media: an unsafe member path is refused",
          raises(lambda: L.extract_zip_member(zf, "../evil.mp4", os.path.join(d, "o4")), ValueError))
    RangeHandler.ignore_range = True
    check("media: a server that ignores Range is detected (so the notebook can fall back to a full download)",
          raises(lambda: L.open_zip(L.HttpRangeFile(url, os.path.getsize(zpath))), L.RangeNotSupported))
    RangeHandler.ignore_range = False
    raw_bytes = bytearray(open(zpath, "rb").read())
    raw_bytes[100:120] = b"\x00" * 20                     # inside the first member's compressed data
    open(os.path.join(d, "bad.zip"), "wb").write(bytes(raw_bytes))
    zb = zipfile.ZipFile(os.path.join(d, "bad.zip"))
    first_member = zb.infolist()[0].filename
    check("media: corrupt member data is detected (CRC or zlib error)",
          raises(lambda: L.extract_zip_member(zb, first_member, os.path.join(d, "o5")), (zipfile.BadZipFile, Exception)))
    srv.shutdown()

man = P.media_manifest(["scannet/a", "scannet/b"], {"scannet.zip": {"scannet/a.mp4": {"bytes": 5, "crc32": 1}}}, {})
check("media: a selected video absent from its archive stays in the manifest, marked missing",
      [m["member_missing"] for m in man] == [False, True] and man[1]["member_bytes"] is None)
check("pilot: a missing video (unknown size) sorts last instead of failing",
      L.pilot_subset([dict(chosen[0], video_key="x/none"), chosen[1]], {"x/none": None, chosen[1]["video_key"]: 5}, 1, 10)
      [0]["video_key"] == chosen[1]["video_key"])
check("pilot: 2 smallest selected videos, at most 10 questions, deterministic",
      L.pilot_subset(chosen, {}, 2, 10) == L.pilot_subset(list(reversed(chosen)), {}, 2, 10)
      and len({q["video_key"] for q in L.pilot_subset(chosen, {}, 2, 10)}) <= 2
      and len(L.pilot_subset(chosen, {}, 2, 10)) <= 10)

failed = [n for n, ok in RESULTS if not ok]
print(f"\n{len(RESULTS) - len(failed)}/{len(RESULTS)} checks passed")
sys.exit(1 if failed else 0)

`vsibench_review_selftest.py`

In [ ]:
%%writefile /content/vsibench/vsibench_review_selftest.py
"""Self-tests for the review / verification / export side (brief tests 6-13 and the review interface).

Builds a synthetic run in a temporary folder with real MP4 files (tiny generated ones, plus a real VSI-Bench video
when one is given with --real_video), runs the real frame exporter (vsibench_frames.py, needs decord or PyAV),
the shared validator, the final export and the packaging, and drives the review cells' widgets. Exits 1 on any
failure. Usage: python vsibench_review_selftest.py [--real_video path.mp4] [--keep]
"""
import argparse
import hashlib
import json
import os
import shutil
import sys
import tempfile
import zipfile

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import numpy as np  # noqa: E402
import vsibench_lib as L  # noqa: E402
import vsibench_review as R  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--real_video")
ap.add_argument("--keep", action="store_true")
ap.add_argument("--frames_python", default=sys.executable,
                help="interpreter with decord/PyAV (the Cambrian-P venv); this test itself needs IPython + ipywidgets")
ap.add_argument("--helper", nargs="*", help=argparse.SUPPRESS)
args = ap.parse_args()
FRAMES_PY = args.frames_python
RESULTS = []


def check(name, cond, detail=""):
    RESULTS.append((name, bool(cond)))
    print(("PASS " if cond else "FAIL ") + name + ("" if cond else f"   [{detail}]"), flush=True)


CONFIG = "cfg-test-0001"
TMP = tempfile.mkdtemp(prefix="vsibench_review_test_")
SRC = os.path.join(TMP, "src")
os.makedirs(SRC)


def _in_frames_python(*helper_args):
    """Run one decode/encode helper in the interpreter that has decord/PyAV (the notebook kernel may not)."""
    import subprocess
    r = subprocess.run([FRAMES_PY, os.path.abspath(__file__), "--helper", *map(str, helper_args)],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"helper {helper_args[0]} failed in {FRAMES_PY}: {r.stderr[-2000:]}")
    return r.stdout


def make_video(path, n=40, fps=10, seed=0):
    try:
        import av
    except ImportError:
        _in_frames_python("make_video", path, n, fps, seed)
        return
    c = av.open(path, "w")
    s = c.add_stream("mpeg4", rate=fps)
    s.width, s.height, s.pix_fmt = 64, 48, "yuv420p"
    for i in range(n):
        a = np.zeros((48, 64, 3), np.uint8)
        a[:, :, 0], a[:, :, 1] = (i * 6) % 256, (seed * 40) % 256
        a[:, i % 64] = 255
        for pk in s.encode(av.VideoFrame.from_ndarray(a, format="rgb24")):
            c.mux(pk)
    for pk in s.encode():
        c.mux(pk)
    c.close()


def meta_for(path, k):
    """Frame provenance exactly as the inference worker records it (decord, linspace indices, sha1 per frame)."""
    try:
        from decord import VideoReader, cpu
    except ImportError:
        return json.loads(_in_frames_python("meta_for", path, k))
    vr = VideoReader(path, ctx=cpu(0), num_threads=0)
    total, fps = len(vr), float(vr.get_avg_fps())
    idx = L.sample_indices(total, k)
    frames = vr.get_batch(idx).asnumpy()
    return {"kind": "video_meta", "frame_indices": idx, "timestamps_s": [i / fps for i in idx], "total_frames": total,
            "fps": fps, "reader": "decord", "num_frames": k, "frame_size_hw": list(frames.shape[1:3]),
            "preprocessing": "expand2square + SigLIP processor (test)",
            "frame_sha1": [hashlib.sha1(np.ascontiguousarray(f).tobytes()).hexdigest() for f in frames]}


if args.helper:                      # helper mode (called by _in_frames_python): do one thing and exit
    if args.helper[0] == "make_video":
        make_video(args.helper[1], int(args.helper[2]), int(args.helper[3]), int(args.helper[4]))
    elif args.helper[0] == "meta_for":
        print(json.dumps(meta_for(args.helper[1], int(args.helper[2]))))
    shutil.rmtree(TMP, ignore_errors=True)
    sys.exit(0)


def build_world(out_root, n_videos, wrong_per_video=1, k=8, real_first=False, extra_unselected=True):
    """A synthetic run: n videos, each with `wrong_per_video` MC wrong candidates and one correct answer."""
    os.makedirs(out_root, exist_ok=True)
    W = {"OUT": out_root, "QA_BY_ID": {}, "SCORES": [], "OK": {}, "META": {}, "MEDIA": {}, "PATHS": {}}
    for v in range(n_videos):
        ds = ("arkitscenes", "scannet", "scannetpp")[v % 3]
        vk = f"{ds}/scene{v:03d}"
        path = os.path.join(SRC, f"{ds}_scene{v:03d}.mp4")
        if real_first and v == 0 and args.real_video:
            shutil.copyfile(args.real_video, path)
        elif not os.path.exists(path):
            make_video(path, seed=v)
        W["PATHS"][vk] = path
        W["META"][vk] = dict(meta_for(path, 32 if (real_first and v == 0 and args.real_video) else k), video_key=vk)
        W["MEDIA"][vk] = {"video_key": vk, "dataset": ds, "scene_name": f"scene{v:03d}", "safe_key": L.safe_key(vk),
                          "archive": f"{ds}.zip", "member": vk + ".mp4", "member_bytes": os.path.getsize(path), "crc32": 0}
        for j in range(wrong_per_video + 1):
            nid = v * 10 + j
            qid = f"vsibench-{nid}"
            qt = ("object_rel_distance", "route_planning", "obj_appearance_order")[j % 3]
            W["QA_BY_ID"][qid] = {"qa_id": qid, "native_id": nid, "video_key": vk, "dataset": ds, "scene_name": f"scene{v:03d}",
                                  "question_type": qt, "task_family": "multiple_choice", "question": f"Q{nid} <b>which</b>?",
                                  "options": ["A. chair", "B. table", "C. sofa", "D. bed"], "reference_answer": "A",
                                  "prompt": "...", "pruned": False}
            ans = "A" if j == wrong_per_video else "C"
            W["OK"][qid] = {"kind": "answer", "qa_id": qid, "video_key": vk, "status": "ok", "raw_answer": ans,
                            "config_hash": CONFIG, "answered_at": "2026-11-08T00:00:00Z", "attempt": 1}
            W["SCORES"].append({"qa_id": qid, "native_id": nid, "video_key": vk, "dataset": ds, "question_type": qt,
                                "task_family": "multiple_choice", "raw_answer": ans, "parsed_answer": ans, "parsed_value": None,
                                "reference_answer": "A", "official_metric": "accuracy",
                                "official_score": 1.0 if ans == "A" else 0.0,
                                "automatic_status": "correct" if ans == "A" else "wrong_candidate",
                                "candidate_reason": None if ans == "A" else "MC option C, benchmark answer A", "error": None})
    if extra_unselected:   # a wrong answer on a video OUTSIDE the selected sample: must never enter any queue
        W["SCORES"].append(dict(W["SCORES"][0], qa_id="vsibench-9999", video_key="scannet/outside", native_id=9999))
    W["SCORES_BY_ID"] = {s["qa_id"]: s for s in W["SCORES"]}
    W["SELECTED"] = sorted(W["MEDIA"])
    W["CTX"] = {"output_root": out_root, "venv_py": FRAMES_PY, "env": dict(os.environ), "lib_dir": HERE,
                "fetch_video": lambda vk: W["PATHS"][vk], "meta": W["META"], "qa_by_id": W["QA_BY_ID"],
                "scores_by_id": W["SCORES_BY_ID"], "config_hash": CONFIG}
    return W


def rctx(W):
    return {"output_root": W["OUT"], "config_hash": CONFIG, "selected_video_keys": W["SELECTED"], "qa_by_id": W["QA_BY_ID"],
            "ok": W["OK"], "scores_by_id": W["SCORES_BY_ID"], "media": W["MEDIA"], "reviews": R.load_reviews(W["OUT"]),
            "run_id": "test-run"}


def state(W, q, target=20, minimum=10):
    active, _ = R.queue_split(q)
    pv = R.evaluate(q["entries"], rctx(W))
    return pv, R.counts(q["entries"], active, pv, W["OUT"], target, minimum)


def good_final(entry, W, verdict="wrong", **kw):
    man = R.load_frames_manifest(R.case_dir(W["OUT"], entry))
    first = next(e for e in man["entries"] if e["frame_set"] == "input")
    f = {"verdict": verdict, "category": "visual_reasoning_error" if verdict == "wrong" else "",
         "evidence_status": "partial_in_input", "confidence": "medium",
         "citations": f"input frame={first['frame_index']} t={first['timestamp_s']} the chair is visible",
         "reviewed_scope": {"input": "all input contact sheets", "extra": ""},
         "observations": "A chair is at the left edge in input frame 0.", "rationale": "Benchmark answer A matches what is visible."}
    f.update(kw)
    return f


def review(W, entry, qid, final=None, answer="B"):
    q = W["QA_BY_ID"][qid]
    _, e1 = R.save_provisional(W["OUT"], q, CONFIG, "test-run", answer)
    if final is None:
        return e1
    cdir = R.case_dir(W["OUT"], entry)
    _, e2 = R.save_final(W["OUT"], q, CONFIG, final, R.load_frames_manifest(cdir), cdir,
                         W["SCORES_BY_ID"][qid]["automatic_status"])
    return e1 + e2


try:
    # ------------------------------------------------------------------ build 20 candidate videos and export them
    W = build_world(os.path.join(TMP, "run20"), 20, real_first=True)
    qpath = os.path.join(W["OUT"], "review_queue.json")
    Q = R.update_queue(qpath, W["SCORES"], W["QA_BY_ID"], W["SELECTED"], 20)
    active, reserve = R.queue_split(Q)
    check("queue: 20 distinct candidate videos active, none outside the selected sample",
          len(active) == 20 and len({e["video_key"] for e in active}) == 20
          and all(e["video_key"] in W["SELECTED"] for e in Q["entries"]), [e["video_key"] for e in Q["entries"]][:3])
    Q2 = R.candidate_entries(list(reversed(W["SCORES"])), W["QA_BY_ID"], W["SELECTED"])
    check("queue: deterministic regardless of score order", [e["video_key"] for e in Q2] == [e["video_key"] for e in Q["entries"]])
    probs = [p for e in active for p in R.export_candidate(e, W["CTX"])]
    check("export: all 20 candidates exported (video on Drive, decode check, input frames, sheets)", not probs, probs[:2])
    man0 = R.load_frames_manifest(R.case_dir(W["OUT"], active[0]))
    check("export: re-decoded input frames match the worker's frame hashes (inputs verified)",
          all(R.load_frames_manifest(R.case_dir(W["OUT"], e))["inputs_verified"] for e in active))
    if args.real_video:
        real = next(e for e in active if W["META"][e["video_key"]]["num_frames"] == 32)
        mr = R.load_frames_manifest(R.case_dir(W["OUT"], real))
        check("export: the real VSI-Bench video decodes and its 32 input frames verify",
              mr["inputs_verified"] and sum(e["frame_set"] == "input" for e in mr["entries"]) == 32, mr.get("verification"))
    ent = man0["entries"][0]
    check("12 frame manifest keeps frame index (int) and timestamp (s, float) in separate fields",
          isinstance(ent["frame_index"], int) and isinstance(ent["timestamp_s"], float) and ent["frame_set"] == "input")

    # ------------------------------------------------------------------ test 6: zero reviews -> zero verified
    pv, C = state(W, Q)
    check("6 zero reviews give zero verified failures", C["verified_wrong_video_count"] == 0
          and all(v["state"] == "unreviewed" for v in pv.values()), C)
    e0 = active[0]
    q0 = e0["primary_qa_id"]
    errs = R.save_provisional(W["OUT"], W["QA_BY_ID"][q0], CONFIG, "test-run", "   ")[1]
    check("6 saving an empty blind judgment is rejected and writes nothing",
          errs and not os.path.exists(R.review_path(W["OUT"], q0)), errs)
    R.save_provisional(W["OUT"], W["QA_BY_ID"][q0], CONFIG, "test-run", "cannot determine")
    blank = {"verdict": "", "category": "", "evidence_status": "", "confidence": "", "citations": "",
             "reviewed_scope": {"input": "", "extra": ""}, "observations": "", "rationale": ""}
    cdir0 = R.case_dir(W["OUT"], e0)
    _, errs = R.save_final(W["OUT"], W["QA_BY_ID"][q0], CONFIG, blank, R.load_frames_manifest(cdir0), cdir0, "wrong_candidate")
    _, errs2 = R.save_final(W["OUT"], W["QA_BY_ID"][q0], CONFIG, good_final(e0, W, observations=""),
                            R.load_frames_manifest(cdir0), cdir0, "wrong_candidate")
    pv, C = state(W, Q)
    check("6 'Save final' with defaults/blank observations is rejected; count stays 0",
          len(errs) >= 5 and any("observations" in x for x in errs2) and C["verified_wrong_video_count"] == 0
          and R.load_review(W["OUT"], q0)["stage"] == "provisional", (errs, errs2))
    check("6 the blind judgment cannot be replaced after the reveal",
          R.save_provisional(W["OUT"], W["QA_BY_ID"][q0], CONFIG, "test-run", "A")[1] != [])

    # ------------------------------------------------------------------ test 7: 8 wrong, 5 correct, 4 uncertain, 3 unreviewed
    plan = ["wrong"] * 8 + ["correct"] * 5 + ["uncertain"] * 4 + [None] * 3
    bad = []
    for e, verdict in zip(active, plan):
        if verdict is None:
            continue
        f = good_final(e, W, verdict=verdict)
        bad += review(W, e, e["primary_qa_id"], f) if e is not e0 else R.save_final(
            W["OUT"], W["QA_BY_ID"][q0], CONFIG, f, R.load_frames_manifest(cdir0), cdir0, "wrong_candidate")[1]
    pv, C = state(W, Q)
    check("7 20 exported = 8 wrong + 5 correct + 4 uncertain + 3 unreviewed -> 8 verified videos, minimum not met",
          not bad and C["verified_wrong_video_count"] == 8 and C["overridden_correct_video_count"] == 5
          and C["uncertain_video_count"] == 4 and C["final_review_video_count"] == 17 and C["exported_video_count"] == 20
          and C["candidate_video_count"] == 20 and not C["minimum_met"] and not C["target_met"], (bad[:2], C))
    gate = R.completion_gate(C, {"inference_complete": True})
    check("7 gate: minimum not met is reported as incomplete; inference completion kept separate",
          "Minimum NOT met" in gate["message"] and gate["inference_complete"] and not gate["case_export_complete"], gate)
    gate2 = R.completion_gate(dict(C, verified_wrong_video_count=12, minimum_met=True), {"inference_complete": False})
    check("7 gate: 12 verified -> minimum met, target shortfall, inference not complete stated",
          "minimum (10) met" in gate2["message"] and "not complete" in gate2["message"], gate2["message"])

    # ------------------------------------------------------------------ test 8: one video counts once; reviews are QA- and run-bound
    W8 = build_world(os.path.join(TMP, "run8"), 3, wrong_per_video=2)
    Q8 = R.update_queue(os.path.join(W8["OUT"], "review_queue.json"), W8["SCORES"], W8["QA_BY_ID"], W8["SELECTED"], 20)
    a8, _ = R.queue_split(Q8)
    for e in a8:
        R.export_candidate(e, W8["CTX"])
    e = a8[0]
    qa_a, qa_b = e["primary_qa_id"], e["supporting_qa_ids"][0]
    errs = review(W8, e, qa_a, good_final(e, W8)) + review(W8, e, qa_b, good_final(e, W8))
    pv, C = state(W8, Q8)
    check("8 two verified wrong questions on one video count as one video",
          not errs and len(pv[e["video_key"]]["verified_qa_ids"]) == 2 and C["verified_wrong_video_count"] == 1, (errs, C))
    e2 = a8[1]
    qa_c, qa_d = e2["primary_qa_id"], e2["supporting_qa_ids"][0]
    review(W8, e2, qa_c, good_final(e2, W8))
    shutil.copyfile(R.review_path(W8["OUT"], qa_c), R.review_path(W8["OUT"], qa_d))   # QA C's review filed under QA D
    res_d = R.validate_verified_failure(e2, W8["QA_BY_ID"][qa_d], W8["OK"][qa_d], R.load_review(W8["OUT"], qa_d),
                                        W8["MEDIA"][e2["video_key"]], score=W8["SCORES_BY_ID"][qa_d], output_root=W8["OUT"],
                                        config_hash=CONFIG, selected_video_keys=W8["SELECTED"])
    check("8 a review for QA A is not valid for QA B of the same video",
          not res_d["eligible"] and any("different question" in r for r in res_d["reasons"]), res_d["reasons"])
    res_run = R.validate_verified_failure(e2, W8["QA_BY_ID"][qa_c], dict(W8["OK"][qa_c], config_hash="other"),
                                          R.load_review(W8["OUT"], qa_c), W8["MEDIA"][e2["video_key"]],
                                          score=W8["SCORES_BY_ID"][qa_c], output_root=W8["OUT"], config_hash="other",
                                          selected_video_keys=W8["SELECTED"])
    check("8 a review from a different run/configuration is not valid",
          not res_run["eligible"] and any("different run" in r for r in res_run["reasons"]), res_run["reasons"])

    # ------------------------------------------------------------------ test 9: missing / broken assets and claims block verification
    W9 = build_world(os.path.join(TMP, "run9"), 6)
    Q9 = R.update_queue(os.path.join(W9["OUT"], "review_queue.json"), W9["SCORES"], W9["QA_BY_ID"], W9["SELECTED"], 20)
    a9, _ = R.queue_split(Q9)
    for e in a9:
        R.export_candidate(e, W9["CTX"])
        review(W9, e, e["primary_qa_id"], good_final(e, W9))
    pv, C = state(W9, Q9)
    check("9 baseline: 6 verified before damage", C["verified_wrong_video_count"] == 6, C)
    os.remove(os.path.join(R.case_dir(W9["OUT"], a9[0]), "video.mp4"))
    with open(os.path.join(R.case_dir(W9["OUT"], a9[1]), "video.mp4"), "r+b") as f:   # corrupt: re-check fails
        f.write(b"\0" * 4096)
    os.remove(os.path.join(R.case_dir(W9["OUT"], a9[1]), "video_check.json"))
    bad_export = R.export_candidate(a9[1], W9["CTX"])
    man2 = R.load_frames_manifest(R.case_dir(W9["OUT"], a9[2]))
    os.remove(os.path.join(R.case_dir(W9["OUT"], a9[2]), man2["entries"][3]["file"]))
    rp = R.review_path(W9["OUT"], a9[3]["primary_qa_id"])
    rv = json.load(open(rp))
    rv["history"][-1]["data"]["citations"] = [{"frame_set": "input", "frame_index": 99999, "timestamp_s": None, "note": ""}]
    rv["final"] = rv["history"][-1]["data"]
    L.write_json_atomic(rp, rv)
    m4 = R.load_frames_manifest(R.case_dir(W9["OUT"], a9[4]))
    err_extra = R.validate_final(good_final(a9[4], W9, evidence_status="found_only_in_extra_frames",
                                            category="evidence_not_in_input") | {"citations": [
                                                {"frame_set": "input", "frame_index": m4["entries"][0]["frame_index"],
                                                 "timestamp_s": None}]}, m4, R.case_dir(W9["OUT"], a9[4]))
    pv, C = state(W9, Q9)
    rs = {e["video_key"]: pv[e["video_key"]]["results"][0]["reasons"] for e in a9}
    check("9 missing video blocks verification with a reason", any("video missing" in r for r in rs[a9[0]["video_key"]]),
          rs[a9[0]["video_key"]])
    check("9 non-decodable video blocks verification with a reason", bad_export and any(
        "decode" in r or "changed" in r for r in rs[a9[1]["video_key"]]), (bad_export, rs[a9[1]["video_key"]]))
    check("9 missing input frame image blocks verification", any("input frame images are missing" in r
                                                                 for r in rs[a9[2]["video_key"]]), rs[a9[2]["video_key"]])
    check("9 invalid image citation blocks verification", any("not one of this video's exported" in r
                                                              for r in rs[a9[3]["video_key"]]), rs[a9[3]["video_key"]])
    check("9 unsupported evidence claim (extra-only without extra-frame evidence) is rejected",
          any("cite at least one valid extra frame" in x for x in err_extra), err_extra)
    check("9 counts: 2 still verified, damaged ones counted as invalid/missing evidence",
          C["verified_wrong_video_count"] == 2 and C["invalid_or_missing_evidence_video_count"] == 4, C)

    # ------------------------------------------------------------------ test 10: documented ambiguous evidence
    W10 = build_world(os.path.join(TMP, "run10"), 2)
    Q10 = R.update_queue(os.path.join(W10["OUT"], "review_queue.json"), W10["SCORES"], W10["QA_BY_ID"], W10["SELECTED"], 20)
    a10, _ = R.queue_split(Q10)
    for e in a10:
        R.export_candidate(e, W10["CTX"])
    amb = good_final(a10[0], W10, evidence_status="ambiguous", category="answer_failure_unresolved_evidence", citations="",
                     reviewed_scope={"input": "all 8 input frames; contact sheet 1", "extra": "1 fps whole video"})
    e_ok = review(W10, a10[0], a10[0]["primary_qa_id"], amb)
    e_vis = R.validate_final(dict(amb, category="visual_reasoning_error"), R.load_frames_manifest(R.case_dir(W10["OUT"], a10[0])),
                             R.case_dir(W10["OUT"], a10[0]))
    e_noscope = R.validate_final(dict(amb, reviewed_scope={"input": "", "extra": ""}),
                                 R.load_frames_manifest(R.case_dir(W10["OUT"], a10[0])), R.case_dir(W10["OUT"], a10[0]))
    pv, C = state(W10, Q10)
    check("10 wrong answer with documented ambiguous evidence qualifies as an answer failure",
          not e_ok and pv[a10[0]["video_key"]]["state"] == "verified_wrong", (e_ok, pv[a10[0]["video_key"]]["results"][0]))
    check("10 ...but cannot be labelled a visual-reasoning error, and needs a reviewed scope",
          any("does not fit" in x for x in e_vis) and any("reviewed scope" in x for x in e_noscope), (e_vis, e_noscope))
    fmt_s = dict(W10["SCORES_BY_ID"][a10[1]["primary_qa_id"]], automatic_status="format_error")
    e_fmt = R.validate_final(good_final(a10[1], W10), R.load_frames_manifest(R.case_dir(W10["OUT"], a10[1])),
                             R.case_dir(W10["OUT"], a10[1]), fmt_s["automatic_status"])
    check("format error confirmed as a failure must be labelled format_failure", any("format_failure" in x for x in e_fmt), e_fmt)

    # ------------------------------------------------------------------ test 11: correction decrements counts, offers reserve
    W11 = build_world(os.path.join(TMP, "run11"), 5)
    q11p = os.path.join(W11["OUT"], "review_queue.json")
    Q11 = R.update_queue(q11p, W11["SCORES"], W11["QA_BY_ID"], W11["SELECTED"], 3)
    a11, r11 = R.queue_split(Q11)
    for e in a11:
        R.export_candidate(e, W11["CTX"])
        review(W11, e, e["primary_qa_id"], good_final(e, W11))
    pv, C = state(W11, Q11, target=3, minimum=2)
    first_count = C["verified_wrong_video_count"]
    q = W11["QA_BY_ID"][a11[0]["primary_qa_id"]]
    cdir = R.case_dir(W11["OUT"], a11[0])
    _, errs = R.save_final(W11["OUT"], q, CONFIG, good_final(a11[0], W11, verdict="correct", rationale="the model's C is right; reference doubtful"),
                           R.load_frames_manifest(cdir), cdir, "wrong_candidate")
    pv, C = state(W11, Q11, target=3, minimum=2)
    offered = R.promote_reserve(q11p, Q11, {k: v["state"] for k, v in pv.items()}, 3)
    hist = R.load_review(W11["OUT"], q["qa_id"])["history"]
    check("11 correction decrements the verified count (3 -> 2) and keeps the history",
          not errs and first_count == 3 and C["verified_wrong_video_count"] == 2 and C["overridden_correct_video_count"] == 1
          and [h["event"] for h in hist] == ["provisional", "final", "final"], (errs, first_count, C))
    check("11 the next reserve candidate from the same sample is offered",
          len(offered) == 1 and offered[0]["video_key"] == r11[0]["video_key"] and offered[0]["video_key"] in W11["SELECTED"]
          and len(R.queue_split(Q11)[0]) == 4, offered)
    rv = R.load_review(W11["OUT"], q["qa_id"])
    rv["final"] = rv["history"][1]["data"]          # pretend the file still shows the old wrong verdict
    L.write_json_atomic(R.review_path(W11["OUT"], q["qa_id"]), rv)
    pv, _ = state(W11, Q11, target=3, minimum=2)
    check("11 a superseded wrong verdict never counts", pv[a11[0]["video_key"]]["state"] != "verified_wrong",
          pv[a11[0]["video_key"]]["results"][0]["reasons"])

    # ------------------------------------------------------------------ test 12: timestamp/index separation, provenance, extra frames
    e12 = a10[1]
    c12 = R.case_dir(W10["OUT"], e12)
    code_, out = R.run_extra_frames(e12, W10["CTX"], 1.0, ["0.5,1.5,10"])
    m12 = R.load_frames_manifest(c12)
    xs = [x for x in m12["entries"] if x["frame_set"] == "extra"]
    ins = [x for x in m12["entries"] if x["frame_set"] == "input"]
    check("12 extra frames are written apart from the inputs, with their own contact sheets",
          code_ == 0 and xs and m12.get("extra_sheets") and all(x["file"].startswith("frames_extra/") for x in xs)
          and len(ins) == 8, out[-200:])
    extra_only = next(x for x in xs if x["frame_index"] not in {i["frame_index"] for i in ins})
    _, err_set = R.resolve_citation({"frame_set": "input", "frame_index": extra_only["frame_index"]}, m12, c12)
    _, err_t = R.resolve_citation({"frame_set": "input", "frame_index": ins[2]["frame_index"], "timestamp_s": ins[2]["timestamp_s"] + 2}, m12, c12)
    hit, err_ok = R.resolve_citation({"frame_set": "extra", "frame_index": None, "timestamp_s": extra_only["timestamp_s"]}, m12, c12)
    cits, perr = R.parse_citations("input frame=12 t=1.2 chair\nextra t=3.5\nwhatever 3")
    check("12 an input citation of an extra-only frame is rejected", err_set and "not one of" in err_set, err_set)
    check("12 index and timestamp that disagree are rejected", err_t and "disagree" in err_t, err_t)
    check("12 a timestamp-only extra citation resolves to that extra frame", err_ok is None and hit["file"] == extra_only["file"])
    check("12 citation parser keeps frame index and seconds in separate fields",
          cits[0] == {"frame_set": "input", "frame_index": 12, "timestamp_s": 1.2, "note": "chair"}
          and cits[1]["frame_index"] is None and cits[1]["timestamp_s"] == 3.5 and len(perr) == 1, (cits, perr))
    tamper = os.path.join(TMP, "tamper")
    Wt = build_world(tamper, 1)
    Wt["META"][Wt["SELECTED"][0]]["frame_sha1"][0] = "0" * 40
    Qt = R.update_queue(os.path.join(tamper, "review_queue.json"), Wt["SCORES"], Wt["QA_BY_ID"], Wt["SELECTED"], 20)
    at, _ = R.queue_split(Qt)
    R.export_candidate(at[0], Wt["CTX"])
    mt = R.load_frames_manifest(R.case_dir(tamper, at[0]))
    e_suff = R.validate_final(good_final(at[0], Wt, evidence_status="sufficient_in_input"), mt, R.case_dir(tamper, at[0]))
    check("12 frames that do not reproduce the recorded inputs are labelled unverified; sufficient_in_input refused",
          mt["inputs_verified"] is False and "1 of 8" in mt["verification"]
          and any("verified model inputs" in x for x in e_suff), (mt.get("verification"), e_suff))

    # ------------------------------------------------------------------ final export + report + test 13 packaging (run20)
    pv, C = state(W, Q)
    gate = R.completion_gate(C, {"inference_complete": True})
    act, _ = R.queue_split(Q)
    FINAL = R.build_final(W["OUT"], pv, act, rctx(W), C, gate, 20)
    R.write_summary(os.path.join(W["OUT"], "summary.csv"), pv, act, rctx(W))
    R.report_html(W["OUT"], FINAL, pv, {"selected videos": 29, "mode": "full"})
    import csv as _csv
    rows = list(_csv.DictReader(open(os.path.join(W["OUT"], "summary.csv"))))
    html_ = open(os.path.join(W["OUT"], "report.html")).read()
    check("final manifest, summary.csv, report and counts agree (8)",
          FINAL["verified_video_count_in_manifest"] == 8 == C["verified_wrong_video_count"]
          and sum(r["verified"] == "True" for r in rows) == 8 and html_.count("<video ") == 8
          and len(os.listdir(os.path.join(W["OUT"], "failure_videos"))) == 8
          and len(os.listdir(os.path.join(W["OUT"], "cases"))) == 8, (FINAL["verified_video_count_in_manifest"], C))
    check("report escapes dataset text", "Q0 &lt;b&gt;which&lt;/b&gt;?" in html_ or "&lt;b&gt;which" in html_)
    import re as _re
    links = _re.findall(r"(?:src|href)='([^']+)'", html_)
    check("report links are relative and resolve", links and all(not l.startswith("/") and
                                                                 os.path.exists(os.path.join(W["OUT"], l)) for l in links),
          [l for l in links if not os.path.exists(os.path.join(W["OUT"], l))][:3])
    zpath = os.path.join(TMP, "pkg.zip")
    R.package(W["OUT"], zpath)
    names = zipfile.ZipFile(zpath).namelist()
    verified_keys = {c["safe_key"] for c in FINAL["cases"]}
    overridden = [v["entry"]["safe_key"] for v in pv.values() if v["state"] == "overridden_correct"]
    check("13 package holds only the manifest's verified videos/cases; overridden candidates stay out",
          {n.split("/")[1][:-4] for n in names if n.startswith("failure_videos/")} == verified_keys
          and {n.split("/")[1] for n in names if n.startswith("cases/")} == verified_keys
          and not any(k in n for k in overridden for n in names) and not any(n.startswith("candidate_cases/") for n in names))
    # correcting a verified case afterwards removes it from the final folders (history kept)
    v0 = FINAL["cases"][0]
    ev = next(e for e in act if e["video_key"] == v0["video_key"])
    cdir = R.case_dir(W["OUT"], ev)
    R.save_final(W["OUT"], W["QA_BY_ID"][ev["primary_qa_id"]], CONFIG, good_final(ev, W, verdict="uncertain"),
                 R.load_frames_manifest(cdir), cdir, "wrong_candidate")
    pv, C = state(W, Q)
    FINAL2 = R.build_final(W["OUT"], pv, act, rctx(W), C, R.completion_gate(C, {}), 20)
    check("13 after a correction the final folders drop the case; its candidate history remains",
          FINAL2["verified_video_count_in_manifest"] == 7 and not os.path.exists(os.path.join(W["OUT"], "cases", v0["safe_key"]))
          and os.path.exists(os.path.join(cdir, "video.mp4")) and len(R.load_review(W["OUT"], ev["primary_qa_id"])["history"]) == 3)

    # ------------------------------------------------------------------ short review form: parsing and mapping
    fm_ = {"input_sheets": ["s1.jpg"], "entries": [
        {"frame_set": "input", "position": 0, "frame_index": 0, "timestamp_s": 0.0, "file": "a"},
        {"frame_set": "input", "position": 1, "frame_index": 30, "timestamp_s": 1.0, "file": "b"},
        {"frame_set": "extra", "frame_index": 45, "timestamp_s": 1.5, "file": "c"}]}
    c_, e_ = R.parse_frame_refs("#1, f45; 0", fm_)
    check("short form: '#1, f45; 0' -> input frame 30, extra frame 45, input frame 0",
          not e_ and [(x["frame_set"], x["frame_index"]) for x in c_] == [("input", 30), ("extra", 45), ("input", 0)], (c_, e_))
    check("short form: frames text round-trips", R.frames_text(c_, fm_) == "#1, f45, #0")
    check("short form: unknown or malformed frame numbers are explained",
          len(R.parse_frame_refs("#7 f99 abc", fm_)[1]) == 3)
    f_, e_ = R.quick_final("wrong", "not_found_in_reviewed_frames", "low", "", "searched all sheets", fm_)
    check("short form: 'couldn't find it' needs no frame numbers; category unresolved-evidence; scope recorded",
          not e_ and f_["category"] == "answer_failure_unresolved_evidence" and f_["reviewed_scope"]["input"]
          and f_["observations"] == f_["rationale"] == "searched all sheets")
    f_, e_ = R.quick_final("wrong", "sufficient_in_input", "high", "#1", "x", fm_, automatic_status="format_error")
    check("short form: a confirmed failure on an unreadable answer is labelled format_failure",
          f_["category"] == "format_failure")
    f_, e_ = R.quick_final("correct", "ambiguous", "medium", "", "x", fm_)
    check("short form: an override carries no failure category", f_["category"] == "" and f_["verdict"] == "correct")
    f_, e_ = R.quick_final("", "", "", "", "", fm_)
    check("short form: an empty form still fails the unchanged validator",
          R.validate_final(f_, fm_, None) and not R.validate_final(
              R.quick_final("wrong", "found_only_in_extra_frames", "high", "f45", "seen in extra", fm_)[0], fm_, None),
          R.validate_final(R.quick_final("wrong", "found_only_in_extra_frames", "high", "f45", "seen in extra", fm_)[0], fm_, None))

    # ------------------------------------------------------------------ the review interface (cells 8a-8c, 9a-9b) driven like a reviewer
    import review_cells
    import IPython.display as _ipd          # no notebook here: keep widget reprs out of the test log
    _ipd.display = lambda *a, **k: None
    _ipd.clear_output = lambda *a, **k: None
    WU = build_world(os.path.join(TMP, "ui"), 3)
    ns = {"__name__": "__main__", "os": os, "json": json, "L": L, "OUTPUT_ROOT": WU["OUT"], "RUN_ID": "test-run",
          "RUN_MODE": "full", "CONFIG_HASH": CONFIG, "IDENTITY": {"DATASET_REPO": "nyu-visionx/VSI-Bench", "DATASET_CONFIG": "full",
                                                                  "DATASET_SPLIT": "test", "DATASET_REVISION": "rev",
                                                                  "MODEL_REPO": "m", "MODEL_REVISION": "r", "NUM_FRAMES": 8},
          "SEL": {"population_videos": 288, "selected_videos": 29, "selected_video_keys": WU["SELECTED"]},
          "QA_BY_ID": WU["QA_BY_ID"], "MANIFEST": list(WU["MEDIA"].values()), "MANIFEST_BY_VIDEO": WU["MEDIA"],
          "SCORES": WU["SCORES"], "SCORES_BY_ID": WU["SCORES_BY_ID"], "OK": WU["OK"], "META": WU["META"],
          "COVERAGE": {"inference_complete": False, "selected": {"videos": 3}}, "TARGET_FAILURE_VIDEOS": 2,
          "MIN_FAILURE_VIDEOS": 1, "VENV_PY": FRAMES_PY, "WORKER_ENV": dict(os.environ),
          "fetch_video": lambda vk: WU["PATHS"][vk]}
    code_cells = [src.replace("/content/vsibench", HERE) for kind, src in review_cells.CELLS if kind == "code"]
    for src in code_cells[:3]:
        exec(compile(src, "cell", "exec"), ns)
    UI = ns["UI"]
    rp = R.review_path(WU["OUT"], UI["qa_id"])
    check("UI: answers hidden before the blind judgment", not UI["revealed"]() and "answers" not in UI and not os.path.exists(rp))
    check("UI: input contact sheets are shown as images", UI["input_sheets"] and all(len(w.value) > 100 for w in UI["input_sheets"]))
    UI["full_pos"].value = 3
    UI["full_show"].click()
    check("UI: a full-resolution input frame can be opened", len(UI["full_img"].value) > 100)
    UI["save_provisional"].click()
    check("UI: empty blind judgment shows a validation error and saves nothing",
          "enter your own answer" in UI["prov_msg"].value and not os.path.exists(rp))
    UI["provisional_answer"].value = "B"
    UI["save_provisional"].click()
    UI = ns["UI"]
    check("UI: blind judgment saved, answers revealed", UI["revealed"]() and json.load(open(rp))["stage"] == "provisional"
          and "Benchmark answer" in UI["answers"].value)
    UI["save_final"].click()
    check("UI: 'Save final' with default fields is rejected visibly; count unchanged",
          "Not saved" in UI["final_msg"].value and ns["_LAST"]["counts"]["verified_wrong_video_count"] == 0
          and json.load(open(rp))["stage"] == "provisional", UI["final_msg"].value[:200])
    UI["windows"].value = "0.5,1.5,10"
    UI["extra"].click()
    check("UI: extra-frame contact sheets are displayed as images in the interface",
          len(UI["extra_images"].children) >= 1 and all(len(w.value) > 100 for w in UI["extra_images"].children),
          UI["extra_msg"].value[:200])
    man_u = R.load_frames_manifest(R.case_dir(WU["OUT"], UI["case"]))
    # short form: evidence "partly visible" without frame numbers is refused with a plain message
    UI["verdict"].value, UI["evidence"].value, UI["confidence"].value = "wrong", "partial_in_input", "high"
    UI["notes"].value = "Chair at the left in #3; the benchmark answer A is supported, the model said C."
    UI["save_final"].click()
    check("UI (short form): 'partly visible' without frame numbers is refused with a plain message",
          "type the frame number" in ns["UI"]["final_msg"].value and json.load(open(rp))["stage"] == "provisional",
          ns["UI"]["final_msg"].value[:300])
    UI = ns["UI"]
    UI["verdict"].value, UI["evidence"].value, UI["confidence"].value = "wrong", "partial_in_input", "high"
    UI["frames"].value = "#99"
    UI["notes"].value = "Chair at the left in #3; the benchmark answer A is supported, the model said C."
    UI["save_final"].click()
    check("UI (short form): a frame number that is not on the sheets is refused", "#99" in ns["UI"]["final_msg"].value
          and json.load(open(rp))["stage"] == "provisional", ns["UI"]["final_msg"].value[:300])
    UI = ns["UI"]
    UI["verdict"].value, UI["evidence"].value, UI["confidence"].value = "wrong", "partial_in_input", "high"
    UI["frames"].value = "#3"
    UI["notes"].value = "Chair at the left in #3; the benchmark answer A is supported, the model said C."
    UI["save_final"].click()
    UI = ns["UI"]
    saved = R.current_final(json.load(open(rp)))
    third = next(x for x in man_u["entries"] if x["frame_set"] == "input" and x["position"] == 3)
    check("UI: valid final review saved; verified count 1; validator shown", ns["_LAST"]["counts"]["verified_wrong_video_count"] == 1
          and "Final review saved" in UI["final_msg"].value and "verified failure" in UI["status"].value, UI["final_msg"].value)
    check("UI (short form): #3 is saved as a structured input citation of that frame; category and scope filled in",
          saved["citations"][0]["frame_set"] == "input" and saved["citations"][0]["frame_index"] == third["frame_index"]
          and saved["category"] == "visual_reasoning_error" and "input frames" in saved["reviewed_scope"]["input"]
          and "extra frames requested" in saved["reviewed_scope"]["extra"], saved)
    check("UI (short form): reopening shows the saved answers, frames as #3", UI["frames"].value == "#3"
          and UI["evidence"].value == "partial_in_input" and UI["verdict"].value == "wrong", UI["frames"].value)
    n_active = len(ns["ACTIVE"])
    UI["verdict"].value = "correct"
    UI["notes"].value = "On a closer look the model's answer C is right; the reference looks wrong."
    UI["save_final"].click()
    UI = ns["UI"]
    C_ui = ns["_LAST"]["counts"]
    check("UI: override to correct decrements the count and offers the next reserve candidate",
          C_ui["verified_wrong_video_count"] == 0 and C_ui["overridden_correct_video_count"] == 1
          and len(ns["ACTIVE"]) == n_active + 1 and "Next reserve candidate offered" in UI["final_msg"].value,
          (C_ui, UI["final_msg"].value))
    check("UI: progress line and picker use the shared counts", "Verified failure videos: 0" in ns["progress"].value
          and len(ns["pick"].options) == len(ns["ACTIVE"]))
    ns["show"](1)
    check("UI: switching case hides the next case's answers again", not ns["UI"]["revealed"]())
    for src in code_cells[3:]:
        exec(compile(src.replace("MAKE_ZIP = False", "MAKE_ZIP = True"), "cell", "exec"), ns)
    cov = json.load(open(os.path.join(WU["OUT"], "coverage.json")))
    check("9a/9b: manifest, coverage and gate written from the same counts; zip built",
          cov["verification"]["verified_wrong_video_count"] == 0 and "Minimum NOT met" in cov["gate"]["message"]
          and os.path.exists(os.path.join(TMP, "test-run_verified_failures.zip")))
finally:
    if not args.keep:
        shutil.rmtree(TMP, ignore_errors=True)
    else:
        print("kept", TMP)

n_fail = sum(1 for _, ok in RESULTS if not ok)
print(f"\nreview self-test: {len(RESULTS) - n_fail} passed, {n_fail} failed")
sys.exit(1 if n_fail else 0)

`vsibench_diagnostics_selftest.py`

In [ ]:
%%writefile /content/vsibench/vsibench_diagnostics_selftest.py
"""Regression tests for the review-queue diagnostics (no GPU, no dataset): the brief's cases 2-13 on small fixtures
built with the real shard writer, merge, scoring-state and queue helpers; case 14 checks the review cell templates.
Case 1 (RUN_MODE defaults to full) is checked on the notebook itself by the build check. Exits nonzero on failure."""
import hashlib
import json
import os
import shutil
import sys
import tempfile

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import vsibench_lib as L  # noqa: E402
import vsibench_review as R  # noqa: E402
import vsibench_diagnostics as D  # noqa: E402

RESULTS = []
OLD_MESSAGE = "No candidates to review yet " + "(run sections 6-7 in full mode"   # split so this file does not contain it
H, SCORER = "cfg-test", "scorer-sha"


def check(name, cond, detail=""):
    RESULTS.append((name, bool(cond)))
    print(("PASS " if cond else "FAIL ") + name + ("" if cond else f"   [{str(detail)[:3000]}]"), flush=True)


def make_run(root, n_videos=3, per_video=3, last_mode=None):
    out = os.path.join(root, "run")
    os.makedirs(os.path.join(out, "logs"), exist_ok=True)
    keys = [f"scannet/scene{v:02d}" for v in range(n_videos)]
    qa = [{"qa_id": f"vsibench-{v * 10 + i}", "native_id": v * 10 + i, "video_key": k, "dataset": "scannet",
           "question_type": "object_counting", "task_family": "numeric", "question": "How many?", "options": None,
           "reference_answer": "2", "prompt": "p"} for v, k in enumerate(keys) for i in range(per_video)]
    L.write_jsonl_atomic(os.path.join(out, "qa_manifest.jsonl"), qa)
    L.write_json_atomic(os.path.join(out, "selection.json"), {"selected_videos": n_videos, "selected_questions": len(qa),
                                                              "selected_video_keys": keys})
    L.write_json_atomic(os.path.join(out, "run_config.json"), {"config_hash": H, "identity": {"SCORER": {"sha256": SCORER}}})
    if last_mode:
        L.write_json_atomic(os.path.join(out, "run_manifest.json"), {"last_run": {"mode": last_mode}})
    return out, qa


def answer(out, qa_rows, text=lambda q: "2", error=None):
    """One durable shard per video, as the worker writes them."""
    by = {}
    for q in qa_rows:
        by.setdefault(q["video_key"], []).append(q)
    for v, qs in by.items():
        rows = [{"kind": "answer", "qa_id": q["qa_id"], "video_key": v, "config_hash": H,
                 "answered_at": L.time.strftime("%Y-%m-%dT%H:%M:%S.") + str(L.time.time_ns()),
                 **({"status": "error", "error": error} if error else {"status": "ok", "raw_answer": text(q)})}
                for q in qs]
        L.write_shard(os.path.join(out, "predictions"), L.safe_key(v) + f"-{L.time.time_ns()}", rows, H)


def score(out, qa, status_of, record=True):
    """What 7a does: merge, score (statuses supplied by the fixture), write scores.jsonl and the scoring state."""
    import vsibench_pipeline as P
    ok, failed, _, _ = P.gather_results(os.path.join(out, "predictions"), "", H)
    missing = P.missing_media(os.path.join(out, "logs", "media_missing.jsonl"))
    scores = []
    for q in qa:
        if q["qa_id"] in ok:
            st = status_of(q)
            scores.append({"qa_id": q["qa_id"], "video_key": q["video_key"], "automatic_status": st,
                           "official_score": {"correct": 1.0, "imperfect": 0.5}.get(st, 0.0),
                           "raw_answer": ok[q["qa_id"]]["raw_answer"]})
        elif q["qa_id"] in failed or q["video_key"] in missing:
            scores.append({"qa_id": q["qa_id"], "video_key": q["video_key"], "automatic_status": "execution_error",
                           "official_score": None, "raw_answer": None})
    L.write_jsonl_atomic(os.path.join(out, "scores.jsonl"), scores)
    if record:
        D.record_scoring(out, D.prediction_fingerprint(qa, ok, failed, missing, H, SCORER),
                         D.scores_fingerprint(scores, [q["qa_id"] for q in qa]), len(scores))
    return scores


def queue(out, scores, qa, active_keys=None, record=True, export=False):
    """A review queue as 8a saves it (entries + active list), optionally with complete exports."""
    elig = {}
    for s in scores:
        if s["automatic_status"] in R.CANDIDATE_STATUSES:
            elig.setdefault(s["video_key"], s["qa_id"])
    entries = [{"video_key": v, "safe_key": L.safe_key(v), "primary_qa_id": q, "supporting_qa_ids": []}
               for v, q in sorted(elig.items())]
    active = list(elig) if active_keys is None else active_keys
    L.write_json_atomic(os.path.join(out, "review_queue.json"), {"entries": entries, "active": sorted(active)})
    if record:
        D.record_queue(out, D.scores_fingerprint(scores, [q["qa_id"] for q in qa]), len(entries))
    if export:
        for e in entries:
            cdir = R.case_dir(out, e)
            os.makedirs(cdir, exist_ok=True)
            open(os.path.join(cdir, "video.mp4"), "wb").write(b"v")
            L.write_json_atomic(os.path.join(cdir, "video_check.json"), {"ok": True})
            L.write_json_atomic(os.path.join(cdir, "frames_manifest.json"), {"input_sheets": ["s.jpg"], "entries": []})


ALL_NAMES = {k: object() for k in D.DEFINED_IN}


def diag(out, mode="full", g=None, needs=D.NEEDS_8C):
    g = dict(ALL_NAMES if g is None else g)
    if mode is not None:
        g["RUN_MODE"] = mode
    else:
        g.pop("RUN_MODE", None)
    rt = D.collect_runtime_state({k: v for k, v in g.items() if k not in ("SCORES", "SEL", "QA", "ACTIVE", "RESERVE")})
    rt["present"] = {k: k in g for k in D.DEFINED_IN}
    d = D.diagnose_candidate_state(rt, D.collect_persisted_state(out, H, None), needs)
    return d, D.render_candidate_diagnostic(d), [f["code"] for f in d["findings"]]


def tree_hash(root):
    h = hashlib.sha256()
    for d, _, fs in sorted(os.walk(root)):
        for f in sorted(fs):
            p = os.path.join(d, f)
            h.update(p.encode())
            h.update(open(p, "rb").read())
    return h.hexdigest()


def no_switch_advice(text):
    return 'RUN_MODE = "full"' not in text and "switch to full" not in text.lower() and "in full mode" not in text.lower()


tmp = tempfile.mkdtemp(prefix="vsibench_diag_test_")
try:
    # 2. full mode, empty queue, pending predictions
    out, qa = make_run(os.path.join(tmp, "c2"), last_mode="pilot")
    answer(out, qa[:2])
    d, text, codes = diag(out, "full")
    check("2 full mode + pending predictions: 'inference has not successfully answered the selected subset'",
          "full_incomplete" in codes and "Full mode is selected, but inference has not successfully answered" in text
          and d["predictions"]["pending"] == 7, text)
    check("2 ...and never tells the user to switch to full", no_switch_advice(text), text)
    check("2 runtime mode and last recorded inference mode are shown separately",
          "Current runtime mode: full" in text and "Last recorded inference mode: pilot" in text, text)

    # 3. pilot mode, partial coverage
    d, text, codes = diag(out, "pilot")
    f = next(x for x in d["findings"] if x["code"] == "pilot_partial")
    check("3 pilot + partial coverage explains pilot scope and the full-run sequence",
          "Pilot mode is selected" in f["text"] and 'RUN_MODE = "full" in 1a, run 1a, then 6b' in f["next"]
          and "7a" in f["next"] and "8a" in f["next"], f)

    # 4. predictions saved, no scoring
    out, qa = make_run(os.path.join(tmp, "c4"))
    answer(out, qa)
    d, text, codes = diag(out)
    check("4 saved predictions without scores -> run 7a, not 'no wrong answers'",
          "scoring_missing" in codes and "Run 7a" in text and "all_correct" not in codes and "correct" not in
          json.dumps(d.get("status_counts")), text)

    # 5. changed answer text, same count
    out, qa = make_run(os.path.join(tmp, "c5"))
    answer(out, qa, text=lambda q: "2")
    score(out, qa, lambda q: "correct")
    d, _, codes = diag(out)
    fresh = d["scoring"]["state"]
    shutil.rmtree(os.path.join(out, "predictions"))
    answer(out, qa, text=lambda q: "7")
    d, text, codes = diag(out)
    check("5 scoring is current right after 7a", fresh == "current", fresh)
    check("5 changed answer text with unchanged question count makes scoring stale",
          d["scoring"]["state"] == "stale" and "scoring_stale" in codes and d["predictions"]["ok"] == len(qa), text)

    # 6. current scores with eligible failures, queue missing / stale
    out, qa = make_run(os.path.join(tmp, "c6"))
    answer(out, qa)
    sc = score(out, qa, lambda q: "wrong_candidate" if q["native_id"] % 10 == 0 else "correct")
    d, text, codes = diag(out)
    check("6 eligible failures + no queue -> run 8a (with the eligible video count)",
          "queue_missing" in codes and "Run 8a" in text and d["queue_eligible_videos"] == 3, text)
    queue(out, sc, qa)
    sc2 = score(out, qa, lambda q: "wrong_candidate" if q["native_id"] % 10 in (0, 1) else "correct")
    d, text, codes = diag(out)
    check("6 scores changed after the queue was built -> queue stale, run 8a", "queue_stale" in codes, text)

    # 7. all correct: complete vs incomplete
    out, qa = make_run(os.path.join(tmp, "c7"))
    answer(out, qa)
    score(out, qa, lambda q: "correct")
    d, text, codes = diag(out)
    check("7 all correct on a complete subset -> no eligible failures, denominator given",
          "all_correct" in codes and "all 9 successfully scored answers are correct" in text
          and "Every selected question has an outcome" in text, text)
    out, qa = make_run(os.path.join(tmp, "c7b"))
    answer(out, qa[:6])
    answer(out, qa[6:7], error="RuntimeError: CUDA error")
    score(out, qa, lambda q: "correct")
    d, text, codes = diag(out)
    check("7 the same on an incomplete subset discloses pending and failed outcomes",
          "all_correct" in codes and "2 pending, 1 with errors" in text, text)

    # 8. only correct + imperfect
    out, qa = make_run(os.path.join(tmp, "c8"))
    answer(out, qa)
    score(out, qa, lambda q: "imperfect" if q["native_id"] % 2 else "correct")
    d, text, codes = diag(out)
    check("8 only correct/imperfect -> partial-credit policy explained, failure_candidates.csv named",
          "only_correct_imperfect" in codes and "failure_candidates.csv" in text and "partial" in text
          and d["inspection_candidates"] == 3 and d["queue_eligible_qa"] == 0, text)

    # 9. execution errors kept apart
    out, qa = make_run(os.path.join(tmp, "c9"))
    answer(out, qa[:3], error="decode: corrupt frame")
    answer(out, qa[3:])
    L.Journal(os.path.join(out, "logs", "media_missing.jsonl")).append(
        {"video_key": qa[3]["video_key"], "reason": "BadZipFile: Bad CRC-32"})
    score(out, qa, lambda q: "correct")
    d, text, codes = diag(out)
    check("9 execution errors and missing media are counted separately, never as wrong answers",
          "errors_present" in codes and d["predictions"]["execution_errors"] == 3 and d["queue_eligible_qa"] == 0
          and "decode x3" in text and "media unavailable" in text and "not wrong answers" in text, text)

    # 10. reserve but no active
    out, qa = make_run(os.path.join(tmp, "c10"))
    answer(out, qa)
    sc = score(out, qa, lambda q: "wrong_candidate" if q["native_id"] % 10 == 0 else "correct")
    queue(out, sc, qa, active_keys=[])
    d, text, codes = diag(out)
    check("10 reserve entries with no active entries -> queue state, not 'no failures'",
          "queue_no_active" in codes and "not an absence of failures" in text and "all_correct" not in codes, text)

    # 11. active candidates, exports missing
    queue(out, sc, qa)
    d, text, codes = diag(out)
    check("11 active candidates without exports -> run 8b", "exports_missing" in codes and "Run 8b" in text, text)
    queue(out, sc, qa, export=True)
    d, text, codes = diag(out)
    check("11 ...and once exported the candidates are reported ready", codes == ["ready"], codes)

    # 12. missing globals; old artifacts without fingerprints; unknown mode
    d, text, codes = diag(out, mode=None, g={"OUTPUT_ROOT": out})
    check("12 missing globals -> named prerequisites and the cells that define them, no traceback",
          "prerequisites_missing" in codes and d["missing_prerequisites"].get("QUEUE") == "8a" and "Run cells 1a, 1c" in text
          and "Current runtime mode: unknown" in text, text)
    dn = D.diagnose_candidate_state(D.collect_runtime_state({}), D.collect_persisted_state(None))
    check("12 no OUTPUT_ROOT -> 'nothing can be diagnosed', no fabricated counts",
          [f["code"] for f in dn["findings"]][-1] == "no_output" and "predictions" not in dn
          and "Saved predictions: unknown" in D.render_candidate_diagnostic(dn), D.render_candidate_diagnostic(dn))
    out, qa = make_run(os.path.join(tmp, "c12"))
    answer(out, qa)
    sc = score(out, qa, lambda q: "wrong_candidate", record=False)
    queue(out, sc, qa, record=False)
    d, text, codes = diag(out)
    check("12 an older run without fingerprints -> freshness unknown, rerun 7a then 8a, nothing discarded",
          d["scoring"]["state"] == "unknown" and "scoring_unknown" in codes and "Rerun 7a" in text
          and os.path.exists(os.path.join(out, "scores.jsonl")), text)

    # 13. read-only
    before = {c: tree_hash(os.path.join(tmp, c)) for c in os.listdir(tmp)}
    for c in os.listdir(tmp):
        for mode in ("full", "pilot", None):
            diag(os.path.join(tmp, c, "run"), mode)
    after = {c: tree_hash(os.path.join(tmp, c)) for c in os.listdir(tmp)}
    check("13 diagnostics change no prediction, score, queue, selection, review or marker file", before == after)

    # in full mode no fixture ever produced 'switch to full' advice
    texts = [diag(os.path.join(tmp, c, "run"), "full")[1] for c in os.listdir(tmp)]
    check("2/13 full mode never yields 'switch to full' advice in any state", all(no_switch_advice(t) for t in texts))

    # 14. the shipped review cells use the diagnostics, and the old unconditional message is gone
    import review_cells
    code = "\n".join(src for kind, src in review_cells.CELLS if kind == "code")
    c8c = next(src for kind, src in review_cells.CELLS if "@title 8c." in src)
    c8a = next(src for kind, src in review_cells.CELLS if "@title 8a." in src)
    check("14 the old unconditional 8c message is gone from the review cells",
          OLD_MESSAGE not in code)
    check("14 8a records the queue fingerprint and shows the diagnostic; 8c renders it when the queue is empty",
          "D.record_queue(" in c8a and "render_candidate_diagnostic" in c8a
          and "render_candidate_diagnostic(_DIAG, \"Review queue is empty\")" in c8c and "_READY" in c8c)
finally:
    shutil.rmtree(tmp, ignore_errors=True)

failed = [n for n, ok in RESULTS if not ok]
print(f"\n{len(RESULTS) - len(failed)}/{len(RESULTS)} checks passed")
sys.exit(1 if failed else 0)

`vsibench_worker_selftest.py`

In [ ]:
%%writefile /content/vsibench/vsibench_worker_selftest.py
"""Runs the REAL vsibench_worker.py on a real video with only the model replaced by a stand-in (tiny fake torch /
cambrianp / lmms_eval packages put first on PYTHONPATH of the worker subprocess). Checks the worker's own code:
decoding and frame provenance, the exact prompt and generation settings handed to the model, the saved answer rows
and shards, retries, out-of-memory stop, deliberate stop, and that a bookkeeping bug stops the worker instead of being
recorded as model errors. Takes seconds; no GPU. Run it with the interpreter the real worker uses (needs decord).

  python vsibench_worker_selftest.py --video <real mp4>
"""
import argparse
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import vsibench_lib as L  # noqa: E402
import vsibench_pipeline as P  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--video", required=True)
ap.add_argument("--worker", default=os.path.join(HERE, "vsibench_worker.py"))
args = ap.parse_args()
args.video = os.path.abspath(args.video)
RESULTS = []


def check(name, cond, detail=""):
    RESULTS.append((name, bool(cond)))
    print(("PASS " if cond else "FAIL ") + name + ("" if cond else f"   [{str(detail)[-1500:]}]"), flush=True)


STUBS = {
    "torch/__init__.py": '''
__version__ = "stub"
class version:
    cuda = "stub"
class _OOM(RuntimeError):
    pass
class cuda:
    OutOfMemoryError = _OOM
    @staticmethod
    def get_device_name(i):
        return "stand-in GPU"
    @staticmethod
    def empty_cache():
        pass
''',
    "cambrianp/__init__.py": "",
    "cambrianp/utils.py": "def cast_cambrian_config_to_llava_ov_style(config, model_path):\n    return config\n",
    "cambrianp/constants.py": 'DEFAULT_IMAGE_TOKEN = "<image>"\nIMAGE_TOKEN_INDEX = -200\n',
    "cambrianp/conversation.py": '''
class _Conv:
    roles = ("<|im_start|>user", "<|im_start|>assistant")
    def __init__(self):
        self.messages = []
    def copy(self):
        return _Conv()
    def append_message(self, role, msg):
        self.messages.append((role, msg))
    def get_prompt(self):
        return "".join(f"{r}\\n{m}<|im_end|>\\n" if m is not None else f"{r}\\n" for r, m in self.messages)
conv_templates = {"qwen_1_5": _Conv()}
''',
    "cambrianp/mm_utils.py": '''
class Ids:
    def __init__(self, prompt):
        self.prompt = prompt
    def unsqueeze(self, i):
        return self
    def to(self, device):
        return self
    def ne(self, pad):
        return ("mask", self.prompt)
def tokenizer_image_token(prompt, tokenizer, index, return_tensors=None):
    return Ids(prompt)
''',
    "lmms_eval/__init__.py": "", "lmms_eval/models/__init__.py": "", "lmms_eval/models/simple/__init__.py": "",
    "lmms_eval/models/simple/cambrian_p.py": '''
import json, os, torch
def expand2square(img, bg):
    return img
class _T:
    shape = (8, 3, 384, 384)
    def half(self):
        return self
    def cuda(self):
        return self
class _Proc:
    image_mean = [0.5, 0.5, 0.5]
    def preprocess(self, frames, return_tensors=None):
        assert len(frames) > 0
        return {"pixel_values": _T()}
class _Cam:
    shape = (2, 3584)
    def detach(self):
        return self
    def float(self):
        return self
    def abs(self):
        return self
    def sum(self):
        return 5.0
class _Inner:
    rec_head = type("H", (), {"camera_tokens": _Cam()})()
class _Model:
    config = type("C", (), {"vocab_size": 151672})()
    def get_model(self):
        return _Inner()
class _Tok:
    pad_token_id = 0
    eos_token_id = 1
class CambrianP:
    calls = {}
    def __init__(self, **kw):
        self.tokenizer, self.model, self._image_processor, self.device = _Tok(), _Model(), _Proc(), "cuda"
    def _run_inference_step(self, ids, mask, images, image_size, task, gen, video):
        prompt = ids.prompt
        log = os.environ["STUB_LOG"]
        n = sum(1 for l in open(log)) if os.path.exists(log) else 0
        with open(log, "a") as f:
            f.write(json.dumps({"prompt": prompt, "gen": gen, "task": task, "image_size": image_size}) + "\\n")
        q = prompt.split("<image>\\n", 1)[1].split("<|im_end|>")[0]
        if os.environ.get("STUB_OOM") and os.environ["STUB_OOM"] in q:
            raise torch.cuda.OutOfMemoryError("stand-in OOM")
        flaky = os.environ.get("STUB_FAIL_ONCE")
        if flaky and flaky in q and not os.path.exists(log + ".failed"):
            open(log + ".failed", "w").close()
            raise RuntimeError("stand-in transient failure")
        return ["B" if "Options:" in q else "3"]
''',
}


def make_stubs(root):
    for rel, text in STUBS.items():
        p = os.path.join(root, rel)
        os.makedirs(os.path.dirname(p), exist_ok=True)
        open(p, "w").write(text)


def qa_rows(video_key):
    mc = {"qa_id": "vsibench-1", "native_id": 1, "video_key": video_key, "question_type": "route_planning",
          "task_family": "multiple_choice", "options": ["A. left", "B. right"], "reference_answer": "B",
          "prompt": "These are frames of a video.\nWhich way? MARK-MC\nOptions:\nA. left\nB. right\n"
                    "Answer with the option's letter from the given choices directly."}
    na = {"qa_id": "vsibench-2", "native_id": 2, "video_key": video_key, "question_type": "object_counting",
          "task_family": "numeric", "options": None, "reference_answer": "3",
          "prompt": "These are frames of a video.\nHow many chairs? MARK-NA\n"
                    "Please answer the question using a single word or phrase."}
    return [mc, na]


def run(tmp, name, env_extra, extra_args=()):
    d = os.path.join(tmp, name)
    os.makedirs(d)
    vk = "scannet/scene_test"
    batch = {"videos": [{"video_key": vk, "safe_key": L.safe_key(vk), "local_path": args.video, "qa": qa_rows(vk)}]}
    L.write_json_atomic(os.path.join(d, "batch.json"), batch)
    gen = {"max_new_tokens": 16, "temperature": 0, "top_p": 1.0, "num_beams": 1, "do_sample": False}
    identity = {"GENERATION": gen, "NUM_FRAMES": 8}
    env = dict(os.environ, PYTHONPATH=os.path.join(tmp, "stubs") + os.pathsep + os.environ.get("PYTHONPATH", ""),
               STUB_LOG=os.path.join(d, "model_calls.jsonl"), MPLBACKEND="Agg", **env_extra)
    cmd = [sys.executable, args.worker, "--batch", os.path.join(d, "batch.json"), "--model_path", "stub",
           "--num_frames", "8", "--config_hash", "H", "--identity", json.dumps(identity),
           "--journal", os.path.join(d, "journal.jsonl"), "--errors", os.path.join(d, "errors.jsonl"),
           "--shard_dir", os.path.join(d, "shards"), "--status", os.path.join(d, "status.json"),
           "--lib_dir", HERE, "--max_retries", "2", *extra_args]
    r = subprocess.run(cmd, env=env, capture_output=True, text=True, cwd=d)
    calls = [json.loads(l) for l in open(env["STUB_LOG"])] if os.path.exists(env["STUB_LOG"]) else []
    status = json.load(open(os.path.join(d, "status.json"))) if os.path.exists(os.path.join(d, "status.json")) else {}
    ok, failed, meta, _ = P.gather_results(os.path.join(d, "shards"), os.path.join(d, "journal.jsonl"), "H")
    return r, calls, status, ok, failed, meta, d, gen


tmp = tempfile.mkdtemp(prefix="vsibench_worker_test_")
try:
    make_stubs(os.path.join(tmp, "stubs"))
    # 1. normal run
    r, calls, status, ok, failed, meta, d, gen = run(tmp, "normal", {})
    check("worker runs to completion (exit 0, phase done, camera head check passed)",
          r.returncode == 0 and status.get("phase") == "done" and status.get("camera_head_loaded"),
          r.stdout + r.stderr)
    shard_rows = [x for f in sorted(os.listdir(os.path.join(d, "shards"))) if f.endswith(".jsonl")
                  for x in L.read_jsonl(os.path.join(d, "shards", f))[0]] if os.path.isdir(os.path.join(d, "shards")) else []
    answers = [x for x in shard_rows if x.get("kind") == "answer"]
    need = ("qa_id", "native_id", "video_key", "status", "raw_answer", "reference_answer", "attempt", "config_hash",
            "answered_at")
    check("every question is saved as an ok answer with the fields scoring and review read",
          set(ok) == {"vsibench-1", "vsibench-2"} and not failed and len(answers) == 2
          and all(all(k in a for k in need) and a["status"] == "ok" and isinstance(a["raw_answer"], str) for a in answers),
          answers)
    check("answers are the model's output, kept verbatim", ok.get("vsibench-1", {}).get("raw_answer") == "B"
          and ok.get("vsibench-2", {}).get("raw_answer") == "3")
    check("each question is answered exactly once (no retries on success)", len(calls) == 2, len(calls))
    check("the model gets '<image>\\n' + the official prompt in the qwen_1_5 template, task 'video'",
          len(calls) == 2 and all(c["task"] == "video" for c in calls)
          and calls[0]["prompt"].startswith("<|im_start|>user\n<image>\nThese are frames of a video.\nWhich way? MARK-MC")
          and calls[0]["prompt"].endswith("<|im_start|>assistant\n"), calls[:1])
    check("generation settings are exactly the run identity's", all(c["gen"] == gen for c in calls), calls[:1])
    m = meta.get("scannet/scene_test") or {}
    from decord import VideoReader, cpu
    import numpy as np
    vr = VideoReader(args.video, ctx=cpu(0), num_threads=0)
    idx = L.sample_indices(len(vr), 8)
    want = [hashlib.sha1(np.ascontiguousarray(f).tobytes()).hexdigest() for f in vr.get_batch(idx).asnumpy()]
    check("video_meta records indices, timestamps, reader and a SHA-1 per decoded input frame",
          m.get("frame_indices") == idx and len(m.get("timestamps_s", [])) == 8 and m.get("reader") == "decord"
          and m.get("frame_sha1") == want and m.get("preprocessing"), {k: m.get(k) for k in ("frame_indices", "reader")})

    # 2. a transient model failure is retried and then answered
    r, calls, status, ok, failed, meta, d, _ = run(tmp, "retry", {"STUB_FAIL_ONCE": "MARK-NA"})
    check("a transient model failure is retried; the answer is saved with attempt 2",
          r.returncode == 0 and ok.get("vsibench-2", {}).get("attempt") == 2 and len(calls) == 3, r.stderr[-500:])

    # 3. out of GPU memory stops the worker, keeps earlier answers
    r, calls, status, ok, failed, meta, d, _ = run(tmp, "oom", {"STUB_OOM": "MARK-NA"})
    check("out of GPU memory stops the worker (nonzero exit) and keeps the earlier answer",
          r.returncode != 0 and status.get("phase") == "stopped: out of GPU memory" and "vsibench-1" in ok
          and "vsibench-2" not in ok, (r.returncode, status))

    # 4. deliberate stop (resume test) exits 75 after one answer
    r, calls, status, ok, failed, meta, d, _ = run(tmp, "stop", {}, ["--stop_after", "1"])
    check("--stop_after 1 exits 75 with exactly one answer kept in the journal", r.returncode == 75 and len(ok) == 1,
          (r.returncode, list(ok)))

    # 5. a bookkeeping bug must stop the worker, not be recorded as model errors with retries
    shutil.copy(args.worker, os.path.join(tmp, "worker_broken.py"))
    src = open(os.path.join(tmp, "worker_broken.py")).read()
    marker = '"native_id": q["native_id"]'
    if marker in src:
        open(os.path.join(tmp, "worker_broken.py"), "w").write(src.replace(marker, '"native_id": q["no_such_field"]'))
        real_worker, args.worker = args.worker, os.path.join(tmp, "worker_broken.py")
        r, calls, status, ok, failed, meta, d, _ = run(tmp, "bookkeeping_bug", {})
        args.worker = real_worker
        check("a bug in saving an answer stops the worker with its traceback (no retries, no fake model errors)",
              r.returncode != 0 and "KeyError" in r.stderr and len(calls) == 1 and not failed, (r.returncode, len(calls)))
    else:
        check("the worker saves native_id (needed by the bookkeeping-bug test)", False, "marker not found")
finally:
    shutil.rmtree(tmp, ignore_errors=True)

bad = [n for n, ok_ in RESULTS if not ok_]
print(f"\n{len(RESULTS) - len(bad)}/{len(RESULTS)} worker checks passed")
sys.exit(1 if bad else 0)

`review_cells.py`

In [ ]:
%%writefile /content/vsibench/review_cells.py
"""Notebook cells for sections 8 (candidates, export, manual review) and 9 (verified failures, report, package).

make_nb.py inserts CELLS after the scoring section. The cells read only: L, P, OUTPUT_ROOT, RUN_ID, RUN_MODE,
CONFIG_HASH, IDENTITY, SEL, QA_BY_ID, MANIFEST, MANIFEST_BY_VIDEO, SCORES, SCORES_BY_ID, OK, META, COVERAGE,
TARGET_FAILURE_VIDEOS, MIN_FAILURE_VIDEOS, VENV_PY, WORKER_ENV, fetch_video; everything else they define.
"""

CELLS = []
CELLS.append(("md", """
## 8. Failure candidates and manual frame-evidence review

**This is a manual stage.** The official score only nominates *candidates*: a multiple-choice miss or a zero-credit number is an automatic failure candidate, a malformed reply is a format-error candidate, and partial numeric credit (`imperfect`) is listed in `failure_candidates.csv` but is never a verified failure by score alone. Exported frames do not show that the answer is visible; a reviewer decides that.

A video counts as a **verified failure** only when `vsibench_review.validate_verified_failure` accepts one of its questions: the run's own prediction, a blind judgment saved before the answers were shown, a final review for that exact question and run with verdict *wrong*, valid evidence citations (or a documented search scope), and a playable, decode-checked video with its exact input frames. The progress line, `summary.csv`, the report and the final manifest all use that one function.

Review order: one primary question per video (up to two supporting), 20 videos at a time; when a reviewer marks a candidate correct or uncertain, the next reserve candidate from the same sample is offered. Nothing is ever deleted: every save is kept in the review's history.
"""))

CELLS.append(("code", r'''
# @title 8a. Candidates and the review queue (automatic; nothing here is a verified failure)
HELPER_DIR = "/content/vsibench"
import importlib, sys
sys.path.insert(0, HELPER_DIR)
import vsibench_review as R
import vsibench_diagnostics as D
importlib.reload(R); importlib.reload(D)
from collections import Counter
_diag = D.diagnose_from_globals(globals(), D.NEEDS_8A)
if _diag["missing_prerequisites"]:
    print(D.render_candidate_diagnostic(_diag, "Cannot build the review queue yet"))
else:
    SELECTED_VIDEO_KEYS = sorted({m["video_key"] for m in MANIFEST})
    R.write_failure_candidates(os.path.join(OUTPUT_ROOT, "failure_candidates.csv"), SCORES, QA_BY_ID)
    QUEUE_PATH = os.path.join(OUTPUT_ROOT, "review_queue.json")
    QUEUE = R.update_queue(QUEUE_PATH, SCORES, QA_BY_ID, SELECTED_VIDEO_KEYS, TARGET_FAILURE_VIDEOS)
    D.record_queue(OUTPUT_ROOT, D.scores_fingerprint(SCORES, list(QA_BY_ID)), len(QUEUE["entries"]))
    ACTIVE, RESERVE = R.queue_split(QUEUE)
    EXPORT_CTX = {"output_root": OUTPUT_ROOT, "venv_py": VENV_PY, "env": WORKER_ENV, "lib_dir": HELPER_DIR,
                  "fetch_video": fetch_video, "meta": META, "qa_by_id": QA_BY_ID, "scores_by_id": SCORES_BY_ID,
                  "config_hash": CONFIG_HASH}

    def review_ctx():
        return {"output_root": OUTPUT_ROOT, "config_hash": CONFIG_HASH, "selected_video_keys": SELECTED_VIDEO_KEYS,
                "qa_by_id": QA_BY_ID, "ok": OK, "scores_by_id": SCORES_BY_ID, "media": MANIFEST_BY_VIDEO,
                "reviews": R.load_reviews(OUTPUT_ROOT), "run_id": RUN_ID}

    print("Automatic status of every scored question:", dict(Counter(s["automatic_status"] for s in SCORES)))
    print(f"Inspection candidates (official score below its maximum): "
          f"{sum(s['automatic_status'] in R.INSPECTION_STATUSES for s in SCORES)} -> failure_candidates.csv")
    print(f"Candidate videos (a wrong or format-error answer): {len(QUEUE['entries'])}; review queue: {len(ACTIVE)} active, "
          f"{len(RESERVE)} in reserve (saved in review_queue.json).")
    for e in ACTIVE:
        print(f"  {e['dataset']:12s} {e['primary_question_type']:28s} {e['automatic_status']:16s} {e['video_key']}")
    print()
    print(D.render_candidate_diagnostic(D.diagnose_from_globals(globals(), D.NEEDS_8A)))
'''))

CELLS.append(("code", r'''
# @title 8b. Export the active candidates: playable video on Drive, decode check, exact input frames, contact sheets
if not all(k in globals() for k in ("R", "D", "ACTIVE", "EXPORT_CTX")):
    print("Cannot export yet: run 8a first (it builds the review queue).")
elif not ACTIVE:
    print(D.render_candidate_diagnostic(D.diagnose_from_globals(globals()), "Nothing to export: the review queue is empty"))
else:
    for e in ACTIVE:
        problems = R.export_candidate(e, EXPORT_CTX)
        man = R.load_frames_manifest(R.case_dir(OUTPUT_ROOT, e)) or {}
        print(f"  {e['safe_key']}: {sum(x['frame_set'] == 'input' for x in man.get('entries', []))} input frames, "
              f"inputs verified: {man.get('inputs_verified')}" + (f"; PROBLEMS: {problems}" if problems else ""))
    print(f"Exported {len(ACTIVE)} candidate video(s) to candidate_cases/ (history; not the final deliverable).")
'''))

CELLS.append(("code", r'''
# @title 8c. Review (manual): blind judgment first, then answers, extra frames and the final verdict
import sys as _sys
_sys.path.insert(0, "/content/vsibench")
try:
    import vsibench_diagnostics as D
    _DIAG = D.diagnose_from_globals(globals())
except ImportError:
    D, _DIAG = None, None
_READY = _DIAG is not None and not _DIAG["missing_prerequisites"]
import ipywidgets as W
from IPython.display import display, clear_output
UI = {}
progress, pick, body = W.HTML(), W.Dropdown(layout=W.Layout(width="90%")), W.Output()
_busy = {"pick": False}
_LAST = {}   # latest validator results and counts (survive re-rendering a case)
RED = "<span style='color:#b00'>{}</span>"

def _blank(label):
    return [(f"— {label} —", "")]

def current_state():
    """Validator results for every candidate, reserve promotion after overrides, and the shared counts."""
    global ACTIVE, RESERVE
    ctx = review_ctx()
    per_video = R.evaluate(QUEUE["entries"], ctx)
    offered = R.promote_reserve(QUEUE_PATH, QUEUE, {k: v["state"] for k, v in per_video.items()}, TARGET_FAILURE_VIDEOS)
    for e in offered:
        R.export_candidate(e, EXPORT_CTX)
    ACTIVE, RESERVE = R.queue_split(QUEUE)
    if offered:
        per_video = R.evaluate(QUEUE["entries"], review_ctx())
    return per_video, R.counts(QUEUE["entries"], ACTIVE, per_video, OUTPUT_ROOT, TARGET_FAILURE_VIDEOS,
                               MIN_FAILURE_VIDEOS), offered

def refresh():
    per_video, C, offered = current_state()
    _LAST.update(counts=C, per_video=per_video, offered=offered)
    progress.value = ("<b>Verified failure videos: {verified_wrong_video_count}</b> (minimum {minimum}, target {target}) · "
                      "final reviews {final_review_video_count} · overridden correct {overridden_correct_video_count} · "
                      "uncertain {uncertain_video_count} · invalid/missing evidence {invalid_or_missing_evidence_video_count}"
                      " · exported {exported_video_count} · candidates {candidate_video_count}").format(**C)
    _busy["pick"] = True
    keep = pick.value
    pick.options = [(f"{i + 1}. [{per_video[e['video_key']]['state']}] {e['video_key']}", i) for i, e in enumerate(ACTIVE)]
    pick.value = keep if keep in range(len(ACTIVE)) else (0 if ACTIVE else None)
    _busy["pick"] = False
    return offered

def show(i, qa_id=None, message=""):
    e = ACTIVE[i]
    qid = qa_id or e["primary_qa_id"]
    q, s = QA_BY_ID[qid], SCORES_BY_ID[qid]
    r = R.load_review(OUTPUT_ROOT, qid)
    r = r if r and r.get("config_hash") == CONFIG_HASH else None
    cdir = R.case_dir(OUTPUT_ROOT, e)
    man = R.load_frames_manifest(cdir) or {"entries": []}
    inputs = [x for x in man["entries"] if x["frame_set"] == "input"]
    UI.clear()
    UI.update(case=e, qa_id=qid, revealed=lambda: "answers" in UI, counts=_LAST.get("counts"),
              offered=_LAST.get("offered"))
    opts = "".join(f"<li>{L.esc(o)}</li>" for o in q.get("options") or [])
    head = W.HTML(f"<h3>{L.esc(e['video_key'])}</h3><p>{L.esc(e['dataset'])} · {L.esc(q['question_type'])} · "
                  f"<code>{L.esc(qid)}</code> (VSI-Bench id {L.esc(q.get('native_id'))})</p>"
                  f"<p><b>Question:</b> {L.esc(q['question'])}</p><ul>{opts}</ul>"
                  f"<p>Video: <code>{L.esc(os.path.join(cdir, 'video.mp4'))}</code></p>")
    qa_pick = W.Dropdown(options=[(f"primary: {e['primary_qa_id']}", e["primary_qa_id"])] +
                         [(f"supporting: {x}", x) for x in e["supporting_qa_ids"]], value=qid, description="Question")
    qa_pick.observe(lambda ch: show(i, ch["new"]) if ch["new"] != qid else None, names="value")
    prov = W.HTML(f"<p><b>Model input:</b> {len(inputs)} frames (indices {inputs[0]['frame_index'] if inputs else '-'}"
                  f"…{inputs[-1]['frame_index'] if inputs else '-'} of {man.get('total_frames')}, {man.get('fps')} fps), "
                  f"reader {L.esc(man.get('reader'))}; preprocessing: {L.esc(man.get('preprocessing'))}<br>"
                  f"<b>Inputs verified:</b> {man.get('inputs_verified')} ({L.esc(man.get('verification'))})</p>")
    sheets = [W.Image(value=open(os.path.join(cdir, f), "rb").read(), format="jpg", layout=W.Layout(width="100%"))
              for f in man.get("input_sheets", [])]
    full_pos = W.BoundedIntText(value=0, min=0, max=max(len(inputs) - 1, 0), description="Input #")
    full_show, full_img = W.Button(description="Show full-resolution frame"), W.Image(format="jpg")
    def _full(_):
        x = inputs[full_pos.value]
        full_img.value = open(os.path.join(cdir, x["file"]), "rb").read()
        full_img.layout.width = "100%"
    full_show.on_click(_full)
    UI.update(qa_pick=qa_pick, full_pos=full_pos, full_show=full_show, full_img=full_img, input_sheets=sheets)
    parts = [head, qa_pick, prov] + sheets + [W.HBox([full_pos, full_show]), full_img]

    if not (r and r.get("provisional")):
        pa = W.Text(description="Your answer", placeholder="letter, number, or 'cannot determine'")
        sp, pm = W.Button(description="Save my answer and reveal", button_style="info"), W.HTML(message)
        def _prov(_):
            _, errs = R.save_provisional(OUTPUT_ROOT, q, CONFIG_HASH, RUN_ID, pa.value, "", "")
            if errs:
                pm.value = RED.format(L.esc("; ".join(errs)))
                return
            refresh()
            show(i, qid, "Blind judgment saved.")
        sp.on_click(_prov)
        UI.update(provisional_answer=pa, save_provisional=sp, prov_msg=pm)
        parts += [W.HTML("<b>Step 1. Your own answer</b>, from the frames above (the model's and the benchmark's answers "
                         "stay hidden until you save it)"), pa, sp, pm]
    else:
        p0 = r["provisional"]
        answers = W.HTML(
            f"<p><i>Blind judgment ({L.esc(p0['saved_at'])}): {L.esc(p0['answer'])} / {L.esc(p0.get('evidence_status') or '-')}"
            f"</i></p><p><b>Model:</b> {L.esc(s.get('raw_answer'))} → officially parsed {L.esc(s.get('parsed_answer'))} · "
            f"<b>Benchmark answer:</b> {L.esc(s.get('reference_answer'))} · <b>{L.esc(s.get('official_metric'))}:</b> "
            f"{L.esc(s.get('official_score'))} · automatic status {L.esc(s.get('automatic_status'))}</p>"
            f"<p class='k'>{L.esc(s.get('candidate_reason'))}</p>")
        fps_w = W.Text(value="1", description="Extra fps")
        win_w = W.Text(description="Windows", placeholder="start,end[,fps]; start,end[,fps] (seconds)")
        xb, xm = W.Button(description="Extract extra frames"), W.HTML()
        xbox = W.VBox()
        def _extra_images():
            m2 = R.load_frames_manifest(cdir) or {}
            xbox.children = [W.Image(value=open(os.path.join(cdir, f), "rb").read(), format="jpg",
                                     layout=W.Layout(width="100%")) for f in m2.get("extra_sheets", [])]
        def _extra(_):
            wins, errs = R.parse_windows(win_w.value)
            try:
                f = float(fps_w.value or 0)
            except ValueError:
                errs.append("fps must be a number")
                f = 0
            if errs:
                xm.value = RED.format(L.esc("; ".join(errs)))
                return
            code_, out = R.run_extra_frames(e, EXPORT_CTX, f, wins)
            _extra_images()
            xm.value = (RED.format(L.esc(out[-300:])) if code_ else
                        L.esc(out[-200:]) + " — extra frames are kept apart from the model inputs (frames_extra/)")
        xb.on_click(_extra)
        _extra_images()
        cur = R.current_final(r) or {}
        wide = W.Layout(width="95%")
        verdict = W.Dropdown(options=_blank("choose") + list(R.VERDICT_CHOICES), value=cur.get("verdict", ""),
                             description="Verdict", layout=wide, style={"description_width": "150px"})
        evidence = W.Dropdown(options=_blank("choose") + list(R.EVIDENCE_CHOICES), value=cur.get("evidence_status", ""),
                              description="Evidence visible?", layout=wide, style={"description_width": "150px"})
        frames_w = W.Text(value=R.frames_text(cur.get("citations"), man), description="Which frames?",
                          placeholder="#12, #40   (numbers printed on the sheets; f1234 for an extra frame)",
                          layout=wide, style={"description_width": "150px"})
        confidence = W.Dropdown(options=_blank("choose") + [(x, x) for x in R.CONFIDENCE], value=cur.get("confidence", ""),
                                description="Confidence", style={"description_width": "150px"})
        notes = W.Textarea(value=cur.get("rationale", ""), description="Notes",
                           placeholder="one or two sentences: what you saw and why you chose this verdict",
                           layout=W.Layout(width="95%", height="60px"), style={"description_width": "150px"})
        sf, fm, st = W.Button(description="Save review", button_style="success"), W.HTML(message), W.HTML()
        res = next((x for x in UI_STATE_RESULTS(e["video_key"]) if x["qa_id"] == qid), None)
        st.value = ("<b>Validator:</b> " + ("verified failure" if res and res["eligible"] else
                    RED.format(L.esc("; ".join((res or {}).get("reasons", [])[:6])))))
        def _final(_):
            man_now = R.load_frames_manifest(cdir)
            final, errs = R.quick_final(verdict.value, evidence.value, confidence.value, frames_w.value, notes.value,
                                        man_now, s.get("automatic_status"))
            if not errs:
                _, errs = R.save_final(OUTPUT_ROOT, q, CONFIG_HASH, final, man_now, cdir, s.get("automatic_status"))
            if errs:
                fm.value = RED.format("Not saved: " + L.esc("; ".join(errs)))
                return
            offered = refresh()
            msg = f"Final review saved. Verified failure videos now: {_LAST['counts']['verified_wrong_video_count']}."
            if offered:
                msg += " Next reserve candidate offered: " + ", ".join(x["video_key"] for x in offered) + "."
            show(i, qid, msg)
        sf.on_click(_final)
        UI.update(answers=answers, fps=fps_w, windows=win_w, extra=xb, extra_msg=xm, extra_images=xbox, verdict=verdict,
                  evidence=evidence, frames=frames_w, confidence=confidence, notes=notes, save_final=sf, final_msg=fm,
                  status=st)
        parts += [answers, W.HTML("<b>Step 2 (optional). More frames</b> from the whole video, kept apart from the "
                                  "model's frames"), W.HBox([fps_w, win_w, xb]), xm, xbox,
                  W.HTML("<b>Step 3. Your review</b>. <i>Evidence visible?</i> asks whether the frames show what is "
                         "needed to answer the question. For <i>Which frames?</i> type the #numbers printed on the "
                         "input sheets above (e.g. #12, #40), or f-numbers from the extra sheets (e.g. f1234); not "
                         "needed if you couldn't find it or can't tell."),
                  verdict, evidence, frames_w, confidence, notes, sf, fm, st]
    with body:
        clear_output()
        display(W.VBox(parts))

def UI_STATE_RESULTS(video_key):
    pv = (_LAST.get("per_video") or {}).get(video_key)
    return pv["results"] if pv else []

def _on_pick(ch):
    if not _busy["pick"] and ch["new"] is not None:
        show(ch["new"])
pick.observe(_on_pick, names="value")
if D is None:
    print("The helper code is not installed in this runtime: run 1a and the 1b cells, then 1c onward.")
elif not _READY:
    print(D.render_candidate_diagnostic(_DIAG, "The review cannot start yet"))
else:
    refresh()
    _DIAG = D.diagnose_from_globals(globals())            # after refresh (it may promote reserve candidates)
    if ACTIVE:
        if any(f["code"] != "ready" for f in _DIAG["findings"]):
            print(D.render_candidate_diagnostic(_DIAG, "Review status"))
        display(W.VBox([progress, pick, body]))
        show(0)
    else:
        print(D.render_candidate_diagnostic(_DIAG, "Review queue is empty"))
'''))

CELLS.append(("md", """
## 9. Verified failures and final outputs
`failure_videos/` and `cases/` contain exactly the cases in `verified_failure_manifest.json` (at most the target number, in queue order). Candidate history stays in `candidate_cases/` and `reviews/`. Inference completion and case-export completion are reported separately: verified cases do not mean the whole sample was run.
"""))

CELLS.append(("code", r'''
# @title 9a. Verified-failure manifest, summary, report (all counts from the one validator)
ctx = review_ctx()
PER_VIDEO = R.evaluate(QUEUE["entries"], ctx)
ACTIVE, RESERVE = R.queue_split(QUEUE)
COUNTS = R.counts(QUEUE["entries"], ACTIVE, PER_VIDEO, OUTPUT_ROOT, TARGET_FAILURE_VIDEOS, MIN_FAILURE_VIDEOS)
GATE = R.completion_gate(COUNTS, COVERAGE)
FINAL = R.build_final(OUTPUT_ROOT, PER_VIDEO, ACTIVE, ctx, COUNTS, GATE, TARGET_FAILURE_VIDEOS)
R.write_summary(os.path.join(OUTPUT_ROOT, "summary.csv"), PER_VIDEO, ACTIVE, ctx)
info = {"dataset": f"{IDENTITY.get('DATASET_REPO')} config {IDENTITY.get('DATASET_CONFIG')} split "
                   f"{IDENTITY.get('DATASET_SPLIT')} @ {IDENTITY.get('DATASET_REVISION')}",
        "model": f"{IDENTITY.get('MODEL_REPO')} @ {IDENTITY.get('MODEL_REVISION')}, {IDENTITY.get('NUM_FRAMES')} frames",
        "run mode": RUN_MODE, "config hash": CONFIG_HASH}
info.update({f"selection: {k}": v for k, v in SEL.items() if isinstance(v, (int, float, str)) and len(str(v)) < 80})
info.update({f"coverage: {k}": v for k, v in COVERAGE.items() if isinstance(v, (int, float, str, bool, dict))
             and len(json.dumps(v, default=str)) < 300})
R.report_html(OUTPUT_ROOT, FINAL, PER_VIDEO, info)
cov_path = os.path.join(OUTPUT_ROOT, "coverage.json")
cov = json.load(open(cov_path)) if os.path.exists(cov_path) else {}
cov.update(verification=COUNTS, gate=GATE)
L.write_json_atomic(cov_path, cov)
assert FINAL["verified_video_count_in_manifest"] == min(COUNTS["verified_wrong_video_count"], TARGET_FAILURE_VIDEOS)
print(GATE["message"])
for k, v in COUNTS.items():
    print(f"  {k}: {v}")
print(f"Inference complete: {GATE['inference_complete']} · case export complete (minimum met): {GATE['case_export_complete']}")
print(f"Wrote verified_failure_manifest.json ({len(FINAL['cases'])} case(s)), summary.csv, report.html in {OUTPUT_ROOT}")
'''))

CELLS.append(("code", r'''
# @title 9b. (Optional) Package the final deliverable (only the manifest's verified cases)
MAKE_ZIP = False  # @param {type:"boolean"}
if MAKE_ZIP:
    zpath = os.path.join(os.path.dirname(OUTPUT_ROOT), RUN_ID + "_verified_failures.zip")
    names = R.package(OUTPUT_ROOT, zpath)
    print(f"Saved {zpath}: {len(names)} files, {os.path.getsize(zpath) / 1e9:.2f} GB")
else:
    print("Set MAKE_ZIP = True to write a zip of report.html, summary.csv, the manifest and its verified cases only.")
'''))

In [ ]:
# @title 1c. Pin the dataset, model and code revisions
_helpers = ['vsibench_lib.py', 'vsibench_pipeline.py', 'vsibench_worker.py', 'vsibench_frames.py', 'vsibench_review.py', 'vsibench_diagnostics.py', 'vsibench_selftest.py', 'vsibench_review_selftest.py', 'vsibench_diagnostics_selftest.py', 'vsibench_worker_selftest.py', 'review_cells.py']
_absent = [n for n in _helpers if not os.path.exists(os.path.join("/content/vsibench", n))]
if _absent:
    raise SystemExit(f"The helper files are not in this runtime ({len(_absent)} of {len(_helpers)} missing, e.g. "
                     f"{_absent[0]}). They are written by the {len(_helpers)} '1b' cells directly below 1a (each starts "
                     "with %%writefile). Run those cells, or use Runtime > Run all / Run after from 1a, then rerun 1c. "
                     "(A restarted or new runtime starts with an empty /content, so they must be run again.)")
sys.path.insert(0, "/content/vsibench")
import importlib, vsibench_lib as L, vsibench_pipeline as P
importlib.reload(L); importlib.reload(P)
assert PROMPT_POLICY == L.PROMPT_POLICY and SAMPLING_UNIT == L.SAMPLING_UNIT, "settings differ from the helper code"
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
pins_path = os.path.join(OUTPUT_ROOT, "pins.json")
pins = json.load(open(pins_path)) if os.path.exists(pins_path) else {}
access_problems = []

def resolve(name, given, fetch):
    """Explicit value > value saved for this run > fetched now. A value differing from the saved one is an error."""
    value = given or pins.get(name)
    if not value:
        try:
            value = fetch()
        except Exception as exc:
            access_problems.append(f"{name}: {type(exc).__name__}: {exc}")
            return ""
    if pins.get(name) and pins[name] != value:
        raise RuntimeError(f"{name} {value} differs from {pins[name]} saved for RUN_ID {RUN_ID}. "
                           "Use the saved value, or set a new RUN_ID for a new run.")
    pins[name] = value
    return value

DATASET_REVISION = resolve("DATASET_REVISION", DATASET_REVISION, lambda: api.dataset_info(DATASET_REPO).sha)
MODEL_REVISION = resolve("MODEL_REVISION", MODEL_REVISION, lambda: api.model_info(MODEL_REPO).sha)
CODE_COMMIT = resolve("CODE_COMMIT", CODE_COMMIT, lambda: subprocess.run(
    ["git", "ls-remote", REPO_URL, "HEAD"], capture_output=True, text=True, check=True).stdout.split()[0])
if access_problems:
    print("Could not resolve:")
    for p in access_problems:
        print("  " + p)
    raise SystemExit("Revisions not pinned; nothing was changed. Check the repositories in a browser and rerun.")
L.write_json_atomic(pins_path, pins)
for k in ("DATASET_REVISION", "MODEL_REVISION", "CODE_COMMIT"):
    print(f"{k:17s} {pins[k]}")

## 2. Environment, model and the official benchmark code
Same Python 3.11 venv and install steps as the source notebook, with the repository checked out at `CODE_COMMIT`. The environment and weights are restored from the Drive cache when an exact match exists (same commit and install steps, same model revision; checked after restoring), otherwise installed/downloaded and then cached.

Cell 2b loads VSI-Bench's **official prompt formatter and scorer** (`lmms-eval/lmms_eval/tasks/vsibench/utils.py`) from that checkout, unmodified, records its SHA-256, and fixes the run identity.

In [ ]:
# @title 2a. Set up the environment and the model (restored from the Drive cache when possible)
import hashlib
REPO = "/content/cambrian-p"
VENV_PY = "/content/venv/bin/python"
MODEL_DIR = f"{REPO}/ckpts/Cambrian-P-7B"
CHECKOUT = r"""
set -e
cd /content
if [ ! -d cambrian-p/.git ]; then git clone -q "$REPO_URL" cambrian-p; fi
cd cambrian-p
if [ "$(git rev-parse HEAD)" != "$CODE_COMMIT" ]; then
    git fetch -q origin "$CODE_COMMIT" 2>/dev/null || git fetch -q origin
    git checkout -q "$CODE_COMMIT"
fi
echo "cambrian-p at $(git rev-parse HEAD)"
"""
INSTALL = r"""
set -e
cd /content/cambrian-p
echo "Installing (about 10-15 minutes) ..."
pip install -q uv
uv venv --seed --python 3.11 /content/venv
source /content/venv/bin/activate
pip install -q -e ./vggt
pip install -q hydra-core tensorboard iopath wcmatch fvcore
pip install -q --upgrade pip
pip install -q -e ".[train]"
pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --index-url https://download.pytorch.org/whl/cu121
pip install -q accelerate==0.29.3 easydict matplotlib roma evo imageio OpenEXR
pip install -q loguru sqlitedict sacrebleu evaluate tenacity pytz
pip install -q pyarrow==15.0.2
"""
# The cache name pins the code commit and the exact install steps: change either and the old copy is never used.
ENV_TAG = f"env-{CODE_COMMIT[:12]}-{hashlib.sha256(INSTALL.encode()).hexdigest()[:12]}"
ENV_TAR = os.path.join(CACHE_ROOT, "envs", ENV_TAG + ".tar")
MODEL_CACHE = os.path.join(CACHE_ROOT, "models", f"Cambrian-P-7B@{MODEL_REVISION}")
env = dict(os.environ, REPO_URL=REPO_URL, CODE_COMMIT=CODE_COMMIT, MODEL_REPO=MODEL_REPO, MODEL_REVISION=MODEL_REVISION,
           MODEL_DIR=MODEL_DIR, UV_PYTHON_INSTALL_DIR="/content/python", HF_HUB_ENABLE_HF_TRANSFER="1")
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN

def sh(script, what):
    if subprocess.run(["bash", "-c", script], env=env).returncode:
        raise SystemExit(f"{what} failed (see the log above).")

def env_ok():
    """The same checks whether the environment was just installed or restored from the cache."""
    if not os.path.exists(VENV_PY) or not os.path.isdir(os.path.join(REPO, ".git")):
        return False                       # nothing installed yet in this runtime
    imports = subprocess.run([VENV_PY, "-c", "import torch, decord"], capture_output=True, text=True)
    head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    return imports.returncode == 0 and head == CODE_COMMIT

def drive_free():
    try:
        return shutil.disk_usage("/content/drive/MyDrive").free
    except OSError:
        return 0

def tree_sizes(root):
    out = {}
    for d, dirs, files in os.walk(root):
        dirs[:] = [x for x in dirs if x != ".cache"]
        for f in files:
            if not f.startswith(".revision-"):
                p = os.path.join(d, f)
                out[os.path.relpath(p, root)] = os.path.getsize(p)
    return out

# --- environment ---
t0 = L.time.time()
if env_ok():
    print("Environment already set up in this session.")
else:
    if CACHE_ENV and os.path.exists(ENV_TAR):
        print(f"Restoring the environment from the Drive cache ({os.path.getsize(ENV_TAR) / 1e9:.1f} GB, {ENV_TAG}) ...", flush=True)
        for d in ("venv", "python", "cambrian-p"):
            shutil.rmtree(os.path.join("/content", d), ignore_errors=True)
        r = subprocess.run(["tar", "-xf", ENV_TAR, "-C", "/content"])
        if r.returncode or not env_ok():
            print("The cached environment failed its checks; ignoring it and installing fresh.")
            for d in ("venv", "python", "cambrian-p"):
                shutil.rmtree(os.path.join("/content", d), ignore_errors=True)
    sh(CHECKOUT, "Checkout")
    if not env_ok():
        sh(INSTALL, "Install")
        if not env_ok():
            raise SystemExit("The environment failed its checks after installing (torch/decord import or commit).")
        if CACHE_ENV:
            print("Saving the environment to the Drive cache for later sessions ...", flush=True)
            os.makedirs(os.path.dirname(ENV_TAR), exist_ok=True)
            r = subprocess.run(["tar", "-cf", ENV_TAR + ".partial", "-C", "/content", "--exclude=cambrian-p/ckpts",
                                "venv", "python", "cambrian-p"])
            if r.returncode:
                print("  could not save it (Drive full?); the next session will reinstall.")
                if os.path.exists(ENV_TAR + ".partial"):
                    os.remove(ENV_TAR + ".partial")
            else:
                os.replace(ENV_TAR + ".partial", ENV_TAR)
                print(f"  saved {os.path.getsize(ENV_TAR) / 1e9:.1f} GB as {ENV_TAR}")
sh(CHECKOUT, "Checkout")
print(f"Environment ready in {(L.time.time() - t0) / 60:.1f} min.")

# --- model weights: loaded only when 6b has questions to answer (scoring and review never need them) ---
def ensure_model():
    t0 = L.time.time()
    local_marker = os.path.join(MODEL_DIR, f".revision-{MODEL_REVISION}")
    cache_marker = os.path.join(MODEL_CACHE, ".complete.json")
    if os.path.exists(local_marker):
        print("Model already on local disk.")
    else:
        if CACHE_MODEL and os.path.exists(cache_marker):
            print(f"Copying the model from the Drive cache ({MODEL_CACHE}) ...", flush=True)
            expected = json.load(open(cache_marker))["files"]
            shutil.rmtree(MODEL_DIR, ignore_errors=True)
            shutil.copytree(MODEL_CACHE, MODEL_DIR, ignore=shutil.ignore_patterns(".complete.json"))
            if tree_sizes(MODEL_DIR) != expected:
                print("  the cached copy is incomplete; downloading instead.")
                shutil.rmtree(MODEL_DIR, ignore_errors=True)
        if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
            sh("""cd /content/cambrian-p && /content/venv/bin/huggingface-cli download "$MODEL_REPO" --revision "$MODEL_REVISION" \\
                  --local-dir "$MODEL_DIR" """, "Model download")
            if CACHE_MODEL:
                files = tree_sizes(MODEL_DIR)
                need = sum(files.values())
                if drive_free() < need + 1e9:
                    print(f"  not cached: Drive has {drive_free() / 1e9:.0f} GB free, the model needs {need / 1e9:.1f} GB.")
                else:
                    print(f"Saving the model ({need / 1e9:.1f} GB) to the Drive cache for later sessions ...", flush=True)
                    shutil.rmtree(MODEL_CACHE + ".partial", ignore_errors=True)
                    shutil.copytree(MODEL_DIR, MODEL_CACHE + ".partial",
                                    ignore=shutil.ignore_patterns(".cache", ".revision-*"))
                    L.write_json_atomic(os.path.join(MODEL_CACHE + ".partial", ".complete.json"),
                                        {"model_repo": MODEL_REPO, "revision": MODEL_REVISION, "files": files})
                    shutil.rmtree(MODEL_CACHE, ignore_errors=True)
                    os.replace(MODEL_CACHE + ".partial", MODEL_CACHE)
        open(local_marker, "w").close()
    print(f"Model ready in {(L.time.time() - t0) / 60:.1f} min.")
print("Model weights: not loaded now; 6b loads them only if questions still need answers.")

WORKER_ENV = dict(os.environ, PYTHONPATH=f"{REPO}/lmms-eval:{REPO}", PATH=f"/content/venv/bin:{os.environ['PATH']}",
                  MPLBACKEND="Agg")

In [ ]:
# @title 2b. Load the official VSI-Bench formatter and scorer; fix and check the run identity
OFFICIAL = L.load_official(REPO)
assert OFFICIAL["dataset_path"] == DATASET_REPO, f"vsibench.yaml uses {OFFICIAL['dataset_path']}"
assert OFFICIAL["test_split"] == DATASET_SPLIT, f"vsibench.yaml uses split {OFFICIAL['test_split']}"
if OFFICIAL["generation_kwargs"] != GENERATION:
    raise SystemExit(f"GENERATION {GENERATION} differs from the pinned vsibench.yaml {OFFICIAL['generation_kwargs']}")
SCORER = {"file": L.OFFICIAL_SCORER_PATH, "code_commit": CODE_COMMIT, "sha256": OFFICIAL["sha256"],
          "task_yaml_sha256": OFFICIAL["yaml_sha256"], "prompt_kwargs": OFFICIAL["prompt_kwargs"]}
IDENTITY = {"DATASET_REPO": DATASET_REPO, "DATASET_CONFIG": DATASET_CONFIG, "DATASET_SPLIT": DATASET_SPLIT,
            "DATASET_REVISION": DATASET_REVISION, "SAMPLING_UNIT": SAMPLING_UNIT, "SUBSET_FRACTION": SUBSET_FRACTION,
            "SEED": SEED, "MODEL_REPO": MODEL_REPO, "MODEL_REVISION": MODEL_REVISION, "CODE_COMMIT": CODE_COMMIT,
            "NUM_FRAMES": NUM_FRAMES, "PROMPT_POLICY": PROMPT_POLICY, "GENERATION": GENERATION,
            "PREPROCESSING": PREPROCESSING, "SCORER": SCORER}
RUN_CONFIG_PATH = os.path.join(OUTPUT_ROOT, "run_config.json")
run_config = L.check_run_config(RUN_CONFIG_PATH, IDENTITY)
CONFIG_HASH = run_config["config_hash"]
SHARD_DIR = os.path.join(OUTPUT_ROOT, "predictions")
JOURNAL = os.path.join(LOCAL_ROOT, "journal", f"answers-{CONFIG_HASH}.jsonl")
ERRORS_JOURNAL = os.path.join(LOCAL_ROOT, "journal", f"errors-{CONFIG_HASH}.jsonl")
for d in ("predictions", "reviews", "candidate_cases", "cases", "failure_videos", "logs"):
    os.makedirs(os.path.join(OUTPUT_ROOT, d), exist_ok=True)

def versions():
    freeze = subprocess.run([VENV_PY, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.splitlines()
    torch_info = subprocess.run([VENV_PY, "-c", "import torch;print(torch.__version__, torch.version.cuda)"],
                                capture_output=True, text=True).stdout.split()
    return {"pip_freeze": freeze, "torch": torch_info[:1], "cuda": torch_info[1:2], "gpu": gpu,
            "code_commit_checked_out": subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                                                      text=True).stdout.strip()}
ENV_INFO = versions()
assert ENV_INFO["code_commit_checked_out"] == CODE_COMMIT, "checked-out commit differs from CODE_COMMIT"

def update_manifest(**extra):
    path = os.path.join(OUTPUT_ROOT, "run_manifest.json")
    man = json.load(open(path)) if os.path.exists(path) else {"created": L.time.strftime("%Y-%m-%dT%H:%M:%SZ")}
    man.update({"run_id": RUN_ID, "identity": IDENTITY, "config_hash": CONFIG_HASH, "environment": ENV_INFO,
                "workarounds": ["cambrianp.utils.cast_cambrian_config_to_llava_ov_style replaced by a function that keeps "
                                "the checkpoint config, sets load_rec_model=True and rec_model_path='None' (needed at the "
                                "pinned commit: builder.py re-casts any 'cambrian' model path; the checkpoint's "
                                "rec_model_path is null). The worker refuses to run unless the camera tokens loaded."],
                "lib": L.LIB_VERSION, "updated": L.time.strftime("%Y-%m-%dT%H:%M:%SZ")}, **extra)
    L.write_json_atomic(path, man)
update_manifest()
print(f"Official scorer: {L.OFFICIAL_SCORER_PATH} @ {CODE_COMMIT[:12]}, sha256 {OFFICIAL['sha256'][:16]}...")
print(f"Generation (from vsibench.yaml): {GENERATION}")
print(f"config hash {CONFIG_HASH}  (any change to data/sample/model/code/frames/prompt/generation/scorer needs a new RUN_ID)")

## 3. Discover the pinned VSI-Bench annotations

In [ ]:
# @title 3a. File list, dataset card, the `full` config's files; load, cross-check and validate every question
import urllib.request
from huggingface_hub import hf_hub_url
DATA_CACHE = os.path.join(CACHE_ROOT, "datasets", f"VSI-Bench@{DATASET_REVISION}")
os.makedirs(DATA_CACHE, exist_ok=True)
AUTH = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
tree_path = os.path.join(DATA_CACHE, "files.json")
if os.path.exists(tree_path):
    FILES = json.load(open(tree_path))
else:
    FILES = {}
    for f in api.list_repo_tree(DATASET_REPO, repo_type="dataset", revision=DATASET_REVISION, recursive=True):
        lfs = getattr(f, "lfs", None)
        FILES[f.path] = {"size": getattr(f, "size", None), "sha256": getattr(lfs, "sha256", None) if lfs else None}
    L.write_json_atomic(tree_path, FILES)
for p, info in sorted(FILES.items()):
    print(f"  {p:24s} {(info['size'] or 0) / 1e6:10.1f} MB")

def url_of(name):
    return hf_hub_url(DATASET_REPO, name, repo_type="dataset", revision=DATASET_REVISION)

def cached_file(name):
    """The Drive copy of a small dataset file at the pinned revision, downloaded once over HTTPS and checked against
    the repository's size (and SHA-256 for LFS files)."""
    path = os.path.join(DATA_CACHE, "files", name)
    want = FILES[name]
    if not (os.path.exists(path) and os.path.getsize(path) == want["size"]):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with urllib.request.urlopen(urllib.request.Request(url_of(name), headers=AUTH), timeout=300) as r, \
                open(path + ".part", "wb") as out:
            shutil.copyfileobj(r, out, 1 << 20)
        if os.path.getsize(path + ".part") != want["size"]:
            raise SystemExit(f"{name}: got {os.path.getsize(path + '.part')} bytes, expected {want['size']}")
        if want["sha256"] and L.sha256_file(path + ".part") != want["sha256"]:
            raise SystemExit(f"{name}: SHA-256 differs from the repository's LFS hash")
        os.replace(path + ".part", path)
    return path

card = open(cached_file("README.md")).read()
CONFIG_FILES = P.card_config_files(card, DATASET_CONFIG, DATASET_SPLIT, list(FILES))
print(f"\nConfig {DATASET_CONFIG!r}, split {DATASET_SPLIT!r}: pattern {CONFIG_FILES['patterns']} -> {CONFIG_FILES['files']}")
print("Other configs on the card:", {k: [d['path'] for d in v] for k, v in CONFIG_FILES["configs"].items() if k != DATASET_CONFIG})
if not CONFIG_FILES["files"]:
    raise SystemExit("The config matched no files; nothing selected.")
CONFIG_ROWS = P.read_parquet_rows([cached_file(f) for f in CONFIG_FILES["files"]])
JSONL_ROWS = [json.loads(l) for l in open(cached_file("test.jsonl"))] if "test.jsonl" in FILES else None
ANNOTATION_SHA = {f: L.sha256_file(cached_file(f)) for f in CONFIG_FILES["files"] + (["test.jsonl"] if JSONL_ROWS else [])}
problems = P.cross_check(CONFIG_ROWS, JSONL_ROWS) if JSONL_ROWS else []
if problems:
    raise SystemExit("The config's rows differ from test.jsonl: " + "; ".join(problems))
print(f"Loaded {len(CONFIG_ROWS):,} rows from the {DATASET_CONFIG!r} config" +
      (f"; identical to test.jsonl ({len(JSONL_ROWS):,} rows)." if JSONL_ROWS else "."))
if "pruned" in CONFIG_ROWS[0]:
    print(f"  of which {sum(bool(r['pruned']) for r in CONFIG_ROWS):,} are marked pruned (not in the debiased subset); "
          "all are kept, since this audit uses the full set.")
PROFILE = P.profile(CONFIG_ROWS)
print("Fields:", PROFILE["fields"])
print(f"Videos (unique dataset/scene_name): {PROFILE['videos']} {PROFILE['datasets']}; questions per video {PROFILE['questions_per_video']}")
print("Question types:", PROFILE["question_types"])
pruned_ids = {r["id"] for r in CONFIG_ROWS if r.get("pruned")} if "pruned" in CONFIG_ROWS[0] else None
QA_ALL = [L.normalize_row(r, OFFICIAL, pruned_ids) for r in sorted(CONFIG_ROWS, key=lambda r: r["id"])]
problems = P.schema_checks(CONFIG_ROWS, QA_ALL, OFFICIAL)
if problems:
    raise SystemExit("Schema checks failed; nothing selected:\n  " + "\n  ".join(problems))
for fam in ("multiple_choice", "numeric"):
    ex = next(q for q in QA_ALL if q["task_family"] == fam)
    print(f"\n[{ex['qa_id']} {ex['video_key']} {ex['question_type']}] prompt given to the model (after the image token):\n"
          f"{ex['prompt']}\n  -> reference (never in the prompt): {ex['reference_answer']!r}")
print("\nSchema checks passed: native ids unique, every question maps to one (dataset, scene_name) video, prompts are the "
      "official formatter's output and independent of the reference, every reference scores 1.0 under the official scorer.")

## 4. Select and freeze 10% of the videos

In [ ]:
# @title 4a. Sample videos once (seed 42), keep all their questions, persist, verify on later runs
SEL, QA, NEW = P.freeze_selection(OUTPUT_ROOT, QA_ALL, IDENTITY, ANNOTATION_SHA)
L.bind_selection(RUN_CONFIG_PATH, SEL["selection_hash"])
QA_BY_ID = {q["qa_id"]: q for q in QA}
assert SEL["selected_videos"] == L.math.ceil(SUBSET_FRACTION * SEL["population_videos"])
assert len({q["video_key"] for q in QA}) == SEL["selected_videos"]
assert sorted(q["qa_id"] for q in QA) == sorted(q["qa_id"] for q in QA_ALL if q["video_key"] in set(SEL["selected_video_keys"]))
print(("New selection saved." if NEW else "Existing selection verified (recomputed and identical; not resampled)."))
print(f"Videos:    {SEL['selected_videos']} of {SEL['population_videos']} ({SEL['selected_videos'] / SEL['population_videos']:.1%}; "
      f"ceil({SUBSET_FRACTION} x {SEL['population_videos']}))")
print(f"Questions: {SEL['selected_questions']} of {SEL['population_questions']:,} "
      f"({SEL['selected_questions'] / SEL['population_questions']:.1%}; all questions of the selected videos)")
print(f"Selection hash {SEL['selection_hash'][:16]}... (bound to this RUN_ID in run_config.json)")
from collections import Counter
print("By dataset (videos):", dict(Counter(v.split("/")[0] for v in SEL["selected_video_keys"])))
print("By question type:   ", dict(Counter(q["question_type"] for q in QA)))

## 5. Plan the media

In [ ]:
# @title 5a. Read each needed archive's directory (no download), map every selected video to its member
import zipfile
LISTING_DIR = os.path.join(DATA_CACHE, "listings")
ARCHIVE_DIR = os.path.join(LOCAL_ROOT, "archives")
ARCHIVE_SIZES = {p: i["size"] for p, i in FILES.items() if p.endswith(".zip")}
_ZIPS = {}

def open_archive_zip(archive):
    """A ZipFile of the pinned archive, reused across videos: HTTP range reads (only the bytes needed), or, if the
    server refuses range requests, a full download to local disk (its size printed first)."""
    if archive in _ZIPS:
        return _ZIPS[archive]
    try:
        _ZIPS[archive] = L.open_zip(L.HttpRangeFile(url_of(archive), ARCHIVE_SIZES[archive], AUTH))
        return _ZIPS[archive]
    except L.RangeNotSupported:
        pass
    os.makedirs(ARCHIVE_DIR, exist_ok=True)
    local = os.path.join(ARCHIVE_DIR, archive)
    if not (os.path.exists(local) and os.path.getsize(local) == ARCHIVE_SIZES[archive]):
        print(f"  range reads refused; downloading the whole {archive} ({ARCHIVE_SIZES[archive] / 1e9:.1f} GB) ...", flush=True)
        with urllib.request.urlopen(urllib.request.Request(url_of(archive), headers=AUTH), timeout=600) as r, \
                open(local + ".part", "wb") as out:
            shutil.copyfileobj(r, out, 1 << 22)
        os.replace(local + ".part", local)
    _ZIPS[archive] = zipfile.ZipFile(local)
    return _ZIPS[archive]

LISTINGS = {}
for archive in sorted({L.archive_for(v)[0] for v in SEL["selected_video_keys"]}):
    lp = os.path.join(LISTING_DIR, archive + ".json")
    if os.path.exists(lp):
        LISTINGS[archive] = json.load(open(lp))
    else:
        LISTINGS[archive] = L.zip_listing(open_archive_zip(archive))
        L.write_json_atomic(lp, LISTINGS[archive])
MANIFEST = P.media_manifest(SEL["selected_video_keys"], LISTINGS, ARCHIVE_SIZES)
MANIFEST_BY_VIDEO = {m["video_key"]: m for m in MANIFEST}
L.write_jsonl_atomic(os.path.join(OUTPUT_ROOT, "media_manifest.jsonl"), MANIFEST)
print(f"{'archive':18s} {'archive GB':>10s} {'videos':>7s} {'selected MB':>12s}  missing")
for p in P.transfer_plan(MANIFEST, ARCHIVE_SIZES):
    print(f"{p['archive']:18s} {(p['archive_bytes'] or 0) / 1e9:10.2f} {p['videos']:7d} {p['selected_bytes'] / 1e6:12.1f}  {p['missing'] or '-'}")
sel_bytes = sum(m["member_bytes"] or 0 for m in MANIFEST)
print(f"Transfer: {sel_bytes / 1e9:.2f} GB of selected videos by range reads (each checked for size and CRC). "
      f"Fallback if range reads are refused: whole archives, {sum(ARCHIVE_SIZES[a] for a in LISTINGS) / 1e9:.1f} GB. "
      f"Local free space {shutil.disk_usage('/content').free / 1e9:.0f} GB.")
missing = [m["video_key"] for m in MANIFEST if m["member_missing"]]
if missing:
    print(f"WARNING: {len(missing)} selected video(s) are not in their archive; they stay in the sample and in coverage "
          f"as missing (never replaced): {missing}")

## 6. Run inference (resumable)

In [ ]:
# @title 6a. Runner (defines functions; run before 6b/6c and before sections 8-9)
import time as _time
MEDIA_DIR = os.path.join(LOCAL_ROOT, "media")
WORK_DIR = os.path.join(LOCAL_ROOT, "batches")
VIDEO_CACHE = os.path.join(DATA_CACHE, "videos")
MISSING_LOG = os.path.join(OUTPUT_ROOT, "logs", "media_missing.jsonl")
for d in (MEDIA_DIR, WORK_DIR, os.path.dirname(JOURNAL)):
    os.makedirs(d, exist_ok=True)

def fetch_video(video_key, dest_dir=None):
    """Local path of a selected video: the Drive video cache (size + CRC checked) or the archive member (range read,
    size + CRC checked, then cached on Drive). Raises KeyError if the member is not in its archive."""
    m = MANIFEST_BY_VIDEO[video_key]
    dest_dir = dest_dir or MEDIA_DIR
    local = L.safe_extract_path(dest_dir, m["member"])
    if os.path.exists(local) and os.path.getsize(local) == m["member_bytes"]:
        return local
    cached = L.safe_extract_path(VIDEO_CACHE, m["member"])
    if os.path.exists(cached) and os.path.getsize(cached) == m["member_bytes"] and L.file_crc32(cached) == m["crc32"]:
        os.makedirs(os.path.dirname(local), exist_ok=True)
        shutil.copyfile(cached, local)
        return local
    if m["member_missing"]:
        raise KeyError(f"{m['member']} is not in {m['archive']}")
    path = L.extract_zip_member(open_archive_zip(m["archive"]), m["member"], dest_dir, m["member_bytes"], m["crc32"])
    if CACHE_VIDEOS:
        os.makedirs(os.path.dirname(cached), exist_ok=True)
        shutil.copyfile(path, cached + ".part")
        os.replace(cached + ".part", cached)
    return path

def release(video_key, path):
    if os.path.exists(path) and os.path.realpath(path).startswith(os.path.realpath(MEDIA_DIR)):
        os.remove(path)

def run_worker(batch_path, extra, journal=None, shard_dir=None, status_name="worker_status.json"):
    cmd = [VENV_PY, "/content/vsibench/vsibench_worker.py", "--batch", batch_path, "--model_path", "ckpts/Cambrian-P-7B",
           "--num_frames", str(NUM_FRAMES), "--config_hash", CONFIG_HASH, "--identity", json.dumps(IDENTITY),
           "--journal", journal or JOURNAL, "--errors", ERRORS_JOURNAL, "--shard_dir", shard_dir or SHARD_DIR,
           "--status", os.path.join(OUTPUT_ROOT, "logs", status_name), "--lib_dir", "/content/vsibench",
           "--max_retries", str(MAX_RETRIES)] + extra
    log_path = os.path.join(OUTPUT_ROOT, "logs", f"worker-{_time.strftime('%Y%m%d-%H%M%S')}-{_time.time_ns() % 10**6:06d}.log")
    with open(log_path, "w") as logf:
        p = subprocess.Popen(cmd, cwd=REPO, env=WORKER_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            logf.write(line)
            if any(k in line for k in ("token_num:", "[cam-load]", "SUCCESS", "WARNING", "Error", "error", "Traceback")):
                print("   " + line.rstrip()[:220])
        code_ = p.wait()
    sp = os.path.join(OUTPUT_ROOT, "logs", status_name)
    st = json.load(open(sp)) if os.path.exists(sp) else {}
    print(f"  worker exit {code_}: {st.get('phase')} | answers {st.get('answers')} | camera head loaded: "
          f"{st.get('camera_head_loaded')} | log {log_path}")
    return code_

def context():
    return {"fetch_video": fetch_video, "run_worker": run_worker, "release": release, "media_dir": MEDIA_DIR,
            "work_dir": WORK_DIR, "shard_dir": SHARD_DIR, "journal": JOURNAL, "config_hash": CONFIG_HASH,
            "batch_bytes": int(MEDIA_BATCH_GB * 1e9),
            "log_missing": lambda v, a, m, why: P.log_missing_media(MISSING_LOG, v, a, m, why),
            "skip_videos": set(P.missing_media(MISSING_LOG))}

PILOT_QA = L.pilot_subset(QA, {m["video_key"]: m["member_bytes"] for m in MANIFEST}, PILOT_VIDEOS, PILOT_MAX_QA)
print(f"Pilot: {len(PILOT_QA)} questions on {len({q['video_key'] for q in PILOT_QA})} videos "
      f"({', '.join(sorted({q['video_key'] for q in PILOT_QA}))}), the smallest selected videos.")

# Pre-flight (seconds, no GPU): run the real worker on a real selected video with a stand-in model, so a bug in the
# worker's own code (decoding, prompt, saving answers, retries) stops here instead of costing a GPU run.
PREFLIGHT_VIDEO = fetch_video(PILOT_QA[0]["video_key"])
_pf = subprocess.run([VENV_PY, "/content/vsibench/vsibench_worker_selftest.py", "--video", PREFLIGHT_VIDEO],
                     capture_output=True, text=True, cwd="/content/vsibench")
print(_pf.stdout.strip().splitlines()[-1] if _pf.stdout.strip() else "worker pre-flight produced no output")
if _pf.returncode:
    print(_pf.stdout[-4000:] + _pf.stderr[-2000:])
    raise SystemExit("Worker pre-flight failed: nothing was sent to the GPU. Send this output for a fix.")

In [ ]:
# @title 6b. Run (full by default, or pilot, per RUN_MODE). Rerun to resume after any interruption.
subset = PILOT_QA if RUN_MODE == "pilot" else QA
_last = (json.load(open(os.path.join(OUTPUT_ROOT, "run_manifest.json"))).get("last_run") or {}).get("mode")     if os.path.exists(os.path.join(OUTPUT_ROOT, "run_manifest.json")) else None
print(f"Runtime mode: {RUN_MODE} -> {len(subset)} question(s) on {len({q['video_key'] for q in subset})} video(s) "
      + ("(the whole selected subset, not the whole benchmark)" if RUN_MODE == "full" else "(pilot smoke test)"))
print(f"Selected subset: {SEL['selected_videos']} videos / {SEL['selected_questions']} questions. "
      f"Last recorded inference mode: {_last or 'none recorded'} (the setting is an intention; the counts below are the evidence).")
# Saved answers are never recomputed: only questions without one are sent to the model, and the model weights are
# loaded only if there are such questions (a review-only session needs no GPU and no weights).
_ok0, _, _, _ = P.gather_results(SHARD_DIR, JOURNAL, CONFIG_HASH)
_skip0 = set(P.missing_media(MISSING_LOG))
_pending = [q for q in subset if q["qa_id"] not in _ok0 and q["video_key"] not in _skip0]
print(f"Saved answers found: {sum(q['qa_id'] in _ok0 for q in subset)}/{len(subset)}; still to answer: {len(_pending)}.")
if _pending and not gpu:
    raise SystemExit(f"{len(_pending)} question(s) still need the model, which needs a GPU: Runtime > Change runtime "
                     "type > A100, then Run all. (Scoring and review, sections 7-9, run on a CPU runtime.)")
if _pending:
    ensure_model()
t0 = _time.time()
summary = P.run_selection(subset, MANIFEST_BY_VIDEO, context())
ok, failed, meta, report = P.gather_results(SHARD_DIR, JOURNAL, CONFIG_HASH)
done = sum(1 for q in subset if q["qa_id"] in ok)
print(f"\n{RUN_MODE}: {done}/{len(subset)} questions answered in this set ({len(ok)}/{len(QA)} for the whole sample); "
      f"{(_time.time() - t0) / 60:.1f} min this session. Shards: {report['complete']} complete; ignored: "
      f"{len(report['incomplete'])} incomplete, {len(report['corrupted'])} corrupted, {len(report['other_config'])} other-config.")
if summary.get("missing"):
    print(f"Media missing or corrupt (kept in coverage, not replaced): {summary['missing']}")
if RUN_MODE == "pilot" and done:
    print("\nSample pilot answers:")
    for q in subset[:6]:
        if q["qa_id"] in ok:
            print(f"  [{q['question_type']}] model {ok[q['qa_id']]['raw_answer']!r} | reference {q['reference_answer']!r}")
if RUN_MODE == "full" and done == len(subset):
    print("\nInference is complete for the whole sample. Continue with sections 7-9 (no GPU needed).")
update_manifest(last_run={"mode": RUN_MODE, "summary": summary, "answered_total": len(ok)})

In [ ]:
# @title 6c. Pilot resume check: interrupt on purpose, restart, and confirm no answer is lost or duplicated
# Uses a separate journal and shard folder, so the run's own results are untouched. Skipped in full mode.
if RUN_MODE != "pilot":
    print("Full mode: the resume check runs in pilot mode only; skipped.")
else:
    ensure_model()
    TEST_DIR = os.path.join(LOCAL_ROOT, "resume_check")
    shutil.rmtree(TEST_DIR, ignore_errors=True)
    os.makedirs(TEST_DIR)
    test_journal, test_shards = os.path.join(TEST_DIR, "journal.jsonl"), os.path.join(TEST_DIR, "shards")
    def run_test_worker(batch_path, extra, stop=0):
        return run_worker(batch_path, extra + (["--stop_after", str(stop)] if stop else []), journal=test_journal,
                          shard_dir=test_shards, status_name="resume_check_status.json")
    ctx = dict(context(), journal=test_journal, shard_dir=test_shards, run_worker=lambda b, e: run_test_worker(b, e, stop=1))
    P.run_selection(PILOT_QA, MANIFEST_BY_VIDEO, ctx)
    ok1, _, _, _ = P.gather_results(test_shards, test_journal, CONFIG_HASH)
    first = {k: v["raw_answer"] for k, v in ok1.items()}
    print(f"After the deliberate stop: {len(first)} answers kept (in the journal; that video's shard was not finished).")
    ctx["run_worker"] = lambda b, e: run_test_worker(b, e)
    P.run_selection(PILOT_QA, MANIFEST_BY_VIDEO, ctx)
    ok2, _, _, _ = P.gather_results(test_shards, test_journal, CONFIG_HASH)
    rows, _ = L.read_jsonl(test_journal)
    per_q = Counter(r["qa_id"] for r in rows if r.get("kind") == "answer" and r.get("status") == "ok")
    checks = {"the deliberate stop happened before all pilot questions were answered": 1 <= len(first) < len(PILOT_QA),
              "answers written before the stop were kept": len(first) >= 1,
              "all pilot questions answered after restart": all(q["qa_id"] in ok2 for q in PILOT_QA),
              "kept answers unchanged": all(ok2[k]["raw_answer"] == v for k, v in first.items()),
              "no question answered twice": bool(per_q) and max(per_q.values()) == 1}
    for k, v in checks.items():
        print(("PASS " if v else "FAIL ") + k)
    update_manifest(resume_check=checks)
    if not all(checks.values()):
        raise SystemExit("Resume check failed; do not start the full run until this is fixed.")

## 7. Official scoring

In [ ]:
# @title 7a. Merge results, score with the official VSI-Bench scorer, write predictions / errors / scores / coverage
OK, FAILED, META, merge_report = P.gather_results(SHARD_DIR, JOURNAL, CONFIG_HASH)
MISSING = P.missing_media(MISSING_LOG)
SCORES = P.score_all(QA, OK, FAILED, MISSING, OFFICIAL)
SCORES_BY_ID = {s["qa_id"]: s for s in SCORES}
P.write_score_outputs(OUTPUT_ROOT, OK, FAILED, SCORES, QA_BY_ID)
import vsibench_diagnostics as D
importlib.reload(D)
# Content fingerprints of what was scored (answers and errors, config, scorer) and of the scores, so 8a/8c can tell
# whether the saved scores and the review queue still match the saved predictions.
D.record_scoring(OUTPUT_ROOT, D.prediction_fingerprint(QA, OK, FAILED, MISSING, CONFIG_HASH, SCORER["sha256"]),
                 D.scores_fingerprint(SCORES, [q["qa_id"] for q in QA]), len(SCORES))
COVERAGE = P.coverage(SEL, QA, OK, FAILED, SCORES, MISSING, RUN_MODE, OFFICIAL)
COVERAGE["scoring"] = {"scorer": SCORER, "automatic_status": {
    "correct": "official score 1",
    "wrong_candidate": "multiple choice: official exact match 0 with a valid option letter; numeric: official MRA 0 "
                       "(automatic failure candidate; counts as verified only after a final review)",
    "imperfect": "numeric partial credit (0 < MRA < 1); never a verified wrong by score alone",
    "format_error": "the official parse is not an option letter / not a number (official score recorded)",
    "unscorable": "invalid reference or unsupported question type",
    "execution_error": "inference or media failure; never a wrong answer"}}
_cov_path = os.path.join(OUTPUT_ROOT, "coverage.json")
_prev = json.load(open(_cov_path)) if os.path.exists(_cov_path) else {}
L.write_json_atomic(_cov_path, dict(COVERAGE, **{k: _prev[k] for k in ("verification", "gate") if k in _prev}))
print(json.dumps({k: COVERAGE[k] for k in ("full_dataset", "selected", "videos", "questions", "automatic_status",
                                            "inference_complete")}, indent=1))
sub = COVERAGE["official_subset_scores"]
if sub["scores"]:
    print(f"\nOfficial scores on the {sub['scope']}:")
    for k, v in sub["scores"].items():
        print(f"  {k:40s} {v:6.1f}")
print("\nFiles: predictions.jsonl, errors.jsonl, scores.jsonl, coverage.json (candidates are listed by 8a)")

## 8. Failure candidates and manual frame-evidence review

**This is a manual stage.** The official score only nominates *candidates*: a multiple-choice miss or a zero-credit number is an automatic failure candidate, a malformed reply is a format-error candidate, and partial numeric credit (`imperfect`) is listed in `failure_candidates.csv` but is never a verified failure by score alone. Exported frames do not show that the answer is visible; a reviewer decides that.

A video counts as a **verified failure** only when `vsibench_review.validate_verified_failure` accepts one of its questions: the run's own prediction, a blind judgment saved before the answers were shown, a final review for that exact question and run with verdict *wrong*, valid evidence citations (or a documented search scope), and a playable, decode-checked video with its exact input frames. The progress line, `summary.csv`, the report and the final manifest all use that one function.

Review order: one primary question per video (up to two supporting), 20 videos at a time; when a reviewer marks a candidate correct or uncertain, the next reserve candidate from the same sample is offered. Nothing is ever deleted: every save is kept in the review's history.

In [ ]:
# @title 8a. Candidates and the review queue (automatic; nothing here is a verified failure)
HELPER_DIR = "/content/vsibench"
import importlib, sys
sys.path.insert(0, HELPER_DIR)
import vsibench_review as R
import vsibench_diagnostics as D
importlib.reload(R); importlib.reload(D)
from collections import Counter
_diag = D.diagnose_from_globals(globals(), D.NEEDS_8A)
if _diag["missing_prerequisites"]:
    print(D.render_candidate_diagnostic(_diag, "Cannot build the review queue yet"))
else:
    SELECTED_VIDEO_KEYS = sorted({m["video_key"] for m in MANIFEST})
    R.write_failure_candidates(os.path.join(OUTPUT_ROOT, "failure_candidates.csv"), SCORES, QA_BY_ID)
    QUEUE_PATH = os.path.join(OUTPUT_ROOT, "review_queue.json")
    QUEUE = R.update_queue(QUEUE_PATH, SCORES, QA_BY_ID, SELECTED_VIDEO_KEYS, TARGET_FAILURE_VIDEOS)
    D.record_queue(OUTPUT_ROOT, D.scores_fingerprint(SCORES, list(QA_BY_ID)), len(QUEUE["entries"]))
    ACTIVE, RESERVE = R.queue_split(QUEUE)
    EXPORT_CTX = {"output_root": OUTPUT_ROOT, "venv_py": VENV_PY, "env": WORKER_ENV, "lib_dir": HELPER_DIR,
                  "fetch_video": fetch_video, "meta": META, "qa_by_id": QA_BY_ID, "scores_by_id": SCORES_BY_ID,
                  "config_hash": CONFIG_HASH}

    def review_ctx():
        return {"output_root": OUTPUT_ROOT, "config_hash": CONFIG_HASH, "selected_video_keys": SELECTED_VIDEO_KEYS,
                "qa_by_id": QA_BY_ID, "ok": OK, "scores_by_id": SCORES_BY_ID, "media": MANIFEST_BY_VIDEO,
                "reviews": R.load_reviews(OUTPUT_ROOT), "run_id": RUN_ID}

    print("Automatic status of every scored question:", dict(Counter(s["automatic_status"] for s in SCORES)))
    print(f"Inspection candidates (official score below its maximum): "
          f"{sum(s['automatic_status'] in R.INSPECTION_STATUSES for s in SCORES)} -> failure_candidates.csv")
    print(f"Candidate videos (a wrong or format-error answer): {len(QUEUE['entries'])}; review queue: {len(ACTIVE)} active, "
          f"{len(RESERVE)} in reserve (saved in review_queue.json).")
    for e in ACTIVE:
        print(f"  {e['dataset']:12s} {e['primary_question_type']:28s} {e['automatic_status']:16s} {e['video_key']}")
    print()
    print(D.render_candidate_diagnostic(D.diagnose_from_globals(globals(), D.NEEDS_8A)))

In [ ]:
# @title 8b. Export the active candidates: playable video on Drive, decode check, exact input frames, contact sheets
if not all(k in globals() for k in ("R", "D", "ACTIVE", "EXPORT_CTX")):
    print("Cannot export yet: run 8a first (it builds the review queue).")
elif not ACTIVE:
    print(D.render_candidate_diagnostic(D.diagnose_from_globals(globals()), "Nothing to export: the review queue is empty"))
else:
    for e in ACTIVE:
        problems = R.export_candidate(e, EXPORT_CTX)
        man = R.load_frames_manifest(R.case_dir(OUTPUT_ROOT, e)) or {}
        print(f"  {e['safe_key']}: {sum(x['frame_set'] == 'input' for x in man.get('entries', []))} input frames, "
              f"inputs verified: {man.get('inputs_verified')}" + (f"; PROBLEMS: {problems}" if problems else ""))
    print(f"Exported {len(ACTIVE)} candidate video(s) to candidate_cases/ (history; not the final deliverable).")

In [ ]:
# @title 8c. Review (manual): blind judgment first, then answers, extra frames and the final verdict
import sys as _sys
_sys.path.insert(0, "/content/vsibench")
try:
    import vsibench_diagnostics as D
    _DIAG = D.diagnose_from_globals(globals())
except ImportError:
    D, _DIAG = None, None
_READY = _DIAG is not None and not _DIAG["missing_prerequisites"]
import ipywidgets as W
from IPython.display import display, clear_output
UI = {}
progress, pick, body = W.HTML(), W.Dropdown(layout=W.Layout(width="90%")), W.Output()
_busy = {"pick": False}
_LAST = {}   # latest validator results and counts (survive re-rendering a case)
RED = "<span style='color:#b00'>{}</span>"

def _blank(label):
    return [(f"— {label} —", "")]

def current_state():
    """Validator results for every candidate, reserve promotion after overrides, and the shared counts."""
    global ACTIVE, RESERVE
    ctx = review_ctx()
    per_video = R.evaluate(QUEUE["entries"], ctx)
    offered = R.promote_reserve(QUEUE_PATH, QUEUE, {k: v["state"] for k, v in per_video.items()}, TARGET_FAILURE_VIDEOS)
    for e in offered:
        R.export_candidate(e, EXPORT_CTX)
    ACTIVE, RESERVE = R.queue_split(QUEUE)
    if offered:
        per_video = R.evaluate(QUEUE["entries"], review_ctx())
    return per_video, R.counts(QUEUE["entries"], ACTIVE, per_video, OUTPUT_ROOT, TARGET_FAILURE_VIDEOS,
                               MIN_FAILURE_VIDEOS), offered

def refresh():
    per_video, C, offered = current_state()
    _LAST.update(counts=C, per_video=per_video, offered=offered)
    progress.value = ("<b>Verified failure videos: {verified_wrong_video_count}</b> (minimum {minimum}, target {target}) · "
                      "final reviews {final_review_video_count} · overridden correct {overridden_correct_video_count} · "
                      "uncertain {uncertain_video_count} · invalid/missing evidence {invalid_or_missing_evidence_video_count}"
                      " · exported {exported_video_count} · candidates {candidate_video_count}").format(**C)
    _busy["pick"] = True
    keep = pick.value
    pick.options = [(f"{i + 1}. [{per_video[e['video_key']]['state']}] {e['video_key']}", i) for i, e in enumerate(ACTIVE)]
    pick.value = keep if keep in range(len(ACTIVE)) else (0 if ACTIVE else None)
    _busy["pick"] = False
    return offered

def show(i, qa_id=None, message=""):
    e = ACTIVE[i]
    qid = qa_id or e["primary_qa_id"]
    q, s = QA_BY_ID[qid], SCORES_BY_ID[qid]
    r = R.load_review(OUTPUT_ROOT, qid)
    r = r if r and r.get("config_hash") == CONFIG_HASH else None
    cdir = R.case_dir(OUTPUT_ROOT, e)
    man = R.load_frames_manifest(cdir) or {"entries": []}
    inputs = [x for x in man["entries"] if x["frame_set"] == "input"]
    UI.clear()
    UI.update(case=e, qa_id=qid, revealed=lambda: "answers" in UI, counts=_LAST.get("counts"),
              offered=_LAST.get("offered"))
    opts = "".join(f"<li>{L.esc(o)}</li>" for o in q.get("options") or [])
    head = W.HTML(f"<h3>{L.esc(e['video_key'])}</h3><p>{L.esc(e['dataset'])} · {L.esc(q['question_type'])} · "
                  f"<code>{L.esc(qid)}</code> (VSI-Bench id {L.esc(q.get('native_id'))})</p>"
                  f"<p><b>Question:</b> {L.esc(q['question'])}</p><ul>{opts}</ul>"
                  f"<p>Video: <code>{L.esc(os.path.join(cdir, 'video.mp4'))}</code></p>")
    qa_pick = W.Dropdown(options=[(f"primary: {e['primary_qa_id']}", e["primary_qa_id"])] +
                         [(f"supporting: {x}", x) for x in e["supporting_qa_ids"]], value=qid, description="Question")
    qa_pick.observe(lambda ch: show(i, ch["new"]) if ch["new"] != qid else None, names="value")
    prov = W.HTML(f"<p><b>Model input:</b> {len(inputs)} frames (indices {inputs[0]['frame_index'] if inputs else '-'}"
                  f"…{inputs[-1]['frame_index'] if inputs else '-'} of {man.get('total_frames')}, {man.get('fps')} fps), "
                  f"reader {L.esc(man.get('reader'))}; preprocessing: {L.esc(man.get('preprocessing'))}<br>"
                  f"<b>Inputs verified:</b> {man.get('inputs_verified')} ({L.esc(man.get('verification'))})</p>")
    sheets = [W.Image(value=open(os.path.join(cdir, f), "rb").read(), format="jpg", layout=W.Layout(width="100%"))
              for f in man.get("input_sheets", [])]
    full_pos = W.BoundedIntText(value=0, min=0, max=max(len(inputs) - 1, 0), description="Input #")
    full_show, full_img = W.Button(description="Show full-resolution frame"), W.Image(format="jpg")
    def _full(_):
        x = inputs[full_pos.value]
        full_img.value = open(os.path.join(cdir, x["file"]), "rb").read()
        full_img.layout.width = "100%"
    full_show.on_click(_full)
    UI.update(qa_pick=qa_pick, full_pos=full_pos, full_show=full_show, full_img=full_img, input_sheets=sheets)
    parts = [head, qa_pick, prov] + sheets + [W.HBox([full_pos, full_show]), full_img]

    if not (r and r.get("provisional")):
        pa = W.Text(description="Your answer", placeholder="letter, number, or 'cannot determine'")
        sp, pm = W.Button(description="Save my answer and reveal", button_style="info"), W.HTML(message)
        def _prov(_):
            _, errs = R.save_provisional(OUTPUT_ROOT, q, CONFIG_HASH, RUN_ID, pa.value, "", "")
            if errs:
                pm.value = RED.format(L.esc("; ".join(errs)))
                return
            refresh()
            show(i, qid, "Blind judgment saved.")
        sp.on_click(_prov)
        UI.update(provisional_answer=pa, save_provisional=sp, prov_msg=pm)
        parts += [W.HTML("<b>Step 1. Your own answer</b>, from the frames above (the model's and the benchmark's answers "
                         "stay hidden until you save it)"), pa, sp, pm]
    else:
        p0 = r["provisional"]
        answers = W.HTML(
            f"<p><i>Blind judgment ({L.esc(p0['saved_at'])}): {L.esc(p0['answer'])} / {L.esc(p0.get('evidence_status') or '-')}"
            f"</i></p><p><b>Model:</b> {L.esc(s.get('raw_answer'))} → officially parsed {L.esc(s.get('parsed_answer'))} · "
            f"<b>Benchmark answer:</b> {L.esc(s.get('reference_answer'))} · <b>{L.esc(s.get('official_metric'))}:</b> "
            f"{L.esc(s.get('official_score'))} · automatic status {L.esc(s.get('automatic_status'))}</p>"
            f"<p class='k'>{L.esc(s.get('candidate_reason'))}</p>")
        fps_w = W.Text(value="1", description="Extra fps")
        win_w = W.Text(description="Windows", placeholder="start,end[,fps]; start,end[,fps] (seconds)")
        xb, xm = W.Button(description="Extract extra frames"), W.HTML()
        xbox = W.VBox()
        def _extra_images():
            m2 = R.load_frames_manifest(cdir) or {}
            xbox.children = [W.Image(value=open(os.path.join(cdir, f), "rb").read(), format="jpg",
                                     layout=W.Layout(width="100%")) for f in m2.get("extra_sheets", [])]
        def _extra(_):
            wins, errs = R.parse_windows(win_w.value)
            try:
                f = float(fps_w.value or 0)
            except ValueError:
                errs.append("fps must be a number")
                f = 0
            if errs:
                xm.value = RED.format(L.esc("; ".join(errs)))
                return
            code_, out = R.run_extra_frames(e, EXPORT_CTX, f, wins)
            _extra_images()
            xm.value = (RED.format(L.esc(out[-300:])) if code_ else
                        L.esc(out[-200:]) + " — extra frames are kept apart from the model inputs (frames_extra/)")
        xb.on_click(_extra)
        _extra_images()
        cur = R.current_final(r) or {}
        wide = W.Layout(width="95%")
        verdict = W.Dropdown(options=_blank("choose") + list(R.VERDICT_CHOICES), value=cur.get("verdict", ""),
                             description="Verdict", layout=wide, style={"description_width": "150px"})
        evidence = W.Dropdown(options=_blank("choose") + list(R.EVIDENCE_CHOICES), value=cur.get("evidence_status", ""),
                              description="Evidence visible?", layout=wide, style={"description_width": "150px"})
        frames_w = W.Text(value=R.frames_text(cur.get("citations"), man), description="Which frames?",
                          placeholder="#12, #40   (numbers printed on the sheets; f1234 for an extra frame)",
                          layout=wide, style={"description_width": "150px"})
        confidence = W.Dropdown(options=_blank("choose") + [(x, x) for x in R.CONFIDENCE], value=cur.get("confidence", ""),
                                description="Confidence", style={"description_width": "150px"})
        notes = W.Textarea(value=cur.get("rationale", ""), description="Notes",
                           placeholder="one or two sentences: what you saw and why you chose this verdict",
                           layout=W.Layout(width="95%", height="60px"), style={"description_width": "150px"})
        sf, fm, st = W.Button(description="Save review", button_style="success"), W.HTML(message), W.HTML()
        res = next((x for x in UI_STATE_RESULTS(e["video_key"]) if x["qa_id"] == qid), None)
        st.value = ("<b>Validator:</b> " + ("verified failure" if res and res["eligible"] else
                    RED.format(L.esc("; ".join((res or {}).get("reasons", [])[:6])))))
        def _final(_):
            man_now = R.load_frames_manifest(cdir)
            final, errs = R.quick_final(verdict.value, evidence.value, confidence.value, frames_w.value, notes.value,
                                        man_now, s.get("automatic_status"))
            if not errs:
                _, errs = R.save_final(OUTPUT_ROOT, q, CONFIG_HASH, final, man_now, cdir, s.get("automatic_status"))
            if errs:
                fm.value = RED.format("Not saved: " + L.esc("; ".join(errs)))
                return
            offered = refresh()
            msg = f"Final review saved. Verified failure videos now: {_LAST['counts']['verified_wrong_video_count']}."
            if offered:
                msg += " Next reserve candidate offered: " + ", ".join(x["video_key"] for x in offered) + "."
            show(i, qid, msg)
        sf.on_click(_final)
        UI.update(answers=answers, fps=fps_w, windows=win_w, extra=xb, extra_msg=xm, extra_images=xbox, verdict=verdict,
                  evidence=evidence, frames=frames_w, confidence=confidence, notes=notes, save_final=sf, final_msg=fm,
                  status=st)
        parts += [answers, W.HTML("<b>Step 2 (optional). More frames</b> from the whole video, kept apart from the "
                                  "model's frames"), W.HBox([fps_w, win_w, xb]), xm, xbox,
                  W.HTML("<b>Step 3. Your review</b>. <i>Evidence visible?</i> asks whether the frames show what is "
                         "needed to answer the question. For <i>Which frames?</i> type the #numbers printed on the "
                         "input sheets above (e.g. #12, #40), or f-numbers from the extra sheets (e.g. f1234); not "
                         "needed if you couldn't find it or can't tell."),
                  verdict, evidence, frames_w, confidence, notes, sf, fm, st]
    with body:
        clear_output()
        display(W.VBox(parts))

def UI_STATE_RESULTS(video_key):
    pv = (_LAST.get("per_video") or {}).get(video_key)
    return pv["results"] if pv else []

def _on_pick(ch):
    if not _busy["pick"] and ch["new"] is not None:
        show(ch["new"])
pick.observe(_on_pick, names="value")
if D is None:
    print("The helper code is not installed in this runtime: run 1a and the 1b cells, then 1c onward.")
elif not _READY:
    print(D.render_candidate_diagnostic(_DIAG, "The review cannot start yet"))
else:
    refresh()
    _DIAG = D.diagnose_from_globals(globals())            # after refresh (it may promote reserve candidates)
    if ACTIVE:
        if any(f["code"] != "ready" for f in _DIAG["findings"]):
            print(D.render_candidate_diagnostic(_DIAG, "Review status"))
        display(W.VBox([progress, pick, body]))
        show(0)
    else:
        print(D.render_candidate_diagnostic(_DIAG, "Review queue is empty"))

## 9. Verified failures and final outputs
`failure_videos/` and `cases/` contain exactly the cases in `verified_failure_manifest.json` (at most the target number, in queue order). Candidate history stays in `candidate_cases/` and `reviews/`. Inference completion and case-export completion are reported separately: verified cases do not mean the whole sample was run.

In [ ]:
# @title 9a. Verified-failure manifest, summary, report (all counts from the one validator)
ctx = review_ctx()
PER_VIDEO = R.evaluate(QUEUE["entries"], ctx)
ACTIVE, RESERVE = R.queue_split(QUEUE)
COUNTS = R.counts(QUEUE["entries"], ACTIVE, PER_VIDEO, OUTPUT_ROOT, TARGET_FAILURE_VIDEOS, MIN_FAILURE_VIDEOS)
GATE = R.completion_gate(COUNTS, COVERAGE)
FINAL = R.build_final(OUTPUT_ROOT, PER_VIDEO, ACTIVE, ctx, COUNTS, GATE, TARGET_FAILURE_VIDEOS)
R.write_summary(os.path.join(OUTPUT_ROOT, "summary.csv"), PER_VIDEO, ACTIVE, ctx)
info = {"dataset": f"{IDENTITY.get('DATASET_REPO')} config {IDENTITY.get('DATASET_CONFIG')} split "
                   f"{IDENTITY.get('DATASET_SPLIT')} @ {IDENTITY.get('DATASET_REVISION')}",
        "model": f"{IDENTITY.get('MODEL_REPO')} @ {IDENTITY.get('MODEL_REVISION')}, {IDENTITY.get('NUM_FRAMES')} frames",
        "run mode": RUN_MODE, "config hash": CONFIG_HASH}
info.update({f"selection: {k}": v for k, v in SEL.items() if isinstance(v, (int, float, str)) and len(str(v)) < 80})
info.update({f"coverage: {k}": v for k, v in COVERAGE.items() if isinstance(v, (int, float, str, bool, dict))
             and len(json.dumps(v, default=str)) < 300})
R.report_html(OUTPUT_ROOT, FINAL, PER_VIDEO, info)
cov_path = os.path.join(OUTPUT_ROOT, "coverage.json")
cov = json.load(open(cov_path)) if os.path.exists(cov_path) else {}
cov.update(verification=COUNTS, gate=GATE)
L.write_json_atomic(cov_path, cov)
assert FINAL["verified_video_count_in_manifest"] == min(COUNTS["verified_wrong_video_count"], TARGET_FAILURE_VIDEOS)
print(GATE["message"])
for k, v in COUNTS.items():
    print(f"  {k}: {v}")
print(f"Inference complete: {GATE['inference_complete']} · case export complete (minimum met): {GATE['case_export_complete']}")
print(f"Wrote verified_failure_manifest.json ({len(FINAL['cases'])} case(s)), summary.csv, report.html in {OUTPUT_ROOT}")

In [ ]:
# @title 9b. (Optional) Package the final deliverable (only the manifest's verified cases)
MAKE_ZIP = False  # @param {type:"boolean"}
if MAKE_ZIP:
    zpath = os.path.join(os.path.dirname(OUTPUT_ROOT), RUN_ID + "_verified_failures.zip")
    names = R.package(OUTPUT_ROOT, zpath)
    print(f"Saved {zpath}: {len(names)} files, {os.path.getsize(zpath) / 1e9:.2f} GB")
else:
    print("Set MAKE_ZIP = True to write a zip of report.html, summary.csv, the manifest and its verified cases only.")

## Verification

In [ ]:
# @title V1. Self-tests (no GPU). A failure stops this cell with an error.
real_video = None
for m_ in MANIFEST:                       # one real selected video for the review/frames tests, if available
    try:
        real_video = fetch_video(m_["video_key"])
        break
    except Exception:
        continue
results = {}
for name, py, args in (("vsibench_selftest.py", sys.executable,
                        ["--repo", REPO, "--annotations", cached_file("test.jsonl")] +
                        sum((["--config_file", cached_file(f)] for f in CONFIG_FILES["files"]), []) +
                        ["--listings", LISTING_DIR]),
                       # the review test drives the widgets in this kernel; frame decoding runs in the model venv
                       ("vsibench_review_selftest.py", sys.executable,
                        ["--frames_python", VENV_PY] + (["--real_video", real_video] if real_video else [])),
                       ("vsibench_diagnostics_selftest.py", sys.executable, []),
                       ("vsibench_worker_selftest.py", VENV_PY, ["--video", PREFLIGHT_VIDEO])):
    r = subprocess.run([py, f"/content/vsibench/{name}"] + args, capture_output=True, text=True, cwd="/content/vsibench")
    print(f"===== {name} (exit {r.returncode})\n" + r.stdout[-8000:] + r.stderr[-3000:])
    results[name] = r.returncode
update_manifest(selftests=results)
if any(results.values()):
    raise RuntimeError(f"Self-tests failed: {results}")
print("All self-tests passed.")

In [ ]:
# @title (Optional) Unmount Google Drive when you are completely done (not during Run all)
# Unmounting while you still review in 8c would send later saves to a local folder that is lost with the runtime,
# so Run all skips it. Set UNMOUNT_DRIVE = True and run this cell by hand at the very end.
UNMOUNT_DRIVE = False  # @param {type:"boolean"}
if UNMOUNT_DRIVE:
    from google.colab import drive
    drive.flush_and_unmount()
else:
    print("Drive stays mounted (results are already saved there as each step finishes).")